# Setup

In [79]:
!pip install requests beautifulsoup4 pandas

import requests
import pandas as pd
import re

# Query OpenStreetMap using Overpass API

In [80]:
overpass_url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:120];
area["ISO3166-1"="PH"][admin_level=2]->.ph;
(
  node["natural"="peak"](area.ph);
  way["natural"="peak"](area.ph);
  relation["natural"="peak"](area.ph);
);
out center tags;
"""

headers = {
    "User-Agent": "PhilippinesMountainsPortfolioProject/1.0"
}

response = requests.get(
    overpass_url,
    params={"data": query},
    headers=headers,
    timeout=180
)

response.raise_for_status()
data = response.json()

print("Total raw OSM peak elements:", len(data["elements"]))

Total raw OSM peak elements: 15847


In [81]:
#Count Name vs unname
elements = data["elements"]

named_count = 0
unnamed_count = 0

for element in elements:
    tags = element.get("tags", {})
    if tags.get("name"):
        named_count += 1
    else:
        unnamed_count += 1

print("Named peaks:", named_count)
print("Unnamed peaks:", unnamed_count)

Named peaks: 2510
Unnamed peaks: 13337


# Convert OSM response to dataframe

In [82]:
osm_rows = []

for element in data["elements"]:
    tags = element.get("tags", {})
    name = tags.get("name")

    if not name:
        continue

    lat = element.get("lat") or element.get("center", {}).get("lat")
    lon = element.get("lon") or element.get("center", {}).get("lon")

    osm_rows.append({
        "name": name,
        "lat": lat,
        "lon": lon,
        "coord": f"{lat}, {lon}" if lat and lon else None,
        "elev_raw": tags.get("ele"),
        "osm_id": element.get("id"),
        "osm_type": element.get("type"),
        "source": "OpenStreetMap",
        "osm_tags": str(tags)
    })

osm_df = pd.DataFrame(osm_rows)

print("Named OSM rows:", len(osm_df))
osm_df.head(20)

Named OSM rows: 2510


,name,lat,lon,coord,elev_raw,osm_id,osm_type,source,osm_tags
0,Mount Luho,11.976895,121.928908,"11.9768946, 121.928908",None,25767748,node,OpenStreetMap,"{'name': 'Mount Luho', 'natural': 'peak'}"
1,Mount Tapyas,12.004879,120.204814,"12.0048793, 120.2048141",163,188124904,node,OpenStreetMap,"{'ele': '163', 'name': 'Mount Tapyas', 'name:z..."
2,Mount Aminduen,11.104079,124.742689,"11.104079, 124.7426887",1200,312118713,node,OpenStreetMap,"{'alt_name': 'Alto Peak', 'ele': '1200', 'name..."
3,Mount Ragang,7.709741,124.534634,"7.7097407, 124.5346337",2640,319478139,node,OpenStreetMap,"{'ele': '2640', 'name': 'Mount Ragang', 'natur..."
4,Mount Akir Akir,7.347843,124.510765,"7.3478433, 124.5107652",970,319588211,node,OpenStreetMap,"{'ele': '970', 'name': 'Mount Akir Akir', 'nat..."
5,South Peak,15.196574,120.743496,"15.1965737, 120.7434961",970,319588221,node,OpenStreetMap,"{'alt_name': 'Arayat South Peak', 'ele': '970'..."
6,Atimbia,14.149219,121.365688,"14.1492186, 121.3656882",650,319588223,node,OpenStreetMap,"{'alt_name': 'Atimla', 'ele': '650', 'gns:dsg'..."
7,Bayaguitos,14.156814,121.395989,"14.1568141, 121.3959889",316,319588244,node,OpenStreetMap,"{'ele': '316', 'is_in:state': 'Laguna', 'name'..."
8,Mount Bonbon,10.899355,121.071159,"10.8993553, 121.0711593",247,319588281,node,OpenStreetMap,"{'ele': '247', 'gns:dsg': 'MT', 'gns:uni': '91..."
9,Cariliao,14.114919,120.765395,"14.1149192, 120.7653955",656,319588319,node,OpenStreetMap,"{'alt_name': 'Lantik', 'ele': '656', 'gns:dsg'..."


# Clean elevation

In [83]:
def clean_elevation(value):
    if value is None or pd.isna(value):
        return None

    value = str(value)
    number = re.search(r"[\d,.]+", value)

    if number:
        return float(number.group(0).replace(",", ""))

    return None

osm_df["elev"] = osm_df["elev_raw"].apply(clean_elevation)

osm_df[["name", "elev_raw", "elev", "lat", "lon"]].head(20)

,name,elev_raw,elev,lat,lon
0,Mount Luho,None,NaN,11.976895,121.928908
1,Mount Tapyas,163,163.0,12.004879,120.204814
2,Mount Aminduen,1200,1200.0,11.104079,124.742689
3,Mount Ragang,2640,2640.0,7.709741,124.534634
4,Mount Akir Akir,970,970.0,7.347843,124.510765
5,South Peak,970,970.0,15.196574,120.743496
6,Atimbia,650,650.0,14.149219,121.365688
7,Bayaguitos,316,316.0,14.156814,121.395989
8,Mount Bonbon,247,247.0,10.899355,121.071159
9,Cariliao,656,656.0,14.114919,120.765395


# Clean mountain names

In [84]:
def standardize_mountain_name(name):
    original_name = str(name).strip()

    if original_name == "" or original_name.lower() == "nan":
        return None, []

    lower = original_name.lower()

    alt_names = []

    # Convert Mt. or Mt to Mount
    if lower.startswith("mt. "):
        standardized = "Mount " + original_name[4:].strip()
        alt_names.append(original_name)
        return standardized, alt_names

    if lower.startswith("mt "):
        standardized = "Mount " + original_name[3:].strip()
        alt_names.append(original_name)
        return standardized, alt_names

    # Already starts with Mount
    if lower.startswith("mount "):
        return original_name, alt_names

    # Do not add Mount if it already ends with Hill or Peak
    if lower.endswith(" hill") or lower.endswith(" peak") or lower.endswith("hill") or lower.endswith("peak"):
        return original_name, alt_names

    # Otherwise, prefix with Mount
    standardized = "Mount " + original_name
    alt_names.append(original_name)

    return standardized, alt_names

In [85]:
#Apply It
osm_df[["name_standard", "alt_names"]] = osm_df["name"].apply(
    lambda x: pd.Series(standardize_mountain_name(x))
)

osm_df[["name", "name_standard", "alt_names"]].head(20)

,name,name_standard,alt_names
0,Mount Luho,Mount Luho,[]
1,Mount Tapyas,Mount Tapyas,[]
2,Mount Aminduen,Mount Aminduen,[]
3,Mount Ragang,Mount Ragang,[]
4,Mount Akir Akir,Mount Akir Akir,[]
5,South Peak,South Peak,[]
6,Atimbia,Mount Atimbia,[Atimbia]
7,Bayaguitos,Mount Bayaguitos,[Bayaguitos]
8,Mount Bonbon,Mount Bonbon,[]
9,Cariliao,Mount Cariliao,[Cariliao]


In [86]:
def clean_mountain_name(name):
    name = str(name).lower().strip()

    replacements = {
        "mount ": "",
        "mt. ": "",
        "mt ": "",
        "mnt. ": "",
        "mnt ": ""
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    name = re.sub(r"\s+", " ", name)
    return name.strip()

osm_df["name_clean"] = osm_df["name_standard"].apply(clean_mountain_name)

print("Named rows:", len(osm_df))
print("Unique cleaned names:", osm_df["name_clean"].nunique())

osm_df[["name", "name_standard", "name_clean", "alt_names", "elev", "lat", "lon"]].head(20)

Named rows: 2510
Unique cleaned names: 2405


,name,name_standard,name_clean,alt_names,elev,lat,lon
0,Mount Luho,Mount Luho,luho,[],NaN,11.976895,121.928908
1,Mount Tapyas,Mount Tapyas,tapyas,[],163.0,12.004879,120.204814
2,Mount Aminduen,Mount Aminduen,aminduen,[],1200.0,11.104079,124.742689
3,Mount Ragang,Mount Ragang,ragang,[],2640.0,7.709741,124.534634
4,Mount Akir Akir,Mount Akir Akir,akir akir,[],970.0,7.347843,124.510765
5,South Peak,South Peak,south peak,[],970.0,15.196574,120.743496
6,Atimbia,Mount Atimbia,atimbia,[Atimbia],650.0,14.149219,121.365688
7,Bayaguitos,Mount Bayaguitos,bayaguitos,[Bayaguitos],316.0,14.156814,121.395989
8,Mount Bonbon,Mount Bonbon,bonbon,[],247.0,10.899355,121.071159
9,Cariliao,Mount Cariliao,cariliao,[Cariliao],656.0,14.114919,120.765395


In [87]:
osm_metadata_df = osm_df.copy()

osm_metadata_df["name_original"] = osm_metadata_df["name"]
osm_metadata_df["name"] = osm_metadata_df["name_standard"]

# Deduplicate named peaks (1km)

In [88]:
#Check duplicates after standardized names
duplicate_names = (
    osm_df[osm_df.duplicated("name_clean", keep=False)]
    .sort_values("name_clean")
)

print("Total named rows:", len(osm_df))
print("Unique cleaned names:", osm_df["name_clean"].nunique())
print("Duplicate rows:", len(duplicate_names))

duplicate_names[[
    "name",
    "name_standard",
    "name_clean",
    "alt_names",
    "elev",
    "lat",
    "lon",
    "osm_id",
    "osm_type"
]].head(50)

Total named rows: 2510
Unique cleaned names: 2405
Duplicate rows: 190


,name,name_standard,name_clean,alt_names,elev,lat,lon,osm_id,osm_type
1816,Mount Agapang,Mount Agapang,agapang,[],NaN,17.516667,120.716667,332021617,node
665,Mount Agapang,Mount Agapang,agapang,[],NaN,17.520600,120.942000,332019737,node
1801,Mount Agudo,Mount Agudo,agudo,[],NaN,11.385983,123.003570,332021591,node
1025,Mount Agudo,Mount Agudo,agudo,[],NaN,9.600000,125.450000,332020507,node
1928,Mount Agudo,Mount Agudo,agudo,[],990.0,14.891656,120.109819,332021778,node
284,Mount Amuyao,Mount Amuyao,amuyao,[],NaN,16.866667,121.416667,332019196,node
744,Mount Amuyao,Mount Amuyao,amuyao,[],2701.0,17.011799,121.129596,332019851,node
732,Mount Bagsak,Mount Bagsak,bagsak,[],1142.0,5.889286,125.567968,332019822,node
1800,Mount Bagsak,Mount Bagsak,bagsak,[],590.0,6.028056,121.139444,332021590,node
1501,Mount Balabag,Mount Balabag,balabag,[],NaN,7.916667,124.666667,332021175,node


Coordinates Sanity Check

In [89]:
# Coordinates sanity check before province assignment

coord_check = osm_metadata_df.copy()

coord_check["lat"] = pd.to_numeric(coord_check["lat"], errors="coerce")
coord_check["lon"] = pd.to_numeric(coord_check["lon"], errors="coerce")

print("Total rows:", len(coord_check))
print("Missing lat:", coord_check["lat"].isna().sum())
print("Missing lon:", coord_check["lon"].isna().sum())

# Philippines rough bounding box
# Latitude: around 4 to 22
# Longitude: around 116 to 127
coord_check["coord_in_ph_bbox"] = (
    coord_check["lat"].between(4, 22)
    & coord_check["lon"].between(116, 127)
)

print("\nCoordinate inside PH bounding box:")
print(coord_check["coord_in_ph_bbox"].value_counts(dropna=False))

coord_check[
    ~coord_check["coord_in_ph_bbox"]
    | coord_check["lat"].isna()
    | coord_check["lon"].isna()
][
    [
        "name",
        "elev",
        "lat",
        "lon",
        "coord",
        "source",
        "osm_id",
        "osm_type"
    ]
].head(100)

Total rows: 2510
Missing lat: 0
Missing lon: 0

Coordinate inside PH bounding box:
coord_in_ph_bbox
True    2510
Name: count, dtype: int64


,name,elev,lat,lon,coord,source,osm_id,osm_type


In [90]:
#Deduplicate by name + location
import math

def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate distance in kilometers between two lat/lon points.
    """
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return None

    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [91]:
def deduplicate_by_name_and_location(df, distance_threshold_km=1):
    """
    Keeps same-name mountains if they are far apart.
    Drops likely duplicates only if same name_clean and within distance threshold.
    """

    kept_rows = []

    for name_clean, group in df.groupby("name_clean"):
        group = group.sort_values(
            by="elev",
            ascending=False,
            na_position="last"
        )

        kept_for_name = []

        for _, row in group.iterrows():
            is_duplicate = False

            for kept_row in kept_for_name:
                distance = haversine_km(
                    row["lat"], row["lon"],
                    kept_row["lat"], kept_row["lon"]
                )

                if distance is not None and distance <= distance_threshold_km:
                    is_duplicate = True
                    break

            if not is_duplicate:
                kept_for_name.append(row)

        kept_rows.extend(kept_for_name)

    return pd.DataFrame(kept_rows).reset_index(drop=True)

In [92]:
#Apply It
osm_dedup_df = deduplicate_by_name_and_location(
    osm_df,
    distance_threshold_km=1
)

print("Before dedup:", len(osm_df))
print("After location-aware dedup:", len(osm_dedup_df))
print("Unique cleaned names:", osm_dedup_df["name_clean"].nunique())

osm_dedup_df[[
    "name",
    "name_standard",
    "name_clean",
    "alt_names",
    "elev",
    "lat",
    "lon",
    "source"
]].head(20)

Before dedup: 2510
After location-aware dedup: 2505
Unique cleaned names: 2405


,name,name_standard,name_clean,alt_names,elev,lat,lon,source
0,Mount 387,Mount 387,387,[],777.0,15.900298,120.973771,OpenStreetMap
1,Mount Abao,Mount Abao,abao,[],2527.0,16.832307,120.914886,OpenStreetMap
2,Abel's Peak,Abel's Peak,abel's peak,[],1464.0,12.431130,122.573755,OpenStreetMap
3,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,[],370.0,5.675000,125.366944,OpenStreetMap
4,Mount Aboabo,Mount Aboabo,aboabo,[],NaN,9.147528,118.060024,OpenStreetMap
5,Abong-abong Peak,Abong-abong Peak,abong-abong peak,[],885.0,6.501684,121.984999,OpenStreetMap
6,Mount Abongabong,Mount Abongabong,abongabong,[],NaN,13.395455,120.750376,OpenStreetMap
7,Mount Aborlan,Mount Aborlan,aborlan,[],745.0,9.510556,118.467222,OpenStreetMap
8,Mount Abra de Ilog,Mount Abra de Ilog,abra de ilog,[],1018.0,13.454526,120.673642,OpenStreetMap
9,Mount Abriringan,Mount Abriringan,abriringan,[],659.0,18.299000,122.270372,OpenStreetMap


In [93]:
## Give unique ID to mountains
osm_dedup_df["mountain_id"] = (
    osm_dedup_df["name_clean"]
    + "_"
    + osm_dedup_df["lat"].round(4).astype(str)
    + "_"
    + osm_dedup_df["lon"].round(4).astype(str)
)

# Format metadata columns

In [94]:
# Make a metadata-style dataframe from the deduplicated OSM data
osm_metadata_df = osm_dedup_df.copy()

# Keep original OSM name for traceability
osm_metadata_df["name_original"] = osm_metadata_df["name"]

# Use standardized name as the official metadata name
osm_metadata_df["name"] = osm_metadata_df["name_standard"]

# Required/optional metadata fields from the standard
osm_metadata_df["prom"] = None          # OSM usually does not provide prominence
osm_metadata_df["is_volc"] = None       # to be enriched later
osm_metadata_df["prov"] = None          # to be assigned later using GIS/province boundaries
osm_metadata_df["region"] = None        # to be assigned from province lookup
osm_metadata_df["isl_grp"] = None       # to be assigned from province lookup
# alt_names already exists from Cell 6



# Helper/source fields
osm_metadata_df["detail_url"] = None
osm_metadata_df["coord_source"] = "OpenStreetMap"
osm_metadata_df["data_quality_notes"] = ""

# Make sure coord exists
osm_metadata_df["coord"] = (
    osm_metadata_df["lat"].astype(str) + ", " + osm_metadata_df["lon"].astype(str)
)


# Add marker_size based on elevation
def assign_marker_size(elev):
    if pd.isna(elev):
        return None
    elif elev <= 500:
        return "small"
    elif elev < 1500:
        return "medium"
    else:
        return "large"

osm_metadata_df["marker_size"] = osm_metadata_df["elev"].apply(assign_marker_size)


# Keep clean column order
osm_metadata_df = osm_metadata_df[[
    "mountain_id",
    "name",
    "name_original",
    "name_clean",
    "elev",
    "prom",
    "marker_size",
    "coord",
    "lat",
    "lon",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "alt_names",
    "detail_url",
    "source",
    "coord_source",
    "elev_raw",
    "osm_id",
    "osm_type",
    "osm_tags",
    "data_quality_notes"
]]
osm_metadata_df.head(20)

,mountain_id,name,name_original,name_clean,elev,prom,marker_size,coord,lat,lon,...,isl_grp,alt_names,detail_url,source,coord_source,elev_raw,osm_id,osm_type,osm_tags,data_quality_notes
0,387_15.9003_120.9738,Mount 387,Mount 387,387,777.0,None,medium,"15.900298, 120.9737706",15.900298,120.973771,...,None,[],None,OpenStreetMap,OpenStreetMap,777,5269019421,node,"{'alt_name': 'Mount Batong Amat', 'ele': '777'...",
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,abao,2527.0,None,large,"16.8323069, 120.9148864",16.832307,120.914886,...,None,[],None,OpenStreetMap,OpenStreetMap,2527,332021090,node,"{'ele': '2527', 'gns:dsg': 'MT', 'gns:uni': '-...",
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,abel's peak,1464.0,None,medium,"12.4311298, 122.5737552",12.431130,122.573755,...,None,[],None,OpenStreetMap,OpenStreetMap,1464,4231475039,node,"{'ele': '1464', 'name': ""Abel's Peak"", 'natura...",
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,370.0,None,small,"5.675, 125.366944",5.675000,125.366944,...,None,[],None,OpenStreetMap,OpenStreetMap,370,332020520,node,"{'ele': '370', 'gns:dsg': 'MT', 'gns:uni': '-3...",
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,aboabo,NaN,None,None,"9.1475284, 118.060024",9.147528,118.060024,...,None,[],None,OpenStreetMap,OpenStreetMap,None,332021655,node,"{'gns:dsg': 'MT', 'gns:uni': '-3321795', 'is_i...",
5,abong-abong peak_6.5017_121.985,Abong-abong Peak,Abong-abong Peak,abong-abong peak,885.0,None,medium,"6.5016836, 121.9849993",6.501684,121.984999,...,None,[],None,OpenStreetMap,OpenStreetMap,885,332019090,node,"{'alt_name': 'Abongabong Peak', 'ele': '885', ...",
6,abongabong_13.3955_120.7504,Mount Abongabong,Mount Abongabong,abongabong,NaN,None,None,"13.3954548, 120.7503755",13.395455,120.750376,...,None,[],None,OpenStreetMap,OpenStreetMap,None,332021575,node,"{'gns:dsg': 'MT', 'gns:uni': '-3321813', 'is_i...",
7,aborlan_9.5106_118.4672,Mount Aborlan,Mount Aborlan,aborlan,745.0,None,medium,"9.510556, 118.467222",9.510556,118.467222,...,None,[],None,OpenStreetMap,OpenStreetMap,745,332018924,node,"{'ele': '745', 'gns:dsg': 'MT', 'gns:uni': '-3...",
8,abra de ilog_13.4545_120.6736,Mount Abra de Ilog,Mount Abra de Ilog,abra de ilog,1018.0,None,medium,"13.4545261, 120.673642",13.454526,120.673642,...,None,[],None,OpenStreetMap,OpenStreetMap,1018,332019826,node,"{'ele': '1018', 'gns:dsg': 'MT', 'gns:uni': '9...",
9,abriringan_18.299_122.2704,Mount Abriringan,Mount Abriringan,abriringan,659.0,None,medium,"18.2989998, 122.2703722",18.299000,122.270372,...,None,[],None,OpenStreetMap,OpenStreetMap,659,332018923,node,"{'ele': '659', 'gns:dsg': 'MT', 'gns:uni': '-3...",


In [95]:
#Check marker size distribution
osm_metadata_df["marker_size"].value_counts(dropna=False)

,count
marker_size,
None,1138
medium,698
small,485
large,184


In [96]:
osm_metadata_df.groupby("marker_size")["elev"].agg(["count", "min", "max"])
#small   max <= 500
#medium  min > 500 and max < 1500
#large   min >= 1500

,count,min,max
marker_size,,,
large,184,1500.0,2954.0
medium,698,501.0,1493.0
small,485,0.0,500.0


# Assign Province using repo file

In [97]:
import geopandas as gpd
import pandas as pd

repo_boundary_path = "/content/drive/MyDrive/Colab Notebooks/Philippine_official boundary dataframe.geojson"

repo_province_polygons_gdf = gpd.read_file(repo_boundary_path)

repo_province_polygons_gdf.head()

,prov,prov_psgc_code,region,region_name,region_code,isl_grp,prov_gis_original,region_gis_original,source_file,admin_note,geometry
0,Ilocos Norte,102800000.0,Region I,Region I (Ilocos Region),100000000.0,Luzon,Ilocos Norte,Ilocos Region (Region I),philippines-json-maps/2011/geojson/provinces/m...,None,"POLYGON ((120.53491 17.70374, 120.51599 17.722..."
1,Ilocos Sur,102900000.0,Region I,Region I (Ilocos Region),100000000.0,Luzon,Ilocos Sur,Ilocos Region (Region I),philippines-json-maps/2011/geojson/provinces/m...,None,"POLYGON ((120.53491 17.70374, 120.4904 17.6016..."
2,La Union,103300000.0,Region I,Region I (Ilocos Region),100000000.0,Luzon,La Union,Ilocos Region (Region I),philippines-json-maps/2011/geojson/provinces/m...,None,"POLYGON ((120.58044 16.65924, 120.54189 16.607..."
3,Pangasinan,105500000.0,Region I,Region I (Ilocos Region),100000000.0,Luzon,Pangasinan,Ilocos Region (Region I),philippines-json-maps/2011/geojson/provinces/m...,None,"MULTIPOLYGON (((119.98972 16.34889, 120.012 16..."
4,Zamboanga del Norte,907200000.0,Region IX,Region IX (Zamboanga Peninsula),900000000.0,Mindanao,Zamboanga del Norte,Zamboanga Peninsula (Region IX),philippines-json-maps/2011/geojson/provinces/m...,None,"POLYGON ((123.54957 8.21537, 123.38226 8.21482..."


In [98]:
repo_province_polygons_gdf.columns.tolist()

['prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'isl_grp',
 'prov_gis_original',
 'region_gis_original',
 'source_file',
 'admin_note',
 'geometry']

In [99]:
#Convert OSM mountains into points
mountains_gdf = gpd.GeoDataFrame(
    osm_metadata_df.copy(),
    geometry=gpd.points_from_xy(
        osm_metadata_df["lon"],
        osm_metadata_df["lat"]
    ),
    crs="EPSG:4326"
)

mountains_gdf.head()

,mountain_id,name,name_original,name_clean,elev,prom,marker_size,coord,lat,lon,...,alt_names,detail_url,source,coord_source,elev_raw,osm_id,osm_type,osm_tags,data_quality_notes,geometry
0,387_15.9003_120.9738,Mount 387,Mount 387,387,777.0,None,medium,"15.900298, 120.9737706",15.900298,120.973771,...,[],None,OpenStreetMap,OpenStreetMap,777,5269019421,node,"{'alt_name': 'Mount Batong Amat', 'ele': '777'...",,POINT (120.97377 15.9003)
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,abao,2527.0,None,large,"16.8323069, 120.9148864",16.832307,120.914886,...,[],None,OpenStreetMap,OpenStreetMap,2527,332021090,node,"{'ele': '2527', 'gns:dsg': 'MT', 'gns:uni': '-...",,POINT (120.91489 16.83231)
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,abel's peak,1464.0,None,medium,"12.4311298, 122.5737552",12.431130,122.573755,...,[],None,OpenStreetMap,OpenStreetMap,1464,4231475039,node,"{'ele': '1464', 'name': ""Abel's Peak"", 'natura...",,POINT (122.57376 12.43113)
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,370.0,None,small,"5.675, 125.366944",5.675000,125.366944,...,[],None,OpenStreetMap,OpenStreetMap,370,332020520,node,"{'ele': '370', 'gns:dsg': 'MT', 'gns:uni': '-3...",,POINT (125.36694 5.675)
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,aboabo,NaN,None,None,"9.1475284, 118.060024",9.147528,118.060024,...,[],None,OpenStreetMap,OpenStreetMap,None,332021655,node,"{'gns:dsg': 'MT', 'gns:uni': '-3321795', 'is_i...",,POINT (118.06002 9.14753)


In [100]:
#Spatial join using repo boundary file
mountains_gdf_clean = mountains_gdf.drop(
    columns=[
        "prov",
        "region",
        "region_name",
        "region_code",
        "isl_grp",
        "prov_psgc_code",
        "admin_note"
    ],
    errors="ignore"
)

mountains_with_repo_location = gpd.sjoin(
    mountains_gdf_clean,
    repo_province_polygons_gdf,
    how="left",
    predicate="within"
)

mountains_with_repo_location[[
    "name",
    "lat",
    "lon",
    "prov",
    "region",
    "region_name",
    "isl_grp",
    "prov_gis_original",
    "admin_note"
]].head(30)


,name,lat,lon,prov,region,region_name,isl_grp,prov_gis_original,admin_note
0,Mount 387,15.900298,120.973771,Nueva Ecija,Region III,Region III (Central Luzon),Luzon,Nueva Ecija,None
1,Mount Abao,16.832307,120.914886,Ifugao,CAR,Cordillera Administrative Region (CAR),Luzon,Ifugao,None
2,Abel's Peak,12.431130,122.573755,Romblon,MIMAROPA,MIMAROPA Region,Luzon,Romblon,None
3,Mount Abnataclayong,5.675000,125.366944,Sarangani,Region XII,Region XII (SOCCSKSARGEN),Mindanao,Sarangani,None
4,Mount Aboabo,9.147528,118.060024,Palawan,MIMAROPA,MIMAROPA Region,Luzon,Palawan,None
5,Abong-abong Peak,6.501684,121.984999,Basilan,BARMM,Bangsamoro Autonomous Region in Muslim Mindana...,Mindanao,Basilan,None
6,Mount Abongabong,13.395455,120.750376,Occidental Mindoro,MIMAROPA,MIMAROPA Region,Luzon,Occidental Mindoro,None
7,Mount Aborlan,9.510556,118.467222,Palawan,MIMAROPA,MIMAROPA Region,Luzon,Palawan,None
8,Mount Abra de Ilog,13.454526,120.673642,Occidental Mindoro,MIMAROPA,MIMAROPA Region,Luzon,Occidental Mindoro,None
9,Mount Abriringan,18.299000,122.270372,Cagayan,Region II,Region II (Cagayan Valley),Luzon,Cagayan,None


In [101]:
#Check Match Rate
valid_location = (
    mountains_with_repo_location["prov"].notna()
    |
    (
        mountains_with_repo_location["prov_gis_original"].isin([
            "Metropolitan Manila",
            "Maguindanao",
            "Shariff Kabunsuan"
        ])
        & mountains_with_repo_location["region"].notna()
    )
)

total_mountains = len(mountains_with_repo_location)
matched_count = valid_location.sum()
missing_count = total_mountains - matched_count

print("Total mountains:", total_mountains)
print("Matched to province/admin area:", matched_count)
print("Missing location:", missing_count)
print("Match rate:", round(matched_count / total_mountains * 100, 2), "%")

Total mountains: 2512
Matched to province/admin area: 2433
Missing location: 79
Match rate: 96.86 %


In [102]:
#Inspect Missing Rows
missing_repo_location_df = mountains_with_repo_location[
    ~valid_location
]

missing_repo_location_df[[
    "name",
    "lat",
    "lon",
    "coord",
    "source"
]].head(50)

,name,lat,lon,coord,source
50,Mount Aharung,20.283333,121.850000,"20.283333, 121.85",OpenStreetMap
81,Mount Ambagyew,16.405556,120.854167,"16.405556, 120.854167",OpenStreetMap
150,Babayon Point Hill,13.207553,124.061042,"13.2075528, 124.0610423",OpenStreetMap
174,Bagnusan Hill,6.586311,125.466192,"6.5863107, 125.4661919",OpenStreetMap
201,Mount Balanayo,5.627222,125.325556,"5.627222, 125.325556",OpenStreetMap
269,Bantayan Hill,12.172035,123.240077,"12.1720355, 123.2400765",OpenStreetMap
279,Barren Hill,11.033333,119.333333,"11.033333, 119.333333",OpenStreetMap
310,Mount Bayang,11.043774,122.943227,"11.0437743, 122.9432267",OpenStreetMap
392,Mount Bolod,6.262059,121.613333,"6.2620586, 121.613333",OpenStreetMap
411,Bubuan Hill,6.205100,120.963700,"6.2051, 120.9637",OpenStreetMap


In [103]:
# Label unmatched admin/province rows as Unassigned

missing_admin_mask = (
    mountains_with_repo_location["prov"].isna()
    | mountains_with_repo_location["region"].isna()
    | mountains_with_repo_location["isl_grp"].isna()
)

mountains_with_repo_location.loc[missing_admin_mask, "prov"] = "Unassigned"
mountains_with_repo_location.loc[missing_admin_mask, "region"] = "Unassigned"
mountains_with_repo_location.loc[missing_admin_mask, "region_name"] = "Unassigned"
mountains_with_repo_location.loc[missing_admin_mask, "isl_grp"] = "Unassigned"

mountains_with_repo_location.loc[missing_admin_mask, "admin_note"] = (
    "Coordinate exists but did not match province boundary"
)

print("Unassigned rows:", missing_admin_mask.sum())

mountains_with_repo_location[
    missing_admin_mask
][
    ["name", "lat", "lon", "prov", "region", "region_name", "isl_grp", "admin_note"]
].head(50)

Unassigned rows: 105


,name,lat,lon,prov,region,region_name,isl_grp,admin_note
50,Mount Aharung,20.283333,121.850000,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
59,Mount Alas,6.895364,124.365224,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
81,Mount Ambagyew,16.405556,120.854167,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
150,Babayon Point Hill,13.207553,124.061042,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
174,Bagnusan Hill,6.586311,125.466192,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
201,Mount Balanayo,5.627222,125.325556,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
257,Banisilan Hill,7.333333,124.216667,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
269,Bantayan Hill,12.172035,123.240077,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
279,Barren Hill,11.033333,119.333333,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...
310,Mount Bayang,11.043774,122.943227,Unassigned,Unassigned,Unassigned,Unassigned,Coordinate exists but did not match province b...


In [104]:
mountains_with_repo_location.columns.tolist()

['mountain_id',
 'name',
 'name_original',
 'name_clean',
 'elev',
 'prom',
 'marker_size',
 'coord',
 'lat',
 'lon',
 'is_volc',
 'alt_names',
 'detail_url',
 'source',
 'coord_source',
 'elev_raw',
 'osm_id',
 'osm_type',
 'osm_tags',
 'data_quality_notes',
 'geometry',
 'index_right',
 'prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'isl_grp',
 'prov_gis_original',
 'region_gis_original',
 'source_file',
 'admin_note']

In [105]:
mountains_with_repo_location = mountains_with_repo_location [['mountain_id',
 'name',
 'name_original',
 'source',
 'name_clean',
 'alt_names',
 'elev',
 'prom',
 'marker_size',
 'coord',
 'lat',
 'lon',
 'is_volc',
 'isl_grp',
 'elev_raw',
 'osm_id',
 'osm_type',
 'osm_tags',
 'data_quality_notes',
 'geometry',
 'index_right',
 'prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'prov_gis_original',
 'region_gis_original',
 'source_file',
 'source',
 'coord_source',
 'admin_note',
  'detail_url']]

In [106]:
mountains_with_repo_location.to_csv(
    "philippines_mountains_osm_with_repo_province_region_island.csv",
    index=False
)

print("Saved rows:", len(mountains_with_repo_location))

Saved rows: 2512


In [107]:
print("Columns in osm_metadata_df but missing in missing_repo_location_df:")
print([col for col in osm_metadata_df.columns if col not in missing_repo_location_df.columns])

print("\nColumns in missing_repo_location_df but not in osm_metadata_df:")
print([col for col in missing_repo_location_df.columns if col not in osm_metadata_df.columns])

Columns in osm_metadata_df but missing in missing_repo_location_df:
[]

Columns in missing_repo_location_df but not in osm_metadata_df:
['geometry', 'index_right', 'prov_psgc_code', 'region_name', 'region_code', 'prov_gis_original', 'region_gis_original', 'source_file', 'admin_note']


# Validate with QuickMap Tools (coordinates)

In [108]:
quickmap_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/mountain-peaks-philippines_quickmap.csv")

quickmap_df.head()

,name,latitude,longitude,country,region,elevation_m,population,feature_type,geoname_id
0,Mount Apo,6.98781,125.27109,Philippines,Soccsksargen,2954.0,0,mountain,1730340
1,Mount Katanglad,8.11833,124.92222,Philippines,Northern Mindanao,2938.0,0,mountain,1709118
2,Mount Pulog,16.59778,120.89914,Philippines,Cordillera,2934.0,0,mountain,1692545
3,Mount Ragang,7.70986,124.53466,Philippines,Autonomous Region in Muslim Mindanao,2815.0,0,mountain,1691947
4,Mount Piapayungan,7.69043,124.50671,Philippines,Soccsksargen,2815.0,0,mountain,1693990


In [109]:
#Standardize QuickMap column names
quickmap_clean = quickmap_df.copy()

quickmap_clean = quickmap_clean.rename(columns={
    "longitude": "lon",
    "latitude": "lat",
    "elevation_m": "elev",
    "name": "name"
})

quickmap_clean.head()

,name,lat,lon,country,region,elev,population,feature_type,geoname_id
0,Mount Apo,6.98781,125.27109,Philippines,Soccsksargen,2954.0,0,mountain,1730340
1,Mount Katanglad,8.11833,124.92222,Philippines,Northern Mindanao,2938.0,0,mountain,1709118
2,Mount Pulog,16.59778,120.89914,Philippines,Cordillera,2934.0,0,mountain,1692545
3,Mount Ragang,7.70986,124.53466,Philippines,Autonomous Region in Muslim Mindanao,2815.0,0,mountain,1691947
4,Mount Piapayungan,7.69043,124.50671,Philippines,Soccsksargen,2815.0,0,mountain,1693990


In [110]:
quickmap_clean[["name", "lat", "lon", "elev"]].head()

,name,lat,lon,elev
0,Mount Apo,6.98781,125.27109,2954.0
1,Mount Katanglad,8.11833,124.92222,2938.0
2,Mount Pulog,16.59778,120.89914,2934.0
3,Mount Ragang,7.70986,124.53466,2815.0
4,Mount Piapayungan,7.69043,124.50671,2815.0


In [111]:
#Clean mountain names for matching
def clean_mountain_name(name):
    name = str(name).lower().strip()

    replacements = {
        "mount ": "",
        "mt. ": "",
        "mt ": "",
        "mnt. ": "",
        "mnt ": ""
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    name = re.sub(r"\s+", " ", name)
    return name.strip()

In [112]:
quickmap_clean["name_clean"] = quickmap_clean["name"].apply(clean_mountain_name)

mountains_with_repo_location["name_clean"] = mountains_with_repo_location["name"].apply(
    clean_mountain_name
)

In [113]:
#Merge QuickMap with your OSM master
quickmap_validation = mountains_with_repo_location.merge(
    quickmap_clean[["name", "name_clean", "lat", "lon", "elev"]],
    on="name_clean",
    how="left",
    suffixes=("", "_quickmap")
)
quickmap_validation[[
    "name",
    "name_quickmap",
    "elev",
    "elev_quickmap",
    "lat",
    "lon",
    "lat_quickmap",
    "lon_quickmap",
    "prov",
    "region",
    "isl_grp"
]].head(20)

,name,name_quickmap,elev,elev_quickmap,lat,lon,lat_quickmap,lon_quickmap,prov,region,isl_grp
0,Mount 387,NaN,777.0,NaN,15.900298,120.973771,NaN,NaN,Nueva Ecija,Region III,Luzon
1,Mount Abao,Mount Abao,2527.0,2524.0,16.832307,120.914886,16.83330,120.91670,Ifugao,CAR,Luzon
2,Abel's Peak,NaN,1464.0,NaN,12.431130,122.573755,NaN,NaN,Romblon,MIMAROPA,Luzon
3,Mount Abnataclayong,Mount Abnataclayong,370.0,370.0,5.675000,125.366944,5.67500,125.36694,Sarangani,Region XII,Mindanao
4,Mount Aboabo,NaN,NaN,NaN,9.147528,118.060024,NaN,NaN,Palawan,MIMAROPA,Luzon
5,Abong-abong Peak,NaN,885.0,NaN,6.501684,121.984999,NaN,NaN,Basilan,BARMM,Mindanao
6,Mount Abongabong,NaN,NaN,NaN,13.395455,120.750376,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon
7,Mount Aborlan,Mount Aborlan,745.0,775.0,9.510556,118.467222,9.51080,118.46690,Palawan,MIMAROPA,Luzon
8,Mount Abra de Ilog,Mount Abra de Ilog,1018.0,1018.0,13.454526,120.673642,13.45167,120.67139,Occidental Mindoro,MIMAROPA,Luzon
9,Mount Abriringan,Mount Abriringan,659.0,659.0,18.299000,122.270372,18.29967,122.26915,Cagayan,Region II,Luzon


In [114]:
#Add match flag
quickmap_validation["quickmap_match"] = quickmap_validation["name_quickmap"].notna()

print("Total OSM mountains:", len(quickmap_validation))
print("Matched with QuickMap:", quickmap_validation["quickmap_match"].sum())
print("Not matched:", (~quickmap_validation["quickmap_match"]).sum())
print(
    "Match rate:",
    round(quickmap_validation["quickmap_match"].mean() * 100, 2),
    "%"
)

Total OSM mountains: 2625
Matched with QuickMap: 900
Not matched: 1725
Match rate: 34.29 %


In [115]:
#Compare coordinates using distance
import math

def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return None

    R = 6371

    lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])

    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [116]:
quickmap_validation["quickmap_distance_km"] = quickmap_validation.apply(
    lambda row: haversine_km(
        row["lat"], row["lon"],
        row["lat_quickmap"], row["lon_quickmap"]
    ),
    axis=1
)

quickmap_validation[[
    "name",
    "name_quickmap",
    "quickmap_distance_km",
    "elev",
    "elev_quickmap"
]].sort_values("quickmap_distance_km", ascending=False).head(20)

,name,name_quickmap,quickmap_distance_km,elev,elev_quickmap
365,Mount Binuang,Mount Binuang,1086.276307,NaN,1095.0
426,Mount Buga,Mount Buga,1053.981200,1435.0,549.0
2135,Sharp Peak,Sharp Peak,1006.764416,NaN,585.0
2265,Spike Peak,Spike Peak,988.515678,NaN,905.0
2136,Sharp Peak,Sharp Peak,975.177964,NaN,NaN
2143,Sharp Peak,Sharp Peak,968.333789,NaN,NaN
2142,Sharp Peak,Sharp Peak,966.126957,NaN,NaN
2133,Sharp Peak,Sharp Peak,945.281432,NaN,1047.0
2437,Thumb Peak,Thumb Peak,937.978578,NaN,1268.0
907,Mount Guiwan,Mount Guiwan,934.244743,16.0,1910.0


In [117]:
#Add validation notes
def quickmap_validation_note(row):
    if not row["quickmap_match"]:
        return "Not found in QuickMap/GeoNames"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] <= 1:
        return "QuickMap match: coordinate very close"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] <= 5:
        return "QuickMap match: coordinate close"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] > 5:
        return "QuickMap match: coordinate differs; review"

    return "QuickMap match: missing coordinate comparison"

quickmap_validation["quickmap_validation_note"] = quickmap_validation.apply(
    quickmap_validation_note,
    axis=1
)

quickmap_validation[[
    "name",
    "name_quickmap",
    "quickmap_match",
    "quickmap_distance_km",
    "quickmap_validation_note"
]].head(30)

,name,name_quickmap,quickmap_match,quickmap_distance_km,quickmap_validation_note
0,Mount 387,NaN,False,NaN,Not found in QuickMap/GeoNames
1,Mount Abao,Mount Abao,True,0.222378,QuickMap match: coordinate very close
2,Abel's Peak,NaN,False,NaN,Not found in QuickMap/GeoNames
3,Mount Abnataclayong,Mount Abnataclayong,True,0.000443,QuickMap match: coordinate very close
4,Mount Aboabo,NaN,False,NaN,Not found in QuickMap/GeoNames
5,Abong-abong Peak,NaN,False,NaN,Not found in QuickMap/GeoNames
6,Mount Abongabong,NaN,False,NaN,Not found in QuickMap/GeoNames
7,Mount Aborlan,Mount Aborlan,True,0.044532,QuickMap match: coordinate very close
8,Mount Abra de Ilog,Mount Abra de Ilog,True,0.400214,QuickMap match: coordinate very close
9,Mount Abriringan,Mount Abriringan,True,0.149004,QuickMap match: coordinate very close


In [118]:
quickmap_validation.to_csv(
    "philippines_mountains_validated_with_quickmap.csv",
    index=False
)

print("Saved rows:", len(quickmap_validation))

Saved rows: 2625


In [119]:
#Validation Quality
quickmap_validation["quickmap_validation_note"].value_counts()

,count
quickmap_validation_note,
Not found in QuickMap/GeoNames,1725
QuickMap match: coordinate very close,616
QuickMap match: coordinate differs; review,183
QuickMap match: coordinate close,101


In [120]:
total_rows = len(quickmap_validation)
matched_rows = quickmap_validation["quickmap_match"].sum()

print("Total OSM mountains:", total_rows)
print("Matched with QuickMap:", matched_rows)
print("Not matched:", total_rows - matched_rows)
print("QuickMap match rate:", round(matched_rows / total_rows * 100, 2), "%")

Total OSM mountains: 2625
Matched with QuickMap: 900
Not matched: 1725
QuickMap match rate: 34.29 %


In [121]:
#Suspicious Rows
quickmap_review_df = quickmap_validation[
    quickmap_validation["quickmap_validation_note"] == "QuickMap match: coordinate differs; review"
]

quickmap_review_df[[
    "name",
    "name_quickmap",
    "lat",
    "lon",
    "lat_quickmap",
    "lon_quickmap",
    "quickmap_distance_km",
    "prov",
    "region",
    "isl_grp"
]].sort_values("quickmap_distance_km", ascending=False).head(30)

,name,name_quickmap,lat,lon,lat_quickmap,lon_quickmap,quickmap_distance_km,prov,region,isl_grp
365,Mount Binuang,Mount Binuang,5.134722,119.950000,14.77509,121.55705,1086.276307,Tawi-Tawi,BARMM,Mindanao
426,Mount Buga,Mount Buga,16.719466,120.643378,7.59570,123.27500,1053.981200,Benguet,CAR,Luzon
2135,Sharp Peak,Sharp Peak,7.904167,116.983889,13.62210,124.13310,1006.764416,Palawan,MIMAROPA,Luzon
2265,Spike Peak,Spike Peak,17.122262,120.636585,8.74151,117.59079,988.515678,Ilocos Sur,Region I,Luzon
2136,Sharp Peak,Sharp Peak,7.904167,116.983889,11.18333,125.23333,975.177964,Palawan,MIMAROPA,Luzon
2143,Sharp Peak,Sharp Peak,7.904167,116.983889,5.81667,125.50000,968.333789,Palawan,MIMAROPA,Luzon
2142,Sharp Peak,Sharp Peak,7.904167,116.983889,5.96667,125.51667,966.126957,Palawan,MIMAROPA,Luzon
2133,Sharp Peak,Sharp Peak,7.904167,116.983889,12.78150,124.06420,945.281432,Palawan,MIMAROPA,Luzon
2437,Thumb Peak,Thumb Peak,17.500000,122.133333,9.79194,118.60389,937.978578,Isabela,Region II,Luzon
907,Mount Guiwan,Mount Guiwan,7.678251,122.826303,15.94420,121.28710,934.244743,Zamboanga Sibugay,Region IX,Mindanao


In [122]:
def quickmap_status(row):
    if not row["quickmap_match"]:
        return "Not found in QuickMap"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] <= 5:
        return "Validated by QuickMap"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] > 5:
        return "Review QuickMap coordinate mismatch"

    return "QuickMap match incomplete"

quickmap_validation["quickmap_status"] = quickmap_validation.apply(
    quickmap_status,
    axis=1
)

quickmap_validation["quickmap_status"].value_counts()

,count
quickmap_status,
Not found in QuickMap,1725
Validated by QuickMap,717
Review QuickMap coordinate mismatch,183


In [123]:
def quickmap_confidence(row):
    if not row["quickmap_match"]:
        return "No QuickMap match"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] <= 5:
        return "High confidence"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] <= 25:
        return "Medium confidence"

    if pd.notna(row["quickmap_distance_km"]) and row["quickmap_distance_km"] > 25:
        return "Low confidence - likely wrong name match"

    return "Incomplete"

quickmap_validation["quickmap_confidence"] = quickmap_validation.apply(
    quickmap_confidence,
    axis=1
)

quickmap_validation["quickmap_confidence"].value_counts()

,count
quickmap_confidence,
No QuickMap match,1725
High confidence,717
Low confidence - likely wrong name match,158
Medium confidence,25


In [124]:
# Fill missing elev directly from High confidence QuickMap matches

quickmap_validation["elev"] = pd.to_numeric(
    quickmap_validation["elev"],
    errors="coerce"
)

quickmap_validation["elev_quickmap"] = pd.to_numeric(
    quickmap_validation["elev_quickmap"],
    errors="coerce"
)

fill_quickmap_elev_mask = (
    quickmap_validation["elev"].isna()
    & quickmap_validation["elev_quickmap"].notna()
    & (quickmap_validation["quickmap_confidence"] == "High confidence")
)

print("Missing elev before:", quickmap_validation["elev"].isna().sum())
print("Rows to fill from QuickMap:", fill_quickmap_elev_mask.sum())

quickmap_validation.loc[
    fill_quickmap_elev_mask,
    "elev"
] = quickmap_validation.loc[
    fill_quickmap_elev_mask,
    "elev_quickmap"
]

print("Missing elev after:", quickmap_validation["elev"].isna().sum())

Missing elev before: 1214
Rows to fill from QuickMap: 143
Missing elev after: 1071


In [125]:
# Recalculate marker_size after QuickMap elevation fill

def assign_marker_size(elev):
    if pd.isna(elev):
        return None
    elif elev <= 500:
        return "small"
    elif elev < 1500:
        return "medium"
    else:
        return "large"

quickmap_validation["marker_size"] = quickmap_validation["elev"].apply(assign_marker_size)

quickmap_validation["marker_size"].value_counts(dropna=False)

,count
marker_size,
None,1071
medium,791
small,550
large,213


In [126]:
quickmap_validation.to_csv(
    "philippines_mountains_validated_with_quickmap.csv",
    index=False
)

print("Saved rows:", len(quickmap_validation))

Saved rows: 2625


In [127]:
print("Columns in osm_metadata_df but missing in quickmap_validation:")
print([col for col in osm_metadata_df.columns if col not in quickmap_validation.columns])

print("\nColumns in quickmap_validation but not in osm_metadata_df:")
print([col for col in quickmap_validation.columns if col not in osm_metadata_df.columns])

Columns in osm_metadata_df but missing in quickmap_validation:
[]

Columns in quickmap_validation but not in osm_metadata_df:
['geometry', 'index_right', 'prov_psgc_code', 'region_name', 'region_code', 'prov_gis_original', 'region_gis_original', 'source_file', 'admin_note', 'name_quickmap', 'lat_quickmap', 'lon_quickmap', 'elev_quickmap', 'quickmap_match', 'quickmap_distance_km', 'quickmap_validation_note', 'quickmap_status', 'quickmap_confidence']


# Deduplicate again (with Quickmap validation now)

1. First dedup: strict coordinate-only early pass — already done
2. Admin assignment — already done
3. QuickMap elevation fill + marker recalculation — you just did
4. Second dedup: same name + admin area + distance + QuickMap

In [128]:
# Second dedup pass:
# same name_clean + same admin area + coordinates within 1 km
# keeps the best row and drops only very close duplicates

import numpy as np
import pandas as pd

dedup_work = quickmap_validation.copy()

dedup_work["lat"] = pd.to_numeric(dedup_work["lat"], errors="coerce")
dedup_work["lon"] = pd.to_numeric(dedup_work["lon"], errors="coerce")
dedup_work["elev"] = pd.to_numeric(dedup_work["elev"], errors="coerce")
dedup_work["quickmap_distance_km"] = pd.to_numeric(
    dedup_work["quickmap_distance_km"],
    errors="coerce"
)

original_columns = quickmap_validation.columns.tolist()

def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def quickmap_rank(confidence):
    confidence = str(confidence).lower()

    if "high confidence" in confidence:
        return 3
    elif "medium confidence" in confidence:
        return 2
    elif "no quickmap match" in confidence:
        return 1
    elif "low confidence" in confidence:
        return 0
    else:
        return 1


dedup_work["_quickmap_rank"] = dedup_work["quickmap_confidence"].apply(quickmap_rank)

dedup_work["_quality_score"] = (
    dedup_work["_quickmap_rank"] * 100
    + dedup_work["elev"].notna().astype(int) * 20
    + dedup_work["marker_size"].notna().astype(int) * 10
    + dedup_work["prov"].notna().astype(int) * 5
    + dedup_work["region"].notna().astype(int) * 5
    + dedup_work["isl_grp"].notna().astype(int) * 5
)

# Do not auto-dedup unassigned admin rows
safe_admin_mask = (
    dedup_work["name_clean"].notna()
    & dedup_work["prov"].notna()
    & dedup_work["region"].notna()
    & dedup_work["isl_grp"].notna()
    & (dedup_work["prov"] != "Unassigned")
    & (dedup_work["region"] != "Unassigned")
    & (dedup_work["isl_grp"] != "Unassigned")
)

group_cols = ["name_clean", "prov", "region", "isl_grp"]

keep_indices = []
drop_indices = []

for _, group in dedup_work[safe_admin_mask].groupby(group_cols):
    group = group.sort_values("_quality_score", ascending=False)

    kept_group_indices = []

    for idx, row in group.iterrows():
        is_duplicate = False

        for kept_idx in kept_group_indices:
            kept_row = dedup_work.loc[kept_idx]

            distance = haversine_km(
                row["lat"],
                row["lon"],
                kept_row["lat"],
                kept_row["lon"]
            )

            if pd.notna(distance) and distance <= 1:
                is_duplicate = True
                break

        if is_duplicate:
            drop_indices.append(idx)
        else:
            kept_group_indices.append(idx)
            keep_indices.append(idx)

# Rows outside safe admin duplicate groups are kept automatically
all_indices = set(dedup_work.index)
drop_indices = set(drop_indices)

quickmap_second_dedup = dedup_work.loc[
    [idx for idx in dedup_work.index if idx not in drop_indices],
    original_columns
].copy()

second_dedup_dropped_rows = dedup_work.loc[
    list(drop_indices),
    original_columns
].copy()

print("Before second dedup:", len(quickmap_validation))
print("After second dedup:", len(quickmap_second_dedup))
print("Rows removed:", len(second_dedup_dropped_rows))

second_dedup_dropped_rows[
    [
        "name",
        "name_clean",
        "elev",
        "marker_size",
        "lat",
        "lon",
        "prov",
        "region",
        "isl_grp",
        "quickmap_confidence",
        "quickmap_distance_km"
    ]
].head(100)

Before second dedup: 2625
After second dedup: 2512
Rows removed: 113


,name,name_clean,elev,marker_size,lat,lon,prov,region,isl_grp,quickmap_confidence,quickmap_distance_km
2100,Sharp Peak,sharp peak,775.0,medium,8.914256,117.663814,Palawan,MIMAROPA,Luzon,Low confidence - likely wrong name match,820.520829
2102,Sharp Peak,sharp peak,775.0,medium,8.914256,117.663814,Palawan,MIMAROPA,Luzon,Low confidence - likely wrong name match,878.302706
2103,Sharp Peak,sharp peak,775.0,medium,8.914256,117.663814,Palawan,MIMAROPA,Luzon,Low confidence - likely wrong name match,866.260608
2104,Sharp Peak,sharp peak,775.0,medium,8.914256,117.663814,Palawan,MIMAROPA,Luzon,Low confidence - likely wrong name match,800.195602
2105,Sharp Peak,sharp peak,775.0,medium,8.914256,117.663814,Palawan,MIMAROPA,Luzon,Low confidence - likely wrong name match,205.898308
...,...,...,...,...,...,...,...,...,...,...,...
1274,Mount Lunas,lunas,698.0,medium,10.254287,124.915176,Southern Leyte,Region VIII,Visayas,Low confidence - likely wrong name match,387.076710
1275,Mount Lunas,lunas,422.0,small,12.291152,122.035352,Romblon,MIMAROPA,Luzon,Low confidence - likely wrong name match,345.507784
1276,Mount Lunas,lunas,422.0,small,12.291152,122.035352,Romblon,MIMAROPA,Luzon,Low confidence - likely wrong name match,387.209762
1823,Mount Pao,pao,1441.0,medium,16.440285,120.816466,Benguet,CAR,Luzon,Low confidence - likely wrong name match,219.672690


1. Group rows with same name_clean + same prov + same region + same isl_grp
2. Sort them by quality_score from highest to lowest
3. Keep the best-scoring row first
4. Drop other rows within 1 km of that kept row

In [129]:
quickmap_second_dedup.to_csv(
    "philippines_mountains_validated_quickmap_dedup.csv",
    index=False
)

print("Saved rows:", len(quickmap_second_dedup))

Saved rows: 2512


OSM cleaned
+ province/region/island group assigned
+ Unassigned admin rows labeled
+ QuickMap validation
+ missing elevation filled from High confidence QuickMap
+ marker_size recalculated
+ second dedup done

# Enrich top 100 large mountains with PeakVisor (volc, elev, prom)

Checkpoint
✅ OSM mountain data
✅ standardized names
✅ coordinates
✅ province / region / island group
✅ QuickMap validation fields

In [130]:
#Load your current masterfile
import pandas as pd

master_df = pd.read_csv("philippines_mountains_validated_quickmap_dedup.csv")

print(master_df.shape)
master_df.head()

(2512, 44)


,mountain_id,name,name_original,source,source.1,name_clean,alt_names,elev,prom,marker_size,...,detail_url,name_quickmap,lat_quickmap,lon_quickmap,elev_quickmap,quickmap_match,quickmap_distance_km,quickmap_validation_note,quickmap_status,quickmap_confidence
0,387_15.9003_120.9738,Mount 387,Mount 387,OpenStreetMap,OpenStreetMap,387,[],777.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,OpenStreetMap,OpenStreetMap,abao,[],2527.0,NaN,large,...,NaN,Mount Abao,16.8333,120.91670,2524.0,True,0.222378,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,OpenStreetMap,OpenStreetMap,abel's peak,[],1464.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,OpenStreetMap,OpenStreetMap,abnataclayong,[],370.0,NaN,small,...,NaN,Mount Abnataclayong,5.6750,125.36694,370.0,True,0.000443,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,OpenStreetMap,OpenStreetMap,aboabo,[],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match


In [131]:
#Check what is inside the columns
master_df.columns.tolist()



['mountain_id',
 'name',
 'name_original',
 'source',
 'source.1',
 'name_clean',
 'alt_names',
 'elev',
 'prom',
 'marker_size',
 'coord',
 'lat',
 'lon',
 'is_volc',
 'isl_grp',
 'elev_raw',
 'osm_id',
 'osm_type',
 'osm_tags',
 'data_quality_notes',
 'geometry',
 'index_right',
 'prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'prov_gis_original',
 'region_gis_original',
 'source_file',
 'source.2',
 'source.3',
 'coord_source',
 'admin_note',
 'detail_url',
 'name_quickmap',
 'lat_quickmap',
 'lon_quickmap',
 'elev_quickmap',
 'quickmap_match',
 'quickmap_distance_km',
 'quickmap_validation_note',
 'quickmap_status',
 'quickmap_confidence']

In [132]:
#Top 100 Large mountains for Peakvisor

# Make sure elevation is numeric
master_df["elev"] = pd.to_numeric(master_df["elev"], errors="coerce")

priority_mountains = master_df.sort_values(
    by="elev",
    ascending=False,
    na_position="last"
).head(100).copy()

priority_mountains[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "quickmap_status",
    "quickmap_confidence"
]].head(30)

,name,elev,prom,is_volc,prov,region,isl_grp,quickmap_status,quickmap_confidence
1584,Mother Peak,2954.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,Not found in QuickMap,No QuickMap match
1086,Kidapawan Peak,2950.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,Not found in QuickMap,No QuickMap match
749,Digos Peak,2945.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,Not found in QuickMap,No QuickMap match
795,Mount Dulang-Dulang,2938.0,NaN,NaN,Bukidnon,Region X,Mindanao,Not found in QuickMap,No QuickMap match
1915,Mount Pulag,2928.0,NaN,NaN,Benguet,CAR,Luzon,Not found in QuickMap,No QuickMap match
2201,Mount Tabayoc,2820.0,NaN,NaN,Benguet,CAR,Luzon,Not found in QuickMap,No QuickMap match
2496,Mount Wiji,2819.0,NaN,NaN,Bukidnon,Region X,Mindanao,Not found in QuickMap,No QuickMap match
1278,Mount Maagnaw,2742.0,NaN,NaN,Bukidnon,Region X,Mindanao,Not found in QuickMap,No QuickMap match
1029,Mount Kalawitan,2714.0,NaN,NaN,Ifugao,CAR,Luzon,Not found in QuickMap,No QuickMap match
98,Mount Amuyao,2701.0,NaN,NaN,Ifugao,CAR,Luzon,Not found in QuickMap,No QuickMap match


In [133]:
#Check which mountains from these top 100 needs more enrichment
priority_needs_peakvisor = priority_mountains[
    priority_mountains["prom"].isna()
    | priority_mountains["is_volc"].isna()
    | priority_mountains["detail_url"].isna()
].copy()

print("Priority mountains needing PeakVisor enrichment:", len(priority_needs_peakvisor))

priority_needs_peakvisor[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "detail_url",
    "prov",
    "region",
    "isl_grp"
]].head(30)

Priority mountains needing PeakVisor enrichment: 100


,name,elev,prom,is_volc,detail_url,prov,region,isl_grp
1584,Mother Peak,2954.0,NaN,NaN,NaN,Davao del Sur,Region XI,Mindanao
1086,Kidapawan Peak,2950.0,NaN,NaN,NaN,Davao del Sur,Region XI,Mindanao
749,Digos Peak,2945.0,NaN,NaN,NaN,Davao del Sur,Region XI,Mindanao
795,Mount Dulang-Dulang,2938.0,NaN,NaN,NaN,Bukidnon,Region X,Mindanao
1915,Mount Pulag,2928.0,NaN,NaN,NaN,Benguet,CAR,Luzon
2201,Mount Tabayoc,2820.0,NaN,NaN,NaN,Benguet,CAR,Luzon
2496,Mount Wiji,2819.0,NaN,NaN,NaN,Bukidnon,Region X,Mindanao
1278,Mount Maagnaw,2742.0,NaN,NaN,NaN,Bukidnon,Region X,Mindanao
1029,Mount Kalawitan,2714.0,NaN,NaN,NaN,Ifugao,CAR,Luzon
98,Mount Amuyao,2701.0,NaN,NaN,NaN,Ifugao,CAR,Luzon


In [134]:
#Setting up helpers
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PhilippinesMountainsPortfolioProject/1.0)"
}

PEAKVISOR_PH_URL = "https://peakvisor.com/adm/philippines.html"

KEYWORDS = [
    "elevation", "prominence", "mountain", "summit",
    "volcano", "volcanic", "stratovolcano",
    "hiking", "trail", "trek", "climb", "route",
    "range", "mountain range"
]

def clean_text(text):
    return (
        str(text)
        .replace("\u2060", " ")
        .replace("\xa0", " ")
        .replace("\n", " ")
        .replace("\t", " ")
        .strip()
    )

def numbers_only(text):
    return int(re.sub(r"[^\d]", "", str(text)))

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = name.replace("mount ", "").replace("mt. ", "").replace("mt ", "")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

def find_keywords(text):
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS if kw in text_lower]

In [135]:
#Scrape the PeakVisor Philippines page links
response = requests.get(PEAKVISOR_PH_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

peakvisor_rows = []

for a in soup.find_all("a", href=True):
    text = clean_text(a.get_text(" ", strip=True))
    href = a["href"]

    pattern = r"^(.*?)\s+([\d\s,]+)\s*m\s*\(prom:\s*([\d\s,]+)\s*m\s*\)"
    match = re.search(pattern, text)

    if match:
        peakvisor_rows.append({
            "peakvisor_name": match.group(1).strip(),
            "peakvisor_elev": numbers_only(match.group(2)),
            "peakvisor_prom": numbers_only(match.group(3)),
            "peakvisor_detail_url": urljoin(PEAKVISOR_PH_URL, href),
            "peakvisor_url_source": "country_index"
        })

peakvisor_index = pd.DataFrame(peakvisor_rows).drop_duplicates()
peakvisor_index["name_clean"] = peakvisor_index["peakvisor_name"].apply(normalize_name)

print("PeakVisor index rows found:", len(peakvisor_index))
peakvisor_index.head(10)

PeakVisor index rows found: 17


,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_url_source,name_clean
0,Mount Apo,2954,2954,https://peakvisor.com/peak/mount-apo.html,country_index,apo
1,87 Degrees,2940,17,https://peakvisor.com/peak/87-degrees.html,country_index,87 degrees
2,Mount Dulang-Dulang,2938,2444,https://peakvisor.com/peak/mount-kitanglad.html,country_index,dulang dulang
3,Mount Pulag,2928,2928,https://peakvisor.com/peak/mount-pulag.html,country_index,pulag
4,Mount Kitanglad,2899,383,https://peakvisor.com/peak/mount-kitanglad-20z...,country_index,kitanglad
5,Mount Kalatungan,2874,1495,https://peakvisor.com/peak/mt-kalatungan.html,country_index,kalatungan
6,Mt. Lumpanag (Wiji),2819,281,https://peakvisor.com/peak/mt-makaupao-wiji.html,country_index,lumpanag wiji
7,Mount Piapayungan,2815,1595,https://peakvisor.com/peak/mount-piapayungan.html,country_index,piapayungan
8,Mount Maagnaw,2742,331,https://peakvisor.com/peak/mount-ma-agnaw.html,country_index,maagnaw
9,Pual,2725,110,https://peakvisor.com/peak/pual.html,country_index,pual


In [136]:
#Match my priority monutains to peakvisor links
priority = priority_needs_peakvisor.copy()
priority["name_clean"] = priority["name"].apply(normalize_name)

priority_matched = priority.merge(
    peakvisor_index,
    on="name_clean",
    how="left"
)

priority_matched["peakvisor_match_status"] = priority_matched["peakvisor_detail_url"].apply(
    lambda x: "matched_country_index" if pd.notna(x) else "not_found_in_country_index"
)

print("Priority mountains:", len(priority_matched))
print(priority_matched["peakvisor_match_status"].value_counts())

priority_matched[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "peakvisor_match_status"
]].head(30)

Priority mountains: 100
peakvisor_match_status
not_found_in_country_index    93
matched_country_index          7
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_match_status
0,Mother Peak,2954.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
1,Kidapawan Peak,2950.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
2,Digos Peak,2945.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
3,Mount Dulang-Dulang,2938.0,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,https://peakvisor.com/peak/mount-kitanglad.html,matched_country_index
4,Mount Pulag,2928.0,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,https://peakvisor.com/peak/mount-pulag.html,matched_country_index
5,Mount Tabayoc,2820.0,NaN,Benguet,CAR,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
6,Mount Wiji,2819.0,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
7,Mount Maagnaw,2742.0,NaN,Bukidnon,Region X,Mindanao,Mount Maagnaw,2742.0,331.0,https://peakvisor.com/peak/mount-ma-agnaw.html,matched_country_index
8,Mount Kalawitan,2714.0,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
9,Mount Amuyao,2701.0,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index


In [137]:
#Generate possible links for those unmatched
def make_slug_candidates(name):
    raw = str(name).lower().strip()
    raw = re.sub(r"[^a-z0-9\s\-]", " ", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    no_mount = (
        raw.replace("mount ", "")
           .replace("mt. ", "")
           .replace("mt ", "")
           .strip()
    )

    candidates = [
        raw.replace(" ", "-"),
        no_mount.replace(" ", "-"),
        "mount-" + no_mount.replace(" ", "-"),
        "mt-" + no_mount.replace(" ", "-")
    ]

    return list(dict.fromkeys([c for c in candidates if c]))

def possible_peakvisor_urls(name):
    urls = []

    for slug in make_slug_candidates(name):
        urls.extend([
            f"https://peakvisor.com/peak/{slug}.html",
            f"https://peakvisor.com/mountain/{slug}.html"
        ])

    return urls

# Quick test
possible_peakvisor_urls("Mount Apo")

['https://peakvisor.com/peak/mount-apo.html',
 'https://peakvisor.com/mountain/mount-apo.html',
 'https://peakvisor.com/peak/apo.html',
 'https://peakvisor.com/mountain/apo.html',
 'https://peakvisor.com/peak/mt-apo.html',
 'https://peakvisor.com/mountain/mt-apo.html']

In [138]:
#Test unmatchedd mountains for guessed urls
def page_matches_mountain(text, mountain_name):
    text_lower = str(text).lower()

    name_clean = normalize_name(mountain_name)
    name_words = name_clean.split()

    found_keywords = find_keywords(text)

    # Good if page contains the mountain name
    name_in_page = name_clean in normalize_name(text)

    # Backup check: important name words appear
    important_words_found = sum(1 for word in name_words if len(word) > 2 and word in text_lower)

    is_useful = name_in_page or important_words_found >= 1 or len(found_keywords) >= 2

    return is_useful, found_keywords

def find_url_by_slug_guess(name):
    for test_url in possible_peakvisor_urls(name):
        try:
            r = requests.get(test_url, headers=headers, timeout=20)

            if r.status_code != 200:
                continue

            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)

            is_useful, found_keywords = page_matches_mountain(detail_text, name)

            if is_useful:
                return {
                    "guessed_detail_url": test_url,
                    "url_guess_status": "found_by_slug_guess",
                    "keywords_found": ", ".join(found_keywords),
                    "detail_text": detail_text
                }

            time.sleep(1)

        except Exception:
            continue

    return {
        "guessed_detail_url": None,
        "url_guess_status": "not_found_after_url_guess",
        "keywords_found": None,
        "detail_text": None
    }

In [139]:
#Applying it
unmatched = priority_matched[
    priority_matched["peakvisor_detail_url"].isna()
].copy()

guess_rows = []

print("Unmatched mountains to URL-guess:", len(unmatched))

for _, row in unmatched.iterrows():
    result = find_url_by_slug_guess(row["name"])

    guess_rows.append({
        "name": row["name"],
        "guessed_detail_url": result["guessed_detail_url"],
        "url_guess_status": result["url_guess_status"],
        "guess_keywords_found": result["keywords_found"],
        "guessed_detail_text": result["detail_text"]
    })

    print(row["name"], "→", result["url_guess_status"])
    time.sleep(2)

url_guess_df = pd.DataFrame(guess_rows)

url_guess_df.head()

Unmatched mountains to URL-guess: 93
Mother Peak → found_by_slug_guess
Kidapawan Peak → found_by_slug_guess
Digos Peak → found_by_slug_guess
Mount Tabayoc → found_by_slug_guess
Mount Wiji → found_by_slug_guess
Mount Kalawitan → found_by_slug_guess
Mount Amuyao → found_by_slug_guess
Mount Alanib → found_by_slug_guess
Langkayugan Peak → found_by_slug_guess
Mount Lumuluyaw → found_by_slug_guess
Mount Tuminungan → found_by_slug_guess
Mount Kapiligan → found_by_slug_guess
Mount Panotoan → found_by_slug_guess
Mount Pandadagsaan → found_by_slug_guess
Mount Ragang → found_by_slug_guess
Mount Lipadas → found_by_slug_guess
Mount Kiabansag → found_by_slug_guess
Mount Alchan → found_by_slug_guess
Mount Osdung → found_by_slug_guess
Mount Nanluyaw → found_by_slug_guess
Mount Zion → found_by_slug_guess
Mount Nangaoto → found_by_slug_guess
Mount Abao → found_by_slug_guess
Mount Mangabon → found_by_slug_guess
Mount Baco → found_by_slug_guess
Mount Bangbanglang → found_by_slug_guess
Mount Sapocoy → foun

,name,guessed_detail_url,url_guess_status,guess_keywords_found,guessed_detail_text
0,Mother Peak,https://peakvisor.com/peak/mother-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Mother Peak (2,954 m) – Philippines | PeakViso..."
1,Kidapawan Peak,https://peakvisor.com/peak/kidapawan-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...","Kidapawan Peak (2,950 m) – Philippines | PeakV..."
2,Digos Peak,https://peakvisor.com/peak/digos-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...","Digos Peak (2,945 m) – Philippines | PeakVisor..."
3,Mount Tabayoc,https://peakvisor.com/peak/mount-tabayoc.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Mount Tabayoc (2,849 m) – Philippines | PeakVi..."
4,Mount Wiji,https://peakvisor.com/peak/mount-wiji.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...","Mount Wiji (2,819 m) – Philippines | PeakVisor..."


In [140]:
#Combine country-index URLs and guessed URLs
priority_urls_final = priority_matched.merge(
    url_guess_df,
    on="name",
    how="left"
)

priority_urls_final["final_detail_url"] = priority_urls_final["peakvisor_detail_url"].fillna(
    priority_urls_final["guessed_detail_url"]
)

priority_urls_final["final_url_source"] = priority_urls_final.apply(
    lambda row:
        "country_index"
        if pd.notna(row["peakvisor_detail_url"])
        else ("slug_guess" if pd.notna(row["guessed_detail_url"]) else "not_found"),
    axis=1
)

priority_urls_final["final_match_status"] = priority_urls_final["final_detail_url"].apply(
    lambda x: "has_detail_url" if pd.notna(x) else "not_found_after_all_methods"
)

print(priority_urls_final["final_url_source"].value_counts())
print(priority_urls_final["final_match_status"].value_counts())

priority_urls_final[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "guessed_detail_url",
    "final_detail_url", "final_url_source", "final_match_status"
]].head(50)

final_url_source
slug_guess       92
country_index     7
not_found         3
Name: count, dtype: int64
final_match_status
has_detail_url                 99
not_found_after_all_methods     3
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,guessed_detail_url,final_detail_url,final_url_source,final_match_status
0,Mother Peak,2954.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mother-peak.html,https://peakvisor.com/peak/mother-peak.html,slug_guess,has_detail_url
1,Kidapawan Peak,2950.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/kidapawan-peak.html,https://peakvisor.com/peak/kidapawan-peak.html,slug_guess,has_detail_url
2,Digos Peak,2945.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/digos-peak.html,https://peakvisor.com/peak/digos-peak.html,slug_guess,has_detail_url
3,Mount Dulang-Dulang,2938.0,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,https://peakvisor.com/peak/mount-kitanglad.html,NaN,https://peakvisor.com/peak/mount-kitanglad.html,country_index,has_detail_url
4,Mount Pulag,2928.0,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,https://peakvisor.com/peak/mount-pulag.html,NaN,https://peakvisor.com/peak/mount-pulag.html,country_index,has_detail_url
5,Mount Tabayoc,2820.0,NaN,Benguet,CAR,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-tabayoc.html,https://peakvisor.com/peak/mount-tabayoc.html,slug_guess,has_detail_url
6,Mount Wiji,2819.0,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-wiji.html,https://peakvisor.com/peak/mount-wiji.html,slug_guess,has_detail_url
7,Mount Maagnaw,2742.0,NaN,Bukidnon,Region X,Mindanao,Mount Maagnaw,2742.0,331.0,https://peakvisor.com/peak/mount-ma-agnaw.html,NaN,https://peakvisor.com/peak/mount-ma-agnaw.html,country_index,has_detail_url
8,Mount Kalawitan,2714.0,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-kalawitan.html,https://peakvisor.com/peak/mount-kalawitan.html,slug_guess,has_detail_url
9,Mount Amuyao,2701.0,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-amuyao.html,https://peakvisor.com/peak/mount-amuyao.html,slug_guess,has_detail_url


In [141]:
#Scrape final detail URLs one by one
#raw scraped page text
detail_rows = []

to_scrape = priority_urls_final[
    priority_urls_final["final_detail_url"].notna()
].copy()

print("Final detail pages available:", len(to_scrape))

for _, row in to_scrape.iterrows():
    try:
        # If detail text already came from slug guessing, reuse it
        if row["final_url_source"] == "slug_guess" and pd.notna(row.get("guessed_detail_text")):
            detail_text = row["guessed_detail_text"]
        else:
            r = requests.get(row["final_detail_url"], headers=headers, timeout=30)
            r.raise_for_status()
            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)
            time.sleep(2)

        found_keywords = find_keywords(detail_text)

        detail_rows.append({
            "name": row["name"],
            "elev": row["elev"],
            "prom": row["prom"],
            "is_volc": row["is_volc"],
            "prov": row["prov"],
            "region": row["region"],
            "isl_grp": row["isl_grp"],
            "peakvisor_name": row.get("peakvisor_name"),
            "peakvisor_elev": row.get("peakvisor_elev"),
            "peakvisor_prom": row.get("peakvisor_prom"),
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "keywords_found": ", ".join(found_keywords),
            "detail_scrape_status": "scraped_with_keywords" if found_keywords else "scraped_no_keywords",
            "detail_text": detail_text
        })

        print(row["name"], "→ scraped")

    except Exception as e:
        detail_rows.append({
            "name": row["name"],
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "detail_scrape_status": "failed",
            "error": str(e)
        })

        print("Failed:", row["name"], e)

priority_detail_pages = pd.DataFrame(detail_rows)

priority_detail_pages.head()

Final detail pages available: 99
Mother Peak → scraped
Kidapawan Peak → scraped
Digos Peak → scraped
Mount Dulang-Dulang → scraped
Mount Pulag → scraped
Mount Tabayoc → scraped
Mount Wiji → scraped
Mount Maagnaw → scraped
Mount Kalawitan → scraped
Mount Amuyao → scraped
Mount Alanib → scraped
Langkayugan Peak → scraped
Mount Lumuluyaw → scraped
Mount Tuminungan → scraped
Mount Kapiligan → scraped
Mount Panotoan → scraped
Mount Pandadagsaan → scraped
Mount Ragang → scraped
Mount Lipadas → scraped
Mount Kiabansag → scraped
Mount Alchan → scraped
Mount Halcon → scraped
Mount Osdung → scraped
Mount Nanluyaw → scraped
Mount Zion → scraped
Mount Nangaoto → scraped
Mount Abao → scraped
Mount Mangabon → scraped
Mount Baco → scraped
Mount Bangbanglang → scraped
Mount Kanlaon → scraped
Mount Sapocoy → scraped
Makawiwili Peak → scraped
Mount Camingingel → scraped
Mount Sicapoo → scraped
Mount Sicapoo → scraped
Mount Sicapoo → scraped
Mount Sicapoo → scraped
Mount Nangkabulos → scraped
Mount Loco-

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,detail_url,url_source,keywords_found,detail_scrape_status,detail_text
0,Mother Peak,2954.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/mother-peak.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Mother Peak (2,954 m) – Philippines | PeakViso..."
1,Kidapawan Peak,2950.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/kidapawan-peak.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Kidapawan Peak (2,950 m) – Philippines | PeakV..."
2,Digos Peak,2945.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/digos-peak.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Digos Peak (2,945 m) – Philippines | PeakVisor..."
3,Mount Dulang-Dulang,2938.0,NaN,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,https://peakvisor.com/peak/mount-kitanglad.html,country_index,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Mount Dulang-Dulang (2,938 m) – Philippines | ..."
4,Mount Pulag,2928.0,NaN,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,https://peakvisor.com/peak/mount-pulag.html,country_index,"elevation, prominence, mountain, summit, volca...",scraped_with_keywords,"Mount Pulag (2,928 m) – Philippines | PeakViso..."


In [142]:
#Main Parser for the detail page
import re
import pandas as pd

def clean_peakvisor_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\u2060", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_elevation_from_peakvisor_text(text):
    text = clean_peakvisor_text(text)

    patterns = [
        # Page title pattern: Mount Apo (2,954 m) – Philippines
        r"\(([\d,]+)\s*m\)\s*[–-]\s*Philippines",

        # About pattern: Mount Apo (2954m/9692ft a.s.l.)
        r"\(([\d,]+)\s*m\s*/\s*[\d,]+\s*ft\s*a\.s\.l\.\)",

        # Generic: Elevation 2,954 m
        r"Elevation\s+([\d,]+)\s*m",

        # Generic: 2,954 m Elevation
        r"([\d,]+)\s*m\s+Elevation"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return int(match.group(1).replace(",", ""))

    return None


def extract_prominence_from_peakvisor_text(text):
    text = clean_peakvisor_text(text)

    patterns = [
        # Left card pattern: 98 ft Prominence may exist, but metric is better from About
        # About pattern: The prominence is 30m/98ft.
        r"prominence\s+is\s+([\d,]+)\s*m",

        # Index/detail pattern: Prominence 30 m
        r"Prominence\s+([\d,]+)\s*m",

        # Backup: (prom: 30 m)
        r"\(prom:\s*([\d,]+)\s*m\s*\)"
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.I)
        if match:
            return int(match.group(1).replace(",", ""))

    return None


def extract_coordinates_from_peakvisor_text(text):
    text = clean_peakvisor_text(text)

    # Pattern: 6.987597 N 125.269249 E
    match = re.search(
        r"(-?\d+\.\d+)\s*([NS])\s+(-?\d+\.\d+)\s*([EW])",
        text,
        flags=re.I
    )

    if not match:
        return pd.Series([None, None])

    lat = float(match.group(1))
    lon = float(match.group(3))

    if match.group(2).upper() == "S":
        lat *= -1
    if match.group(4).upper() == "W":
        lon *= -1

    return pd.Series([lat, lon])


def extract_location_block(text):
    text = clean_peakvisor_text(text)

    # Pattern around PeakVisor page: Location Philippines Davao del Sur Cotabato Davao City 6.987...
    match = re.search(
        r"Location\s+(.*?)(-?\d+\.\d+\s*[NS]\s+-?\d+\.\d+\s*[EW])",
        text,
        flags=re.I
    )

    if match:
        return match.group(1).strip()

    return None


def extract_about_text(text):
    text = clean_peakvisor_text(text)

    # Look for the sentence after About
    match = re.search(
        r"About\s+(.*?)(?=By elevation|Weather|Show more|Today|Sunday|Monday|$)",
        text,
        flags=re.I
    )

    if match:
        about = match.group(1).strip()
        return about if len(about) > 20 else None

    # Backup: find sentence that says "is a mountain in Philippines"
    match = re.search(
        r"([A-Z][A-Za-z0-9\s\-\.'’]+?\s+\([\d,]+m/[\d,]+ft a\.s\.l\.\)\s+is a mountain in Philippines.*?)(?=By elevation|Weather|Show more|$)",
        text,
        flags=re.I
    )

    if match:
        return match.group(1).strip()

    return None


def extract_trail_sentence(text):
    text = clean_peakvisor_text(text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    trail_terms = ["trail", "hiking", "trek", "climb", "route", "summit"]

    found = [
        s.strip()
        for s in sentences
        if any(term in s.lower() for term in trail_terms)
    ]

    return " ".join(found[:2]) if found else None


def extract_volcano_sentence(text):
    text = clean_peakvisor_text(text)

    sentences = re.split(r"(?<=[.!?])\s+", text)
    volcano_terms = ["volcano", "volcanic", "stratovolcano"]

    found = [
        s.strip()
        for s in sentences
        if any(term in s.lower() for term in volcano_terms)
    ]

    return " ".join(found[:2]) if found else None


def has_trail_info(text):
    text = clean_peakvisor_text(text).lower()
    return any(term in text for term in ["trail", "hiking", "trek", "climb", "route", "summit"])


def has_volcano_info(text):
    text = clean_peakvisor_text(text).lower()
    return any(term in text for term in ["volcano", "volcanic", "stratovolcano"])

FT_TO_M = 0.3048

def parse_number(num):
    if pd.isna(num):
        return None
    return float(re.sub(r"[^\d.]", "", str(num)))


def ft_to_m(ft_value):
    if ft_value is None:
        return None
    return round(float(ft_value) * FT_TO_M)


def extract_elevation_unit_aware(text):
    text = clean_peakvisor_text(text)

    # 1. Title pattern: Mount Apo (2,954 m) – Philippines
    match = re.search(r"\(([\d,\s]+)\s*m\)\s*[–-]\s*Philippines", text, flags=re.I)
    if match:
        return pd.Series([int(re.sub(r"[^\d]", "", match.group(1))), "m_from_title"])

    # 2. About pattern: (2950m/9678ft a.s.l.)
    match = re.search(r"\(([\d,\s]+)\s*m\s*/\s*[\d,\s]+\s*ft\s*a\.s\.l\.\)", text, flags=re.I)
    if match:
        return pd.Series([int(re.sub(r"[^\d]", "", match.group(1))), "m_from_about"])

    # 3. Card pattern: 9,678 ft Elevation
    match = re.search(r"([\d,\s]+)\s*ft\s+Elevation", text, flags=re.I)
    if match:
        ft = int(re.sub(r"[^\d]", "", match.group(1)))
        return pd.Series([ft_to_m(ft), "ft_converted_to_m"])

    return pd.Series([None, "not_found"])


def extract_prominence_unit_aware(text):
    text = clean_peakvisor_text(text)

    # 1. About pattern: The prominence is 30m/98ft.
    match = re.search(r"prominence\s+is\s+([\d,\s]+)\s*m\s*/\s*[\d,\s]+\s*ft", text, flags=re.I)
    if match:
        return pd.Series([int(re.sub(r"[^\d]", "", match.group(1))), "m_from_about"])

    # 2. Generic metric: Prominence 30 m
    match = re.search(r"Prominence\s+([\d,\s]+)\s*m", text, flags=re.I)
    if match:
        return pd.Series([int(re.sub(r"[^\d]", "", match.group(1))), "m_from_text"])

    # 3. Card pattern: 98 ft Prominence
    match = re.search(r"([\d,\s]+)\s*ft\s+Prominence", text, flags=re.I)
    if match:
        ft = int(re.sub(r"[^\d]", "", match.group(1)))
        return pd.Series([ft_to_m(ft), "ft_converted_to_m"])

    return pd.Series([None, "not_found"])

In [143]:
#Apply Parser to yor scrapedd detail page
parsed_peakvisor = priority_detail_pages.copy()

parsed_peakvisor["detail_text_clean"] = parsed_peakvisor["detail_text"].apply(clean_peakvisor_text)

parsed_peakvisor["pv_elev_from_page"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_elevation_from_peakvisor_text
)

parsed_peakvisor["pv_prom_from_page"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_prominence_from_peakvisor_text
)

parsed_peakvisor[["pv_lat", "pv_lon"]] = parsed_peakvisor["detail_text_clean"].apply(
    extract_coordinates_from_peakvisor_text
)

parsed_peakvisor["pv_location_text"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_location_block
)

parsed_peakvisor["pv_about_text"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_about_text
)

parsed_peakvisor["pv_has_trail_info"] = parsed_peakvisor["detail_text_clean"].apply(
    has_trail_info
)

parsed_peakvisor["pv_has_volcano_info"] = parsed_peakvisor["detail_text_clean"].apply(
    has_volcano_info
)

parsed_peakvisor["pv_trail_sentence"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_trail_sentence
)

parsed_peakvisor["pv_volcano_sentence"] = parsed_peakvisor["detail_text_clean"].apply(
    extract_volcano_sentence
)

#back up parsing from the main page
parsed_peakvisor["pv_final_elev"] = parsed_peakvisor["pv_elev_from_page"].fillna(
    parsed_peakvisor["peakvisor_elev"]
)

parsed_peakvisor["pv_final_prom"] = parsed_peakvisor["pv_prom_from_page"].fillna(
    parsed_peakvisor["peakvisor_prom"]
)
parsed_peakvisor[["pv_elev_from_page_v2", "pv_elev_unit_source"]] = parsed_peakvisor["detail_text_clean"].apply(
    extract_elevation_unit_aware
)

parsed_peakvisor[["pv_prom_from_page_v2", "pv_prom_unit_source"]] = parsed_peakvisor["detail_text_clean"].apply(
    extract_prominence_unit_aware
)
parsed_peakvisor["pv_final_elev_v2"] = (
    parsed_peakvisor["pv_elev_from_page_v2"]
    .fillna(parsed_peakvisor["peakvisor_elev"])
)

parsed_peakvisor["pv_final_prom_v2"] = (
    parsed_peakvisor["pv_prom_from_page_v2"]
    .fillna(parsed_peakvisor["peakvisor_prom"])
)

In [144]:
#Validation flags
parsed_peakvisor["pv_elev_difference_v2"] = (
    parsed_peakvisor["pv_final_elev_v2"] - parsed_peakvisor["elev"]
)

def elevation_validation_flag_v2(row):
    if pd.isna(row["elev"]):
        return "no_original_elevation_to_compare"
    elif pd.isna(row["pv_final_elev_v2"]):
        return "no_peakvisor_elevation"

    diff = row["pv_final_elev_v2"] - row["elev"]

    if abs(diff) <= 30:
        return "close_match"
    elif abs(diff) <= 100:
        return "minor_difference"
    else:
        return "needs_review"

parsed_peakvisor["pv_elev_validation_flag_v2"] = parsed_peakvisor.apply(
    elevation_validation_flag_v2,
    axis=1
)

In [145]:
parsed_peakvisor.columns.tolist()

['name',
 'elev',
 'prom',
 'is_volc',
 'prov',
 'region',
 'isl_grp',
 'peakvisor_name',
 'peakvisor_elev',
 'peakvisor_prom',
 'detail_url',
 'url_source',
 'keywords_found',
 'detail_scrape_status',
 'detail_text',
 'detail_text_clean',
 'pv_elev_from_page',
 'pv_prom_from_page',
 'pv_lat',
 'pv_lon',
 'pv_location_text',
 'pv_about_text',
 'pv_has_trail_info',
 'pv_has_volcano_info',
 'pv_trail_sentence',
 'pv_volcano_sentence',
 'pv_final_elev',
 'pv_final_prom',
 'pv_elev_from_page_v2',
 'pv_elev_unit_source',
 'pv_prom_from_page_v2',
 'pv_prom_unit_source',
 'pv_final_elev_v2',
 'pv_final_prom_v2',
 'pv_elev_difference_v2',
 'pv_elev_validation_flag_v2']

In [146]:
parsed_peakvisor[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "peakvisor_name",
    "peakvisor_elev",
    "peakvisor_prom",
    "pv_elev_from_page_v2",
    "pv_elev_unit_source",
    "pv_prom_from_page_v2",
    "pv_prom_unit_source",
    "peakvisor_elev",
    "peakvisor_prom",
    "pv_final_elev_v2",
    "pv_final_prom_v2",
    "pv_elev_difference_v2",
    "pv_elev_validation_flag_v2",
    "detail_url"
]].head(50)

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,peakvisor_elev,peakvisor_prom,pv_final_elev_v2,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2,detail_url
0,Mother Peak,2954.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,24,m_from_about,NaN,NaN,2954.0,24,0.0,close_match,https://peakvisor.com/peak/mother-peak.html
1,Kidapawan Peak,2950.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,30,m_from_about,NaN,NaN,2950.0,30,0.0,close_match,https://peakvisor.com/peak/kidapawan-peak.html
2,Digos Peak,2945.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,34,m_from_about,NaN,NaN,2945.0,34,0.0,close_match,https://peakvisor.com/peak/digos-peak.html
3,Mount Dulang-Dulang,2938.0,NaN,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,...,m_from_title,2934,m_from_text,2938.0,2444.0,2938.0,2934,0.0,close_match,https://peakvisor.com/peak/mount-kitanglad.html
4,Mount Pulag,2928.0,NaN,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,...,m_from_title,2928,m_from_text,2928.0,2928.0,2928.0,2928,0.0,close_match,https://peakvisor.com/peak/mount-pulag.html
5,Mount Tabayoc,2820.0,NaN,NaN,Benguet,CAR,Luzon,NaN,NaN,NaN,...,m_from_title,536,m_from_about,NaN,NaN,2849.0,536,29.0,close_match,https://peakvisor.com/peak/mount-tabayoc.html
6,Mount Wiji,2819.0,NaN,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,...,m_from_title,281,m_from_about,NaN,NaN,2819.0,281,0.0,close_match,https://peakvisor.com/peak/mount-wiji.html
7,Mount Maagnaw,2742.0,NaN,NaN,Bukidnon,Region X,Mindanao,Mount Maagnaw,2742.0,331.0,...,m_from_title,347,m_from_about,2742.0,331.0,2758.0,347,16.0,close_match,https://peakvisor.com/peak/mount-ma-agnaw.html
8,Mount Kalawitan,2714.0,NaN,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,...,m_from_title,709,m_from_about,NaN,NaN,2714.0,709,0.0,close_match,https://peakvisor.com/peak/mount-kalawitan.html
9,Mount Amuyao,2701.0,NaN,NaN,Ifugao,CAR,Luzon,NaN,NaN,NaN,...,m_from_title,84,m_from_about,NaN,NaN,386.0,84,-2315.0,needs_review,https://peakvisor.com/peak/mount-amuyao.html


In [147]:
print("Rows:", len(parsed_peakvisor))

print("\nElevation parsed from page:")
print(parsed_peakvisor["pv_elev_from_page"].notna().value_counts())

print("\nProminence parsed from page:")
print(parsed_peakvisor["pv_prom_from_page"].notna().value_counts())

print("\nAbout text parsed:")
print(parsed_peakvisor["pv_about_text"].notna().value_counts())

print("\nLocation parsed:")
print(parsed_peakvisor["pv_location_text"].notna().value_counts())

Rows: 99

Elevation parsed from page:
pv_elev_from_page
True    99
Name: count, dtype: int64

Prominence parsed from page:
pv_prom_from_page
True     76
False    23
Name: count, dtype: int64

About text parsed:
pv_about_text
True    99
Name: count, dtype: int64

Location parsed:
pv_location_text
True    99
Name: count, dtype: int64


In [148]:
# View rows where unit-aware parsing still needs review
parse_review_v2 = parsed_peakvisor[
    (parsed_peakvisor["pv_final_elev_v2"].isna()) |
    (parsed_peakvisor["pv_final_prom_v2"].isna()) |
    (parsed_peakvisor["pv_about_text"].isna()) |
    (parsed_peakvisor["pv_elev_validation_flag_v2"] == "needs_review")
].copy()

parse_review_v2[[
    "name",
    "detail_url",
    "elev",
    "pv_elev_from_page_v2",
    "pv_elev_unit_source",
    "peakvisor_elev",
    "pv_final_elev_v2",
    "pv_prom_from_page_v2",
    "pv_prom_unit_source",
    "peakvisor_prom",
    "pv_final_prom_v2",
    "pv_elev_difference_v2",
    "pv_elev_validation_flag_v2",
    "pv_about_text",
    "detail_text"
]].head(10)

,name,detail_url,elev,pv_elev_from_page_v2,pv_elev_unit_source,peakvisor_elev,pv_final_elev_v2,pv_prom_from_page_v2,pv_prom_unit_source,peakvisor_prom,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2,pv_about_text,detail_text
9,Mount Amuyao,https://peakvisor.com/peak/mount-amuyao.html,2701.0,386.0,m_from_title,NaN,386.0,84,m_from_about,NaN,84,-2315.0,needs_review,Mount Amuyao (386m/1 266ft a.s.l.) is a hill i...,Mount Amuyao (386 m) – Philippines | PeakVisor...
24,Mount Zion,https://peakvisor.com/peak/mount-zion.html,2600.0,NaN,not_found,NaN,NaN,37,m_from_text,NaN,37,NaN,no_peakvisor_elevation,"Mount Zion (Hebrew: הַר צִיּוֹן, Har Ṣīyyōn; A...","Mount Zion (775 m) – Levant Ranges, Israel | P..."
44,Mount Napulauan,https://peakvisor.com/peak/mount-napulauan.html,2298.0,2611.0,m_from_title,NaN,2611.0,284,m_from_about,NaN,284,313.0,needs_review,Mount Napulauan (2 611m/8 566ft a.s.l.) is a m...,"Mount Napulauan (2,611 m) – Philippines | Peak..."
46,Mount Pack,https://peakvisor.com/peak/mount-pack.html,2290.0,587.0,m_from_about,NaN,587.0,254,m_from_about,NaN,254,-1703.0,needs_review,Mount Pack (587m/1 926ft a.s.l.) is a mountain...,"Mount Pack (587 m) – Great Dividing Range, Aus..."
65,Mount Lumot,https://peakvisor.com/peak/mount-lumot.html,2110.0,595.0,m_from_title,NaN,595.0,232,m_from_about,NaN,232,-1515.0,needs_review,Mount Lumot (595m/1 952ft a.s.l.) is a mountai...,Mount Lumot (595 m) – Philippines | PeakVisor ...
70,Mount Sipitan,https://peakvisor.com/peak/mount-sipitan.html,2100.0,2287.0,m_from_title,NaN,2287.0,122,m_from_about,NaN,122,187.0,needs_review,Mount Sipitan (2 287m/7 503ft a.s.l.) is a mou...,"Mount Sipitan (2,287 m) – Philippines | PeakVi..."
75,Mount Pa’Pa,https://peakvisor.com/peak/mount-pa-pa.html,2083.0,2490.0,m_from_title,NaN,2490.0,28,m_from_about,NaN,28,407.0,needs_review,Mount Pa’pa (2 490m/8 169ft a.s.l.) is a mount...,"Mount Pa’pa (2,490 m) – Philippines | PeakViso..."
79,Mount Palansa,https://peakvisor.com/peak/mount-palansa.html,2060.0,2169.0,m_from_title,NaN,2169.0,31,m_from_about,NaN,31,109.0,needs_review,Mount Palansa (2 169m/7 116ft a.s.l.) is a mou...,"Mount Palansa (2,169 m) – Philippines | PeakVi..."
89,Mount Wood,https://peakvisor.com/peak/mount-wood.html,2005.0,NaN,not_found,NaN,NaN,3575,m_from_text,NaN,3575,NaN,no_peakvisor_elevation,Mount Wood (sometimes referred to as Wood Peak...,"Mount Wood (4,842 m) – Saint Elias Mountains, ..."
90,Mount Malaya,https://peakvisor.com/peak/mount-malaya.html,2000.0,2259.0,m_from_title,NaN,2259.0,1,m_from_about,NaN,1,259.0,needs_review,Mount Malaya (2 259m/7 411ft a.s.l.) is a moun...,"Mount Malaya (2,259 m) – Philippines | PeakVis..."


In [149]:
#No match at all after all methods from the priority mountains
priority_not_found = priority_urls_final[
    priority_urls_final["final_match_status"] == "not_found_after_all_methods"
].copy()

priority_not_found.to_csv(
    "priority_mountains_not_found_in_peakvisor.csv",
    index=False
)

print("Not found after all methods:", len(priority_not_found))

priority_not_found[[
    "name", "elev", "prov", "region", "isl_grp"
]].head(50)

Not found after all methods: 3


,name,elev,prov,region,isl_grp
74,Mount Damocnoc,2086.0,Benguet,CAR,Luzon
96,Mount Hilong-Hilong,1920.0,Agusan del Norte,Region XIII,Mindanao
97,Mount Tenglawan,1910.0,Benguet,CAR,Luzon


In [150]:
print("Rows needing review:", len(parse_review_v2))

print("\nElevation unit source:")
print(parsed_peakvisor["pv_elev_unit_source"].value_counts(dropna=False))

print("\nProminence unit source:")
print(parsed_peakvisor["pv_prom_unit_source"].value_counts(dropna=False))

print("\nValidation flags:")
print(parsed_peakvisor["pv_elev_validation_flag_v2"].value_counts(dropna=False))

Rows needing review: 11

Elevation unit source:
pv_elev_unit_source
m_from_title    96
not_found        2
m_from_about     1
Name: count, dtype: int64

Prominence unit source:
pv_prom_unit_source
m_from_about         81
m_from_text          17
ft_converted_to_m     1
Name: count, dtype: int64

Validation flags:
pv_elev_validation_flag_v2
close_match               72
minor_difference          16
needs_review               9
no_peakvisor_elevation     2
Name: count, dtype: int64


In [151]:
#Renaming
parsed_peakvisor = parsed_peakvisor.rename(columns={
    "pv_final_elev_v2": "peakvisor_elev_m",
    "pv_final_prom_v2": "peakvisor_prom_m",
    "pv_elev_unit_source": "peakvisor_elev_source_unit",
    "pv_prom_unit_source": "peakvisor_prom_source_unit",
    "pv_elev_difference_v2": "peakvisor_elev_difference_m",
    "pv_elev_validation_flag_v2": "peakvisor_elev_validation_flag",
    "pv_lat": "peakvisor_lat",
    "pv_lon": "peakvisor_lon",
    "pv_location_text": "peakvisor_location_text",
    "pv_about_text": "peakvisor_about_text",
    "pv_has_trail_info": "has_trail_info",
    "pv_has_volcano_info": "has_volcano_info",
    "pv_trail_sentence": "trail_sentence",
    "pv_volcano_sentence": "volcano_sentence"
})

parsed_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_final_elev,pv_final_prom,pv_elev_from_page_v2,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,peakvisor_elev_difference_m,peakvisor_elev_validation_flag
0,Mother Peak,2954.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,2954,24.0,2954.0,m_from_title,24,m_from_about,2954.0,24,0.0,close_match
1,Kidapawan Peak,2950.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,2950,30.0,2950.0,m_from_title,30,m_from_about,2950.0,30,0.0,close_match
2,Digos Peak,2945.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,2945,34.0,2945.0,m_from_title,34,m_from_about,2945.0,34,0.0,close_match
3,Mount Dulang-Dulang,2938.0,NaN,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,...,2938,2444.0,2938.0,m_from_title,2934,m_from_text,2938.0,2934,0.0,close_match
4,Mount Pulag,2928.0,NaN,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,...,2928,2928.0,2928.0,m_from_title,2928,m_from_text,2928.0,2928,0.0,close_match


In [152]:
#Safe Checkpoint
parsed_peakvisor.to_csv(
    "priority_mountains_peakvisor_parsed_enrichment_v3_unit_aware.csv",
    index=False
)

print("Saved final PeakVisor parsed enrichment file.")

Saved final PeakVisor parsed enrichment file.


In [153]:
print("Columns in osm_metadata_df but missing in parsed_peakvisor_df:")
print([col for col in missing_repo_location_df.columns if col not in parsed_peakvisor.columns])

print("\nColumns in parsed_peakvisor but not in osm_metadata_df:")
print([col for col in parsed_peakvisor.columns if col not in osm_metadata_df.columns])



Columns in osm_metadata_df but missing in parsed_peakvisor_df:
['mountain_id', 'name_original', 'name_clean', 'marker_size', 'coord', 'lat', 'lon', 'alt_names', 'source', 'coord_source', 'elev_raw', 'osm_id', 'osm_type', 'osm_tags', 'data_quality_notes', 'geometry', 'index_right', 'prov_psgc_code', 'region_name', 'region_code', 'prov_gis_original', 'region_gis_original', 'source_file', 'admin_note']

Columns in parsed_peakvisor but not in osm_metadata_df:
['peakvisor_name', 'peakvisor_elev', 'peakvisor_prom', 'url_source', 'keywords_found', 'detail_scrape_status', 'detail_text', 'detail_text_clean', 'pv_elev_from_page', 'pv_prom_from_page', 'peakvisor_lat', 'peakvisor_lon', 'peakvisor_location_text', 'peakvisor_about_text', 'has_trail_info', 'has_volcano_info', 'trail_sentence', 'volcano_sentence', 'pv_final_elev', 'pv_final_prom', 'pv_elev_from_page_v2', 'peakvisor_elev_source_unit', 'pv_prom_from_page_v2', 'peakvisor_prom_source_unit', 'peakvisor_elev_m', 'peakvisor_prom_m', 'peakvis

In [154]:
parsed_peakvisor["enrichment_source"] = "PeakVisor"

parsed_peakvisor["peakvisor_enrichment_status"] = parsed_peakvisor.apply(
    lambda row:
        "strong_enrichment"
        if pd.notna(row.get("peakvisor_elev_m")) and pd.notna(row.get("peakvisor_prom_m")) and pd.notna(row.get("detail_url"))
        else "partial_enrichment"
        if pd.notna(row.get("detail_url"))
        else "not_enriched",
    axis=1
)

In [155]:
parsed_peakvisor["peakvisor_elev_validation_flag"].value_counts()
parsed_peakvisor["peakvisor_prom_source_unit"].value_counts()
parsed_peakvisor["peakvisor_elev_source_unit"].value_counts()

,count
peakvisor_elev_source_unit,
m_from_title,96
not_found,2
m_from_about,1


# Enrich top 100 medium mountains

In [156]:
#Load your current masterfile
import pandas as pd

master_df = pd.read_csv("philippines_mountains_validated_quickmap_dedup.csv")

print(master_df.shape)
master_df.head()

(2512, 44)


,mountain_id,name,name_original,source,source.1,name_clean,alt_names,elev,prom,marker_size,...,detail_url,name_quickmap,lat_quickmap,lon_quickmap,elev_quickmap,quickmap_match,quickmap_distance_km,quickmap_validation_note,quickmap_status,quickmap_confidence
0,387_15.9003_120.9738,Mount 387,Mount 387,OpenStreetMap,OpenStreetMap,387,[],777.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,OpenStreetMap,OpenStreetMap,abao,[],2527.0,NaN,large,...,NaN,Mount Abao,16.8333,120.91670,2524.0,True,0.222378,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,OpenStreetMap,OpenStreetMap,abel's peak,[],1464.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,OpenStreetMap,OpenStreetMap,abnataclayong,[],370.0,NaN,small,...,NaN,Mount Abnataclayong,5.6750,125.36694,370.0,True,0.000443,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,OpenStreetMap,OpenStreetMap,aboabo,[],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match


In [157]:
# Top 100 Medium mountains for PeakVisor

# Make sure elevation is numeric
master_df["elev"] = pd.to_numeric(master_df["elev"], errors="coerce")

priority_medium_mountains = (
    master_df[
        (master_df["elev"] > 500) &
        (master_df["elev"] < 1500)
    ]
    .sort_values(
        by="elev",
        ascending=False,
        na_position="last"
    )
    .head(100)
    .copy()
)

priority_medium_mountains[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "quickmap_status",
    "quickmap_confidence"
]].head(30)

,name,elev,prom,is_volc,prov,region,isl_grp,quickmap_status,quickmap_confidence
1971,Rene's Peak,1493.0,NaN,NaN,Romblon,MIMAROPA,Luzon,Not found in QuickMap,No QuickMap match
1106,Kit's Peak,1492.0,NaN,NaN,Romblon,MIMAROPA,Luzon,Not found in QuickMap,No QuickMap match
2250,Mount Tallulah,1488.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,Validated by QuickMap,High confidence
1324,Mount Magolo,1486.0,NaN,NaN,South Cotabato,Region XII,Mindanao,Validated by QuickMap,High confidence
462,Mount Burburungan,1482.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,Validated by QuickMap,High confidence
982,Mount Irene,1479.0,NaN,NaN,Negros Oriental,NIR,Visayas,Not found in QuickMap,No QuickMap match
938,Mount Igbanig,1477.0,NaN,NaN,Antique,Region VI,Visayas,Not found in QuickMap,No QuickMap match
2160,Stripe Peak,1475.0,NaN,NaN,Palawan,MIMAROPA,Luzon,Validated by QuickMap,High confidence
2142,Mount Sonogong,1474.0,NaN,NaN,Antique,Region VI,Visayas,Validated by QuickMap,High confidence
1241,Lorna's Peak,1472.0,NaN,NaN,Romblon,MIMAROPA,Luzon,Not found in QuickMap,No QuickMap match


In [158]:
# Check which medium mountains need more PeakVisor enrichment

priority_medium_needs_peakvisor = priority_medium_mountains[
    priority_medium_mountains["prom"].isna()
    | priority_medium_mountains["is_volc"].isna()
    | priority_medium_mountains["detail_url"].isna()
].copy()

print("Medium mountains needing PeakVisor enrichment:", len(priority_medium_needs_peakvisor))

priority_medium_needs_peakvisor[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "detail_url",
    "prov",
    "region",
    "isl_grp"
]].head(30)

Medium mountains needing PeakVisor enrichment: 100


,name,elev,prom,is_volc,detail_url,prov,region,isl_grp
1971,Rene's Peak,1493.0,NaN,NaN,NaN,Romblon,MIMAROPA,Luzon
1106,Kit's Peak,1492.0,NaN,NaN,NaN,Romblon,MIMAROPA,Luzon
2250,Mount Tallulah,1488.0,NaN,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon
1324,Mount Magolo,1486.0,NaN,NaN,NaN,South Cotabato,Region XII,Mindanao
462,Mount Burburungan,1482.0,NaN,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon
982,Mount Irene,1479.0,NaN,NaN,NaN,Negros Oriental,NIR,Visayas
938,Mount Igbanig,1477.0,NaN,NaN,NaN,Antique,Region VI,Visayas
2160,Stripe Peak,1475.0,NaN,NaN,NaN,Palawan,MIMAROPA,Luzon
2142,Mount Sonogong,1474.0,NaN,NaN,NaN,Antique,Region VI,Visayas
1241,Lorna's Peak,1472.0,NaN,NaN,NaN,Romblon,MIMAROPA,Luzon


In [159]:
#Setting up helpers
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PhilippinesMountainsPortfolioProject/1.0)"
}

PEAKVISOR_PH_URL = "https://peakvisor.com/adm/philippines.html"

KEYWORDS = [
    "elevation", "prominence", "mountain", "summit",
    "volcano", "volcanic", "stratovolcano",
    "hiking", "trail", "trek", "climb", "route",
    "range", "mountain range"
]

def clean_text(text):
    return (
        str(text)
        .replace("\u2060", " ")
        .replace("\xa0", " ")
        .replace("\n", " ")
        .replace("\t", " ")
        .strip()
    )

def numbers_only(text):
    return int(re.sub(r"[^\d]", "", str(text)))

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = name.replace("mount ", "").replace("mt. ", "").replace("mt ", "")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

def find_keywords(text):
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS if kw in text_lower]

In [160]:
#Scrape the PeakVisor Philippines page links
response = requests.get(PEAKVISOR_PH_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

peakvisor_rows = []

for a in soup.find_all("a", href=True):
    text = clean_text(a.get_text(" ", strip=True))
    href = a["href"]

    pattern = r"^(.*?)\s+([\d\s,]+)\s*m\s*\(prom:\s*([\d\s,]+)\s*m\s*\)"
    match = re.search(pattern, text)

    if match:
        peakvisor_rows.append({
            "peakvisor_name": match.group(1).strip(),
            "peakvisor_elev": numbers_only(match.group(2)),
            "peakvisor_prom": numbers_only(match.group(3)),
            "peakvisor_detail_url": urljoin(PEAKVISOR_PH_URL, href),
            "peakvisor_url_source": "country_index"
        })

peakvisor_index = pd.DataFrame(peakvisor_rows).drop_duplicates()
peakvisor_index["name_clean"] = peakvisor_index["peakvisor_name"].apply(normalize_name)

print("PeakVisor index rows found:", len(peakvisor_index))
peakvisor_index.head(10)

PeakVisor index rows found: 17


,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_url_source,name_clean
0,Mount Apo,2954,2954,https://peakvisor.com/peak/mount-apo.html,country_index,apo
1,87 Degrees,2940,17,https://peakvisor.com/peak/87-degrees.html,country_index,87 degrees
2,Mount Dulang-Dulang,2938,2444,https://peakvisor.com/peak/mount-kitanglad.html,country_index,dulang dulang
3,Mount Pulag,2928,2928,https://peakvisor.com/peak/mount-pulag.html,country_index,pulag
4,Mount Kitanglad,2899,383,https://peakvisor.com/peak/mount-kitanglad-20z...,country_index,kitanglad
5,Mount Kalatungan,2874,1495,https://peakvisor.com/peak/mt-kalatungan.html,country_index,kalatungan
6,Mt. Lumpanag (Wiji),2819,281,https://peakvisor.com/peak/mt-makaupao-wiji.html,country_index,lumpanag wiji
7,Mount Piapayungan,2815,1595,https://peakvisor.com/peak/mount-piapayungan.html,country_index,piapayungan
8,Mount Maagnaw,2742,331,https://peakvisor.com/peak/mount-ma-agnaw.html,country_index,maagnaw
9,Pual,2725,110,https://peakvisor.com/peak/pual.html,country_index,pual


In [161]:
#Match my priority monutains to peakvisor links
priority = priority_medium_needs_peakvisor.copy()
priority["name_clean"] = priority["name"].apply(normalize_name)

priority_medium_matched = priority.merge(
    peakvisor_index,
    on="name_clean",
    how="left"
)

priority_medium_matched["peakvisor_match_status"] = priority_medium_matched["peakvisor_detail_url"].apply(
    lambda x: "matched_country_index" if pd.notna(x) else "not_found_in_country_index"
)

print("Priority mountains:", len(priority_medium_matched))
print(priority_medium_matched["peakvisor_match_status"].value_counts())

priority_medium_matched[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "peakvisor_match_status"
]].head(30)

Priority mountains: 100
peakvisor_match_status
not_found_in_country_index    100
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_match_status
0,Rene's Peak,1493.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
1,Kit's Peak,1492.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
2,Mount Tallulah,1488.0,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
3,Mount Magolo,1486.0,NaN,South Cotabato,Region XII,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
4,Mount Burburungan,1482.0,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
5,Mount Irene,1479.0,NaN,Negros Oriental,NIR,Visayas,NaN,NaN,NaN,NaN,not_found_in_country_index
6,Mount Igbanig,1477.0,NaN,Antique,Region VI,Visayas,NaN,NaN,NaN,NaN,not_found_in_country_index
7,Stripe Peak,1475.0,NaN,Palawan,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
8,Mount Sonogong,1474.0,NaN,Antique,Region VI,Visayas,NaN,NaN,NaN,NaN,not_found_in_country_index
9,Lorna's Peak,1472.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index


In [162]:
#Generate possible links for those unmatched
def make_slug_candidates(name):
    raw = str(name).lower().strip()
    raw = re.sub(r"[^a-z0-9\s\-]", " ", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    no_mount = (
        raw.replace("mount ", "")
           .replace("mt. ", "")
           .replace("mt ", "")
           .strip()
    )

    candidates = [
        raw.replace(" ", "-"),
        no_mount.replace(" ", "-"),
        "mount-" + no_mount.replace(" ", "-"),
        "mt-" + no_mount.replace(" ", "-")
    ]

    return list(dict.fromkeys([c for c in candidates if c]))

def possible_peakvisor_urls(name):
    urls = []

    for slug in make_slug_candidates(name):
        urls.extend([
            f"https://peakvisor.com/peak/{slug}.html",
            f"https://peakvisor.com/mountain/{slug}.html"
        ])

    return urls

# Quick test
possible_peakvisor_urls("Mount Apo")

['https://peakvisor.com/peak/mount-apo.html',
 'https://peakvisor.com/mountain/mount-apo.html',
 'https://peakvisor.com/peak/apo.html',
 'https://peakvisor.com/mountain/apo.html',
 'https://peakvisor.com/peak/mt-apo.html',
 'https://peakvisor.com/mountain/mt-apo.html']

In [163]:
#Test unmatchedd mountains for guessed urls
def page_matches_mountain(text, mountain_name):
    text_lower = str(text).lower()

    name_clean = normalize_name(mountain_name)
    name_words = name_clean.split()

    found_keywords = find_keywords(text)

    # Good if page contains the mountain name
    name_in_page = name_clean in normalize_name(text)

    # Backup check: important name words appear
    important_words_found = sum(1 for word in name_words if len(word) > 2 and word in text_lower)

    is_useful = name_in_page or important_words_found >= 1 or len(found_keywords) >= 2

    return is_useful, found_keywords

def find_url_by_slug_guess(name):
    for test_url in possible_peakvisor_urls(name):
        try:
            r = requests.get(test_url, headers=headers, timeout=20)

            if r.status_code != 200:
                continue

            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)

            is_useful, found_keywords = page_matches_mountain(detail_text, name)

            if is_useful:
                return {
                    "guessed_detail_url": test_url,
                    "url_guess_status": "found_by_slug_guess",
                    "keywords_found": ", ".join(found_keywords),
                    "detail_text": detail_text
                }

            time.sleep(1)

        except Exception:
            continue

    return {
        "guessed_detail_url": None,
        "url_guess_status": "not_found_after_url_guess",
        "keywords_found": None,
        "detail_text": None
    }

In [164]:
# Apply URL guessing to unmatched MEDIUM mountains

unmatched_medium = priority_medium_matched[
    priority_medium_matched["peakvisor_detail_url"].isna()
].copy()

guess_medium_rows = []

print("Unmatched medium mountains to URL-guess:", len(unmatched_medium))

for _, row in unmatched_medium.iterrows():
    result = find_url_by_slug_guess(row["name"])

    guess_medium_rows.append({
        "name": row["name"],
        "guessed_detail_url": result["guessed_detail_url"],
        "url_guess_status": result["url_guess_status"],
        "guess_keywords_found": result["keywords_found"],
        "guessed_detail_text": result["detail_text"]
    })

    print(row["name"], "→", result["url_guess_status"])
    time.sleep(2)

url_guess_medium_df = pd.DataFrame(guess_medium_rows)

url_guess_medium_df.head()

Unmatched medium mountains to URL-guess: 100
Rene's Peak → found_by_slug_guess
Kit's Peak → found_by_slug_guess
Mount Tallulah → found_by_slug_guess
Mount Magolo → found_by_slug_guess
Mount Burburungan → found_by_slug_guess
Mount Irene → found_by_slug_guess
Mount Igbanig → found_by_slug_guess
Stripe Peak → found_by_slug_guess
Mount Sonogong → found_by_slug_guess
Lorna's Peak → found_by_slug_guess
Mount Sinaka → found_by_slug_guess
Mount Inoman → found_by_slug_guess
Mount Tugew → found_by_slug_guess
Mount San Cristobal → found_by_slug_guess
Mount Magampao → found_by_slug_guess
Mount Nara → found_by_slug_guess
Mount Linao → found_by_slug_guess
Abel's Peak → found_by_slug_guess
Mount Irid → found_by_slug_guess
Volcanic Peak → found_by_slug_guess
Mount Otundo → found_by_slug_guess
Buribid Peak → found_by_slug_guess
Mount Maliz → found_by_slug_guess
Mount Salat → found_by_slug_guess
Mount Calibugon → found_by_slug_guess
Mount Madocay → found_by_slug_guess
Mount Portoc → found_by_slug_guess


,name,guessed_detail_url,url_guess_status,guess_keywords_found,guessed_detail_text
0,Rene's Peak,https://peakvisor.com/peak/rene-s-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Rene's Peak (1,523 m) – Philippines | PeakViso..."
1,Kit's Peak,https://peakvisor.com/peak/kit-s-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Kit's Peak (1,525 m) – Philippines | PeakVisor..."
2,Mount Tallulah,https://peakvisor.com/peak/mount-tallulah.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Mount Tallulah (1,495 m) – Philippines | PeakV..."
3,Mount Magolo,https://peakvisor.com/peak/mount-magolo.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Mount Magolo (1,487 m) – Philippines | PeakVis..."
4,Mount Burburungan,https://peakvisor.com/peak/mount-burburungan.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Abongabong (792 m) – Philippines | PeakV...


In [165]:
# Combine country-index URLs and guessed URLs for MEDIUM mountains

priority_medium_urls_final = priority_medium_matched.merge(
    url_guess_medium_df,
    on="name",
    how="left"
)

priority_medium_urls_final["final_detail_url"] = priority_medium_urls_final["peakvisor_detail_url"].fillna(
    priority_medium_urls_final["guessed_detail_url"]
)

priority_medium_urls_final["final_url_source"] = priority_medium_urls_final.apply(
    lambda row:
        "country_index"
        if pd.notna(row["peakvisor_detail_url"])
        else ("slug_guess" if pd.notna(row["guessed_detail_url"]) else "not_found"),
    axis=1
)

priority_medium_urls_final["final_match_status"] = priority_medium_urls_final["final_detail_url"].apply(
    lambda x: "has_detail_url" if pd.notna(x) else "not_found_after_all_methods"
)

print(priority_medium_urls_final["final_url_source"].value_counts())
print(priority_medium_urls_final["final_match_status"].value_counts())

priority_medium_urls_final[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "guessed_detail_url",
    "final_detail_url", "final_url_source", "final_match_status"
]].head(50)

final_url_source
slug_guess    103
not_found       1
Name: count, dtype: int64
final_match_status
has_detail_url                 103
not_found_after_all_methods      1
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,guessed_detail_url,final_detail_url,final_url_source,final_match_status
0,Rene's Peak,1493.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/rene-s-peak.html,https://peakvisor.com/peak/rene-s-peak.html,slug_guess,has_detail_url
1,Kit's Peak,1492.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/kit-s-peak.html,https://peakvisor.com/peak/kit-s-peak.html,slug_guess,has_detail_url
2,Mount Tallulah,1488.0,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-tallulah.html,https://peakvisor.com/peak/mount-tallulah.html,slug_guess,has_detail_url
3,Mount Magolo,1486.0,NaN,South Cotabato,Region XII,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-magolo.html,https://peakvisor.com/peak/mount-magolo.html,slug_guess,has_detail_url
4,Mount Burburungan,1482.0,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-burburungan.html,https://peakvisor.com/peak/mount-burburungan.html,slug_guess,has_detail_url
5,Mount Irene,1479.0,NaN,Negros Oriental,NIR,Visayas,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-irene.html,https://peakvisor.com/peak/mount-irene.html,slug_guess,has_detail_url
6,Mount Igbanig,1477.0,NaN,Antique,Region VI,Visayas,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-igbanig.html,https://peakvisor.com/peak/mount-igbanig.html,slug_guess,has_detail_url
7,Stripe Peak,1475.0,NaN,Palawan,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/stripe-peak.html,https://peakvisor.com/peak/stripe-peak.html,slug_guess,has_detail_url
8,Mount Sonogong,1474.0,NaN,Antique,Region VI,Visayas,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-sonogong.html,https://peakvisor.com/peak/mount-sonogong.html,slug_guess,has_detail_url
9,Lorna's Peak,1472.0,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/lorna-s-peak.html,https://peakvisor.com/peak/lorna-s-peak.html,slug_guess,has_detail_url


In [166]:
# Scrape final detail URLs one by one for MEDIUM mountains
# raw scraped page text

detail_medium_rows = []

to_scrape_medium = priority_medium_urls_final[
    priority_medium_urls_final["final_detail_url"].notna()
].copy()

print("Final medium detail pages available:", len(to_scrape_medium))

for _, row in to_scrape_medium.iterrows():
    try:
        # If detail text already came from slug guessing, reuse it
        if row["final_url_source"] == "slug_guess" and pd.notna(row.get("guessed_detail_text")):
            detail_text = row["guessed_detail_text"]
        else:
            r = requests.get(row["final_detail_url"], headers=headers, timeout=30)
            r.raise_for_status()
            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)
            time.sleep(2)

        found_keywords = find_keywords(detail_text)

        detail_medium_rows.append({
            "name": row["name"],
            "elev": row["elev"],
            "prom": row["prom"],
            "is_volc": row["is_volc"],
            "prov": row["prov"],
            "region": row["region"],
            "isl_grp": row["isl_grp"],
            "peakvisor_name": row.get("peakvisor_name"),
            "peakvisor_elev": row.get("peakvisor_elev"),
            "peakvisor_prom": row.get("peakvisor_prom"),
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "keywords_found": ", ".join(found_keywords),
            "detail_scrape_status": "scraped_with_keywords" if found_keywords else "scraped_no_keywords",
            "detail_text": detail_text
        })

        print(row["name"], "→ scraped")

    except Exception as e:
        detail_medium_rows.append({
            "name": row["name"],
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "detail_scrape_status": "failed",
            "error": str(e)
        })

        print("Failed:", row["name"], e)

priority_medium_detail_pages = pd.DataFrame(detail_medium_rows)

priority_medium_detail_pages.head()

Final medium detail pages available: 103
Rene's Peak → scraped
Kit's Peak → scraped
Mount Tallulah → scraped
Mount Magolo → scraped
Mount Burburungan → scraped
Mount Irene → scraped
Mount Igbanig → scraped
Stripe Peak → scraped
Mount Sonogong → scraped
Lorna's Peak → scraped
Mount Sinaka → scraped
Mount Inoman → scraped
Mount Tugew → scraped
Mount San Cristobal → scraped
Mount Magampao → scraped
Mount Nara → scraped
Mount Linao → scraped
Abel's Peak → scraped
Mount Irid → scraped
Volcanic Peak → scraped
Mount Otundo → scraped
Buribid Peak → scraped
Mount Maliz → scraped
Mount Salat → scraped
Mount Calibugon → scraped
Mount Madocay → scraped
Mount Portoc → scraped
Mount Kadang → scraped
Mount Pao → scraped
Mount Pao → scraped
Mount Baliagan → scraped
Mount Pitot Kalabaw → scraped
Mount Bugdan → scraped
Mount Cayudungan → scraped
Mount Buga → scraped
Mount Peak 2 → scraped
Mount Sugarloaf → scraped
Mount Iglit → scraped
Mount Caladang → scraped
Mount Balatic → scraped
Mount Macaayao → sc

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,detail_url,url_source,keywords_found,detail_scrape_status,detail_text
0,Rene's Peak,1493.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/rene-s-peak.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Rene's Peak (1,523 m) – Philippines | PeakViso..."
1,Kit's Peak,1492.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/kit-s-peak.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Kit's Peak (1,525 m) – Philippines | PeakVisor..."
2,Mount Tallulah,1488.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/mount-tallulah.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Mount Tallulah (1,495 m) – Philippines | PeakV..."
3,Mount Magolo,1486.0,NaN,NaN,South Cotabato,Region XII,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/mount-magolo.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Mount Magolo (1,487 m) – Philippines | PeakVis..."
4,Mount Burburungan,1482.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/mount-burburungan.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Abongabong (792 m) – Philippines | PeakV...


In [167]:
# Applying Main parse for MEDIUM PeakVisor detail pages

parsed_medium_peakvisor = priority_medium_detail_pages.copy()

parsed_medium_peakvisor["detail_text_clean"] = parsed_medium_peakvisor["detail_text"].apply(
    clean_peakvisor_text
)

parsed_medium_peakvisor["pv_elev_from_page"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_elevation_from_peakvisor_text
)

parsed_medium_peakvisor["pv_prom_from_page"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_prominence_from_peakvisor_text
)

parsed_medium_peakvisor[["pv_lat", "pv_lon"]] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_coordinates_from_peakvisor_text
)

parsed_medium_peakvisor["pv_location_text"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_location_block
)

parsed_medium_peakvisor["pv_about_text"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_about_text
)

parsed_medium_peakvisor["pv_has_trail_info"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    has_trail_info
)

parsed_medium_peakvisor["pv_has_volcano_info"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    has_volcano_info
)

parsed_medium_peakvisor["pv_trail_sentence"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_trail_sentence
)

parsed_medium_peakvisor["pv_volcano_sentence"] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_volcano_sentence
)

# Unit-aware elevation/prominence parsing
parsed_medium_peakvisor[["pv_elev_from_page_v2", "pv_elev_unit_source"]] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_elevation_unit_aware
)

parsed_medium_peakvisor[["pv_prom_from_page_v2", "pv_prom_unit_source"]] = parsed_medium_peakvisor["detail_text_clean"].apply(
    extract_prominence_unit_aware
)

# Final PeakVisor values
parsed_medium_peakvisor["pv_final_elev_v2"] = parsed_medium_peakvisor["pv_elev_from_page_v2"].fillna(
    parsed_medium_peakvisor["peakvisor_elev"]
)

parsed_medium_peakvisor["pv_final_prom_v2"] = parsed_medium_peakvisor["pv_prom_from_page_v2"].fillna(
    parsed_medium_peakvisor["peakvisor_prom"]
)

# Validation flags
parsed_medium_peakvisor["pv_elev_difference_v2"] = (
    parsed_medium_peakvisor["pv_final_elev_v2"] - parsed_medium_peakvisor["elev"]
)

def elevation_validation_flag_v2(row):
    if pd.isna(row["elev"]):
        return "no_original_elevation_to_compare"
    elif pd.isna(row["pv_final_elev_v2"]):
        return "no_peakvisor_elevation"

    diff = row["pv_final_elev_v2"] - row["elev"]

    if abs(diff) <= 30:
        return "close_match"
    elif abs(diff) <= 100:
        return "minor_difference"
    else:
        return "needs_review"

parsed_medium_peakvisor["pv_elev_validation_flag_v2"] = parsed_medium_peakvisor.apply(
    elevation_validation_flag_v2,
    axis=1
)

parsed_medium_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_trail_sentence,pv_volcano_sentence,pv_elev_from_page_v2,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,pv_final_elev_v2,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2
0,Rene's Peak,1493.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1523,m_from_title,11,m_from_about,1523,11,30.0,close_match
1,Kit's Peak,1492.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1525,m_from_title,40,m_from_about,1525,40,33.0,minor_difference
2,Mount Tallulah,1488.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1495,m_from_title,591,m_from_about,1495,591,7.0,close_match
3,Mount Magolo,1486.0,NaN,NaN,South Cotabato,Region XII,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1487,m_from_title,287,m_from_about,1487,287,1.0,close_match
4,Mount Burburungan,1482.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,792,m_from_title,15,m_from_about,792,15,-690.0,needs_review


In [168]:
print("Rows:", len(parsed_medium_peakvisor))

print("\nElevation parsed from page:")
print(parsed_medium_peakvisor["pv_elev_from_page"].notna().value_counts())

print("\nProminence parsed from page:")
print(parsed_medium_peakvisor["pv_prom_from_page"].notna().value_counts())

print("\nAbout text parsed:")
print(parsed_medium_peakvisor["pv_about_text"].notna().value_counts())

print("\nLocation parsed:")
print(parsed_medium_peakvisor["pv_location_text"].notna().value_counts())

Rows: 103

Elevation parsed from page:
pv_elev_from_page
True    103
Name: count, dtype: int64

Prominence parsed from page:
pv_prom_from_page
True     97
False     6
Name: count, dtype: int64

About text parsed:
pv_about_text
True    103
Name: count, dtype: int64

Location parsed:
pv_location_text
True    103
Name: count, dtype: int64


In [169]:
print("\nElevation unit-aware parsed:")
print(parsed_medium_peakvisor["pv_elev_from_page_v2"].notna().value_counts())

print("\nProminence unit-aware parsed:")
print(parsed_medium_peakvisor["pv_prom_from_page_v2"].notna().value_counts())

print("\nElevation validation:")
print(parsed_medium_peakvisor["pv_elev_validation_flag_v2"].value_counts(dropna=False))


Elevation unit-aware parsed:
pv_elev_from_page_v2
True    103
Name: count, dtype: int64

Prominence unit-aware parsed:
pv_prom_from_page_v2
True    103
Name: count, dtype: int64

Elevation validation:
pv_elev_validation_flag_v2
close_match         72
needs_review        22
minor_difference     9
Name: count, dtype: int64


In [170]:
# View medium rows where unit-aware parsing still needs review

parse_medium_review_v2 = parsed_medium_peakvisor[
    (parsed_medium_peakvisor["pv_final_elev_v2"].isna()) |
    (parsed_medium_peakvisor["pv_final_prom_v2"].isna()) |
    (parsed_medium_peakvisor["pv_about_text"].isna()) |
    (parsed_medium_peakvisor["pv_elev_validation_flag_v2"] == "needs_review")
].copy()

parse_medium_review_v2[[
    "name",
    "detail_url",
    "elev",
    "pv_elev_from_page_v2",
    "pv_elev_unit_source",
    "peakvisor_elev",
    "pv_final_elev_v2",
    "pv_prom_from_page_v2",
    "pv_prom_unit_source",
    "peakvisor_prom",
    "pv_final_prom_v2",
    "pv_elev_difference_v2",
    "pv_elev_validation_flag_v2",
    "pv_about_text",
    "detail_text"
]].head(10)

,name,detail_url,elev,pv_elev_from_page_v2,pv_elev_unit_source,peakvisor_elev,pv_final_elev_v2,pv_prom_from_page_v2,pv_prom_unit_source,peakvisor_prom,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2,pv_about_text,detail_text
4,Mount Burburungan,https://peakvisor.com/peak/mount-burburungan.html,1482.0,792,m_from_title,NaN,792,15,m_from_about,NaN,15,-690.0,needs_review,Mount Abongabong (792m/2 598ft a.s.l.) is a mo...,Mount Abongabong (792 m) – Philippines | PeakV...
5,Mount Irene,https://peakvisor.com/peak/mount-irene.html,1479.0,1860,m_from_about,NaN,1860,1015,m_from_about,NaN,1015,381.0,needs_review,Mount Irene (1 860m/6 102ft a.s.l.) is a mount...,"Mount Irene (1,860 m) – New Zealand | PeakViso..."
8,Mount Sonogong,https://peakvisor.com/peak/mount-sonogong.html,1474.0,1935,m_from_title,NaN,1935,575,m_from_about,NaN,575,461.0,needs_review,Mount Baloy (1 935m/6 348ft a.s.l.) is a mount...,"Mount Baloy (1,935 m) – Philippines | PeakViso..."
22,Mount Maliz,https://peakvisor.com/peak/mount-maliz.html,1450.0,1294,m_from_title,NaN,1294,157,m_from_about,NaN,157,-156.0,needs_review,Mount Maliz (1 294m/4 245ft a.s.l.) is a mount...,"Mount Maliz (1,294 m) – Philippines | PeakViso..."
28,Mount Pao,https://peakvisor.com/peak/mount-pao.html,1441.0,1323,m_from_title,NaN,1323,409,m_from_about,NaN,409,-118.0,needs_review,Mount Pao (1 323m/4 341ft a.s.l.) is a mountai...,"Mount Pao (1,323 m) – Philippines | PeakVisor ..."
29,Mount Pao,https://peakvisor.com/peak/mount-pao.html,1441.0,1323,m_from_title,NaN,1323,409,m_from_about,NaN,409,-118.0,needs_review,Mount Pao (1 323m/4 341ft a.s.l.) is a mountai...,"Mount Pao (1,323 m) – Philippines | PeakVisor ..."
34,Mount Buga,https://peakvisor.com/peak/mount-buga.html,1435.0,528,m_from_title,NaN,528,291,m_from_about,NaN,291,-907.0,needs_review,Mount Buga (528m/1 732ft a.s.l.) is a mountain...,Mount Buga (528 m) – Philippines | PeakVisor W...
35,Mount Peak 2,https://peakvisor.com/peak/peak-2.html,1434.0,2830,m_from_about,NaN,2830,89,m_from_about,NaN,89,1396.0,needs_review,Peak 2 (2 830m/9 285ft a.s.l.) is a mountain i...,"Peak 2 (2,830 m) – Rocky Mountains, Canada | P..."
36,Mount Sugarloaf,https://peakvisor.com/peak/mount-sugarloaf.html,1432.0,309,m_from_about,NaN,309,63,m_from_about,NaN,63,-1123.0,needs_review,Mount Sugarloaf (309m/1 014ft a.s.l.) is a hil...,Mount Sugarloaf (309 m) – Great Dividing Range...
43,Mount Peak 1,https://peakvisor.com/peak/peak-1.html,1420.0,3900,m_from_about,NaN,3900,80,m_from_about,NaN,80,2480.0,needs_review,Peak 1 (3 900m/12 795ft a.s.l.) is a mountain ...,"Peak 1 (3,900 m) – Mosquito Range, USA | PeakV..."


In [171]:
# No match at all after all methods from the MEDIUM priority mountains

priority_medium_not_found = priority_medium_urls_final[
    priority_medium_urls_final["final_match_status"] == "not_found_after_all_methods"
].copy()

priority_medium_not_found.to_csv(
    "priority_medium_mountains_not_found_in_peakvisor.csv",
    index=False
)

print("Medium not found after all methods:", len(priority_medium_not_found))

priority_medium_not_found[[
    "name", "elev", "prov", "region", "isl_grp"
]].head(50)

Medium not found after all methods: 1


,name,elev,prov,region,isl_grp
64,Mount Buñeco,1370.0,Benguet,CAR,Luzon


In [172]:
print("Medium rows needing review:", len(parse_medium_review_v2))

print("\nMedium elevation unit source:")
print(parsed_medium_peakvisor["pv_elev_unit_source"].value_counts(dropna=False))

print("\nMedium prominence unit source:")
print(parsed_medium_peakvisor["pv_prom_unit_source"].value_counts(dropna=False))

print("\nMedium validation flags:")
print(parsed_medium_peakvisor["pv_elev_validation_flag_v2"].value_counts(dropna=False))

Medium rows needing review: 22

Medium elevation unit source:
pv_elev_unit_source
m_from_title    97
m_from_about     6
Name: count, dtype: int64

Medium prominence unit source:
pv_prom_unit_source
m_from_about    99
m_from_text      4
Name: count, dtype: int64

Medium validation flags:
pv_elev_validation_flag_v2
close_match         72
needs_review        22
minor_difference     9
Name: count, dtype: int64


In [173]:
# Rename MEDIUM PeakVisor parsed columns

parsed_medium_peakvisor = parsed_medium_peakvisor.rename(columns={
    "pv_final_elev_v2": "peakvisor_elev_m",
    "pv_final_prom_v2": "peakvisor_prom_m",
    "pv_elev_unit_source": "peakvisor_elev_source_unit",
    "pv_prom_unit_source": "peakvisor_prom_source_unit",
    "pv_elev_difference_v2": "peakvisor_elev_difference_m",
    "pv_elev_validation_flag_v2": "peakvisor_elev_validation_flag",
    "pv_lat": "peakvisor_lat",
    "pv_lon": "peakvisor_lon",
    "pv_location_text": "peakvisor_location_text",
    "pv_about_text": "peakvisor_about_text",
    "pv_has_trail_info": "has_trail_info",
    "pv_has_volcano_info": "has_volcano_info",
    "pv_trail_sentence": "trail_sentence",
    "pv_volcano_sentence": "volcano_sentence"
})

parsed_medium_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,trail_sentence,volcano_sentence,pv_elev_from_page_v2,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,peakvisor_elev_difference_m,peakvisor_elev_validation_flag
0,Rene's Peak,1493.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1523,m_from_title,11,m_from_about,1523,11,30.0,close_match
1,Kit's Peak,1492.0,NaN,NaN,Romblon,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1525,m_from_title,40,m_from_about,1525,40,33.0,minor_difference
2,Mount Tallulah,1488.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1495,m_from_title,591,m_from_about,1495,591,7.0,close_match
3,Mount Magolo,1486.0,NaN,NaN,South Cotabato,Region XII,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1487,m_from_title,287,m_from_about,1487,287,1.0,close_match
4,Mount Burburungan,1482.0,NaN,NaN,Occidental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,792,m_from_title,15,m_from_about,792,15,-690.0,needs_review


In [174]:
#Safe Checkpoint
parsed_medium_peakvisor.to_csv(
    "priority_mountains_peakvisor_parsed_medium_peakvisor.csv",
    index=False
)

print("Saved final PeakVisor parsed Top 100 Medium enrichment file.")

Saved final PeakVisor parsed Top 100 Medium enrichment file.


# Enrich top 100 small mountains

In [175]:
#Load your current masterfile
import pandas as pd

master_df = pd.read_csv("philippines_mountains_validated_quickmap_dedup.csv")

print(master_df.shape)
master_df.head()

(2512, 44)


,mountain_id,name,name_original,source,source.1,name_clean,alt_names,elev,prom,marker_size,...,detail_url,name_quickmap,lat_quickmap,lon_quickmap,elev_quickmap,quickmap_match,quickmap_distance_km,quickmap_validation_note,quickmap_status,quickmap_confidence
0,387_15.9003_120.9738,Mount 387,Mount 387,OpenStreetMap,OpenStreetMap,387,[],777.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,OpenStreetMap,OpenStreetMap,abao,[],2527.0,NaN,large,...,NaN,Mount Abao,16.8333,120.91670,2524.0,True,0.222378,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,OpenStreetMap,OpenStreetMap,abel's peak,[],1464.0,NaN,medium,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,OpenStreetMap,OpenStreetMap,abnataclayong,[],370.0,NaN,small,...,NaN,Mount Abnataclayong,5.6750,125.36694,370.0,True,0.000443,QuickMap match: coordinate very close,Validated by QuickMap,High confidence
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,OpenStreetMap,OpenStreetMap,aboabo,[],NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,NaN,Not found in QuickMap/GeoNames,Not found in QuickMap,No QuickMap match


In [176]:
# Top 100 Small mountains for PeakVisor

# Make sure elevation is numeric
master_df["elev"] = pd.to_numeric(master_df["elev"], errors="coerce")

priority_small_mountains = (
    master_df[
        master_df["elev"] <= 500
    ]
    .sort_values(
        by="elev",
        ascending=False,
        na_position="last"
    )
    .head(100)
    .copy()
)

priority_small_mountains[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "quickmap_status",
    "quickmap_confidence"
]].head(30)

,name,elev,prom,is_volc,prov,region,isl_grp,quickmap_status,quickmap_confidence
1986,Mount Saamong,500.0,NaN,NaN,Northern Samar,Region VIII,Visayas,Not found in QuickMap,No QuickMap match
2110,Mount Simsiman,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,Validated by QuickMap,High confidence
1856,Mount Pigtuyuan,500.0,NaN,NaN,Bukidnon,Region X,Mindanao,Not found in QuickMap,No QuickMap match
1605,Mount Nacday,500.0,NaN,NaN,Zambales,Region III,Luzon,Not found in QuickMap,No QuickMap match
636,Mount Caypipili,500.0,NaN,NaN,Rizal,Region IV-A,Luzon,Not found in QuickMap,No QuickMap match
1180,Mount Lasubong,500.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,Not found in QuickMap,No QuickMap match
845,Mount Gakit,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,Validated by QuickMap,High confidence
1528,Mount Maybe,500.0,NaN,NaN,Zambales,Region III,Luzon,Not found in QuickMap,No QuickMap match
1794,Mount Panlubongan Mountain,498.0,NaN,NaN,Negros Occidental,NIR,Visayas,Validated by QuickMap,High confidence
834,Mount Fitzhugh Lee,498.0,NaN,NaN,Oriental Mindoro,MIMAROPA,Luzon,Not found in QuickMap,No QuickMap match


In [177]:
# Check which SMALL mountains need more PeakVisor enrichment

priority_small_needs_peakvisor = priority_small_mountains[
    priority_small_mountains["prom"].isna()
    | priority_small_mountains["is_volc"].isna()
    | priority_small_mountains["detail_url"].isna()
].copy()

print("Small mountains needing PeakVisor enrichment:", len(priority_small_needs_peakvisor))

priority_small_needs_peakvisor[[
    "name",
    "elev",
    "prom",
    "is_volc",
    "detail_url",
    "prov",
    "region",
    "isl_grp"
]].head(30)

Small mountains needing PeakVisor enrichment: 100


,name,elev,prom,is_volc,detail_url,prov,region,isl_grp
1986,Mount Saamong,500.0,NaN,NaN,NaN,Northern Samar,Region VIII,Visayas
2110,Mount Simsiman,500.0,NaN,NaN,NaN,Davao del Norte,Region XI,Mindanao
1856,Mount Pigtuyuan,500.0,NaN,NaN,NaN,Bukidnon,Region X,Mindanao
1605,Mount Nacday,500.0,NaN,NaN,NaN,Zambales,Region III,Luzon
636,Mount Caypipili,500.0,NaN,NaN,NaN,Rizal,Region IV-A,Luzon
1180,Mount Lasubong,500.0,NaN,NaN,NaN,Davao del Sur,Region XI,Mindanao
845,Mount Gakit,500.0,NaN,NaN,NaN,Davao del Norte,Region XI,Mindanao
1528,Mount Maybe,500.0,NaN,NaN,NaN,Zambales,Region III,Luzon
1794,Mount Panlubongan Mountain,498.0,NaN,NaN,NaN,Negros Occidental,NIR,Visayas
834,Mount Fitzhugh Lee,498.0,NaN,NaN,NaN,Oriental Mindoro,MIMAROPA,Luzon


In [178]:
#Setting up helpers
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PhilippinesMountainsPortfolioProject/1.0)"
}

PEAKVISOR_PH_URL = "https://peakvisor.com/adm/philippines.html"

KEYWORDS = [
    "elevation", "prominence", "mountain", "summit",
    "volcano", "volcanic", "stratovolcano",
    "hiking", "trail", "trek", "climb", "route",
    "range", "mountain range"
]

def clean_text(text):
    return (
        str(text)
        .replace("\u2060", " ")
        .replace("\xa0", " ")
        .replace("\n", " ")
        .replace("\t", " ")
        .strip()
    )

def numbers_only(text):
    return int(re.sub(r"[^\d]", "", str(text)))

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = name.replace("mount ", "").replace("mt. ", "").replace("mt ", "")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

def find_keywords(text):
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS if kw in text_lower]

In [179]:
#Scrape the PeakVisor Philippines page links
response = requests.get(PEAKVISOR_PH_URL, headers=headers, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

peakvisor_rows = []

for a in soup.find_all("a", href=True):
    text = clean_text(a.get_text(" ", strip=True))
    href = a["href"]

    pattern = r"^(.*?)\s+([\d\s,]+)\s*m\s*\(prom:\s*([\d\s,]+)\s*m\s*\)"
    match = re.search(pattern, text)

    if match:
        peakvisor_rows.append({
            "peakvisor_name": match.group(1).strip(),
            "peakvisor_elev": numbers_only(match.group(2)),
            "peakvisor_prom": numbers_only(match.group(3)),
            "peakvisor_detail_url": urljoin(PEAKVISOR_PH_URL, href),
            "peakvisor_url_source": "country_index"
        })

peakvisor_index = pd.DataFrame(peakvisor_rows).drop_duplicates()
peakvisor_index["name_clean"] = peakvisor_index["peakvisor_name"].apply(normalize_name)

print("PeakVisor index rows found:", len(peakvisor_index))
peakvisor_index.head(10)

PeakVisor index rows found: 17


,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_url_source,name_clean
0,Mount Apo,2954,2954,https://peakvisor.com/peak/mount-apo.html,country_index,apo
1,87 Degrees,2940,17,https://peakvisor.com/peak/87-degrees.html,country_index,87 degrees
2,Mount Dulang-Dulang,2938,2444,https://peakvisor.com/peak/mount-kitanglad.html,country_index,dulang dulang
3,Mount Pulag,2928,2928,https://peakvisor.com/peak/mount-pulag.html,country_index,pulag
4,Mount Kitanglad,2899,383,https://peakvisor.com/peak/mount-kitanglad-20z...,country_index,kitanglad
5,Mount Kalatungan,2874,1495,https://peakvisor.com/peak/mt-kalatungan.html,country_index,kalatungan
6,Mt. Lumpanag (Wiji),2819,281,https://peakvisor.com/peak/mt-makaupao-wiji.html,country_index,lumpanag wiji
7,Mount Piapayungan,2815,1595,https://peakvisor.com/peak/mount-piapayungan.html,country_index,piapayungan
8,Mount Maagnaw,2742,331,https://peakvisor.com/peak/mount-ma-agnaw.html,country_index,maagnaw
9,Pual,2725,110,https://peakvisor.com/peak/pual.html,country_index,pual


In [180]:
# Match SMALL priority mountains to PeakVisor country-index links

priority_small = priority_small_needs_peakvisor.copy()
priority_small["name_clean"] = priority_small["name"].apply(normalize_name)

priority_small_matched = priority_small.merge(
    peakvisor_index,
    on="name_clean",
    how="left"
)

priority_small_matched["peakvisor_match_status"] = priority_small_matched["peakvisor_detail_url"].apply(
    lambda x: "matched_country_index" if pd.notna(x) else "not_found_in_country_index"
)

print("Small priority mountains:", len(priority_small_matched))
print(priority_small_matched["peakvisor_match_status"].value_counts())

priority_small_matched[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "peakvisor_match_status"
]].head(30)

Small priority mountains: 100
peakvisor_match_status
not_found_in_country_index    100
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,peakvisor_match_status
0,Mount Saamong,500.0,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,NaN,not_found_in_country_index
1,Mount Simsiman,500.0,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
2,Mount Pigtuyuan,500.0,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
3,Mount Nacday,500.0,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
4,Mount Caypipili,500.0,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
5,Mount Lasubong,500.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
6,Mount Gakit,500.0,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,NaN,not_found_in_country_index
7,Mount Maybe,500.0,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index
8,Mount Panlubongan Mountain,498.0,NaN,Negros Occidental,NIR,Visayas,NaN,NaN,NaN,NaN,not_found_in_country_index
9,Mount Fitzhugh Lee,498.0,NaN,Oriental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,not_found_in_country_index


In [181]:
#Generate possible links for those unmatched
def make_slug_candidates(name):
    raw = str(name).lower().strip()
    raw = re.sub(r"[^a-z0-9\s\-]", " ", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    no_mount = (
        raw.replace("mount ", "")
           .replace("mt. ", "")
           .replace("mt ", "")
           .strip()
    )

    candidates = [
        raw.replace(" ", "-"),
        no_mount.replace(" ", "-"),
        "mount-" + no_mount.replace(" ", "-"),
        "mt-" + no_mount.replace(" ", "-")
    ]

    return list(dict.fromkeys([c for c in candidates if c]))

def possible_peakvisor_urls(name):
    urls = []

    for slug in make_slug_candidates(name):
        urls.extend([
            f"https://peakvisor.com/peak/{slug}.html",
            f"https://peakvisor.com/mountain/{slug}.html"
        ])

    return urls

# Quick test
possible_peakvisor_urls("Mount Apo")

['https://peakvisor.com/peak/mount-apo.html',
 'https://peakvisor.com/mountain/mount-apo.html',
 'https://peakvisor.com/peak/apo.html',
 'https://peakvisor.com/mountain/apo.html',
 'https://peakvisor.com/peak/mt-apo.html',
 'https://peakvisor.com/mountain/mt-apo.html']

In [182]:
#Test unmatchedd mountains for guessed urls
def page_matches_mountain(text, mountain_name):
    text_lower = str(text).lower()

    name_clean = normalize_name(mountain_name)
    name_words = name_clean.split()

    found_keywords = find_keywords(text)

    # Good if page contains the mountain name
    name_in_page = name_clean in normalize_name(text)

    # Backup check: important name words appear
    important_words_found = sum(1 for word in name_words if len(word) > 2 and word in text_lower)

    is_useful = name_in_page or important_words_found >= 1 or len(found_keywords) >= 2

    return is_useful, found_keywords

def find_url_by_slug_guess(name):
    for test_url in possible_peakvisor_urls(name):
        try:
            r = requests.get(test_url, headers=headers, timeout=20)

            if r.status_code != 200:
                continue

            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)

            is_useful, found_keywords = page_matches_mountain(detail_text, name)

            if is_useful:
                return {
                    "guessed_detail_url": test_url,
                    "url_guess_status": "found_by_slug_guess",
                    "keywords_found": ", ".join(found_keywords),
                    "detail_text": detail_text
                }

            time.sleep(1)

        except Exception:
            continue

    return {
        "guessed_detail_url": None,
        "url_guess_status": "not_found_after_url_guess",
        "keywords_found": None,
        "detail_text": None
    }

In [183]:
# Apply URL guessing to unmatched SMALL mountains

unmatched_small = priority_small_matched[
    priority_small_matched["peakvisor_detail_url"].isna()
].copy()

guess_small_rows = []

print("Unmatched small mountains to URL-guess:", len(unmatched_small))

for _, row in unmatched_small.iterrows():
    result = find_url_by_slug_guess(row["name"])

    guess_small_rows.append({
        "name": row["name"],
        "guessed_detail_url": result["guessed_detail_url"],
        "url_guess_status": result["url_guess_status"],
        "guess_keywords_found": result["keywords_found"],
        "guessed_detail_text": result["detail_text"]
    })

    print(row["name"], "→", result["url_guess_status"])
    time.sleep(2)

url_guess_small_df = pd.DataFrame(guess_small_rows)

url_guess_small_df.head()

Unmatched small mountains to URL-guess: 100
Mount Saamong → found_by_slug_guess
Mount Simsiman → found_by_slug_guess
Mount Pigtuyuan → found_by_slug_guess
Mount Nacday → found_by_slug_guess
Mount Caypipili → found_by_slug_guess
Mount Lasubong → found_by_slug_guess
Mount Gakit → found_by_slug_guess
Mount Maybe → found_by_slug_guess
Mount Panlubongan Mountain → found_by_slug_guess
Mount Fitzhugh Lee → found_by_slug_guess
Shark Fin Peak → found_by_slug_guess
Mount Monondon → found_by_slug_guess
Mount Salimbao → found_by_slug_guess
Mount Pingas → found_by_slug_guess
Mount Syniop → found_by_slug_guess
Knob Peak → found_by_slug_guess
Mount Balungabing → found_by_slug_guess
Mount Navaja Heights → found_by_slug_guess
Lubo Peak → found_by_slug_guess
Mount Pulantuna → found_by_slug_guess
Mount Santa Rita → found_by_slug_guess
Mount Gitna → found_by_slug_guess
Mount Mabilog → found_by_slug_guess
Mount Nabongsuran → found_by_slug_guess
Mount Maria → found_by_slug_guess
Mount Ynantagung → found_by_

,name,guessed_detail_url,url_guess_status,guess_keywords_found,guessed_detail_text
0,Mount Saamong,https://peakvisor.com/peak/mount-saamong.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Saamong (513 m) – Philippines | PeakViso...
1,Mount Simsiman,https://peakvisor.com/peak/mount-simsiman.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Simsiman (525 m) – Philippines | PeakVis...
2,Mount Pigtuyuan,https://peakvisor.com/peak/mount-pigtuyuan.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...",Mount Pigtuyuan (515 m) – Philippines | PeakVi...
3,Mount Nacday,https://peakvisor.com/peak/mount-nacday.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Calpalngan (420 m) – Philippines | PeakV...
4,Mount Caypipili,https://peakvisor.com/peak/mount-caypipili.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...",Mount Caypipili (518 m) – Philippines | PeakVi...


In [184]:
# Combine country-index URLs and guessed URLs for SMALL mountains

priority_small_urls_final = priority_small_matched.merge(
    url_guess_small_df,
    on="name",
    how="left"
)

priority_small_urls_final["final_detail_url"] = priority_small_urls_final["peakvisor_detail_url"].fillna(
    priority_small_urls_final["guessed_detail_url"]
)

priority_small_urls_final["final_url_source"] = priority_small_urls_final.apply(
    lambda row:
        "country_index"
        if pd.notna(row["peakvisor_detail_url"])
        else ("slug_guess" if pd.notna(row["guessed_detail_url"]) else "not_found"),
    axis=1
)

priority_small_urls_final["final_match_status"] = priority_small_urls_final["final_detail_url"].apply(
    lambda x: "has_detail_url" if pd.notna(x) else "not_found_after_all_methods"
)

print(priority_small_urls_final["final_url_source"].value_counts())
print(priority_small_urls_final["final_match_status"].value_counts())

priority_small_urls_final[[
    "name", "elev", "prom", "prov", "region", "isl_grp",
    "peakvisor_name", "peakvisor_elev", "peakvisor_prom",
    "peakvisor_detail_url", "guessed_detail_url",
    "final_detail_url", "final_url_source", "final_match_status"
]].head(50)

final_url_source
slug_guess    98
not_found      2
Name: count, dtype: int64
final_match_status
has_detail_url                 98
not_found_after_all_methods     2
Name: count, dtype: int64


,name,elev,prom,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,peakvisor_detail_url,guessed_detail_url,final_detail_url,final_url_source,final_match_status
0,Mount Saamong,500.0,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-saamong.html,https://peakvisor.com/peak/mount-saamong.html,slug_guess,has_detail_url
1,Mount Simsiman,500.0,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-simsiman.html,https://peakvisor.com/peak/mount-simsiman.html,slug_guess,has_detail_url
2,Mount Pigtuyuan,500.0,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-pigtuyuan.html,https://peakvisor.com/peak/mount-pigtuyuan.html,slug_guess,has_detail_url
3,Mount Nacday,500.0,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-nacday.html,https://peakvisor.com/peak/mount-nacday.html,slug_guess,has_detail_url
4,Mount Caypipili,500.0,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-caypipili.html,https://peakvisor.com/peak/mount-caypipili.html,slug_guess,has_detail_url
5,Mount Lasubong,500.0,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-lasubong.html,https://peakvisor.com/peak/mount-lasubong.html,slug_guess,has_detail_url
6,Mount Gakit,500.0,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-gakit.html,https://peakvisor.com/peak/mount-gakit.html,slug_guess,has_detail_url
7,Mount Maybe,500.0,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-maybe.html,https://peakvisor.com/peak/mount-maybe.html,slug_guess,has_detail_url
8,Mount Panlubongan Mountain,498.0,NaN,Negros Occidental,NIR,Visayas,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/panlubongan-mountai...,https://peakvisor.com/peak/panlubongan-mountai...,slug_guess,has_detail_url
9,Mount Fitzhugh Lee,498.0,NaN,Oriental Mindoro,MIMAROPA,Luzon,NaN,NaN,NaN,NaN,https://peakvisor.com/peak/mount-fitzhugh-lee....,https://peakvisor.com/peak/mount-fitzhugh-lee....,slug_guess,has_detail_url


In [185]:
# Scrape final detail URLs one by one for SMALL mountains
# raw scraped page text

detail_small_rows = []

to_scrape_small = priority_small_urls_final[
    priority_small_urls_final["final_detail_url"].notna()
].copy()

print("Final small detail pages available:", len(to_scrape_small))

for _, row in to_scrape_small.iterrows():
    try:
        # If detail text already came from slug guessing, reuse it
        if row["final_url_source"] == "slug_guess" and pd.notna(row.get("guessed_detail_text")):
            detail_text = row["guessed_detail_text"]
        else:
            r = requests.get(row["final_detail_url"], headers=headers, timeout=30)
            r.raise_for_status()
            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)
            time.sleep(2)

        found_keywords = find_keywords(detail_text)

        detail_small_rows.append({
            "name": row["name"],
            "elev": row["elev"],
            "prom": row["prom"],
            "is_volc": row["is_volc"],
            "prov": row["prov"],
            "region": row["region"],
            "isl_grp": row["isl_grp"],
            "peakvisor_name": row.get("peakvisor_name"),
            "peakvisor_elev": row.get("peakvisor_elev"),
            "peakvisor_prom": row.get("peakvisor_prom"),
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "keywords_found": ", ".join(found_keywords),
            "detail_scrape_status": "scraped_with_keywords" if found_keywords else "scraped_no_keywords",
            "detail_text": detail_text
        })

        print(row["name"], "→ scraped")

    except Exception as e:
        detail_small_rows.append({
            "name": row["name"],
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "detail_scrape_status": "failed",
            "error": str(e)
        })

        print("Failed:", row["name"], e)

priority_small_detail_pages = pd.DataFrame(detail_small_rows)

priority_small_detail_pages.head()

Final small detail pages available: 98
Mount Saamong → scraped
Mount Simsiman → scraped
Mount Pigtuyuan → scraped
Mount Nacday → scraped
Mount Caypipili → scraped
Mount Lasubong → scraped
Mount Gakit → scraped
Mount Maybe → scraped
Mount Panlubongan Mountain → scraped
Mount Fitzhugh Lee → scraped
Shark Fin Peak → scraped
Mount Monondon → scraped
Mount Salimbao → scraped
Mount Pingas → scraped
Mount Syniop → scraped
Knob Peak → scraped
Mount Balungabing → scraped
Mount Navaja Heights → scraped
Lubo Peak → scraped
Mount Pulantuna → scraped
Mount Santa Rita → scraped
Mount Gitna → scraped
Mount Mabilog → scraped
Mount Nabongsuran → scraped
Mount Maria → scraped
Mount Ynantagung → scraped
Mount Lundag → scraped
Mount Luntan → scraped
Mount Lagula → scraped
Mount Bacan → scraped
Anggas Peak → scraped
Mount Maon → scraped
Mount Tirad → scraped
Mount Dinalag → scraped
Gorro Peak → scraped
Mount Agad-Agad → scraped
Mount Buluan → scraped
Mount Bingo → scraped
Mount Sujac → scraped
Mount Asgad 

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,detail_url,url_source,keywords_found,detail_scrape_status,detail_text
0,Mount Saamong,500.0,NaN,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,https://peakvisor.com/peak/mount-saamong.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Saamong (513 m) – Philippines | PeakViso...
1,Mount Simsiman,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/mount-simsiman.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Simsiman (525 m) – Philippines | PeakVis...
2,Mount Pigtuyuan,500.0,NaN,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,https://peakvisor.com/peak/mount-pigtuyuan.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,Mount Pigtuyuan (515 m) – Philippines | PeakVi...
3,Mount Nacday,500.0,NaN,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/mount-nacday.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Calpalngan (420 m) – Philippines | PeakV...
4,Mount Caypipili,500.0,NaN,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,https://peakvisor.com/peak/mount-caypipili.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,Mount Caypipili (518 m) – Philippines | PeakVi...


In [186]:
# Apply parser to SMALL scraped detail pages

parsed_small_peakvisor = priority_small_detail_pages.copy()

parsed_small_peakvisor["detail_text_clean"] = parsed_small_peakvisor["detail_text"].apply(clean_peakvisor_text)

parsed_small_peakvisor["pv_elev_from_page"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_elevation_from_peakvisor_text
)

parsed_small_peakvisor["pv_prom_from_page"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_prominence_from_peakvisor_text
)

parsed_small_peakvisor[["pv_lat", "pv_lon"]] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_coordinates_from_peakvisor_text
)

parsed_small_peakvisor["pv_location_text"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_location_block
)

parsed_small_peakvisor["pv_about_text"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_about_text
)

parsed_small_peakvisor["pv_has_trail_info"] = parsed_small_peakvisor["detail_text_clean"].apply(
    has_trail_info
)

parsed_small_peakvisor["pv_has_volcano_info"] = parsed_small_peakvisor["detail_text_clean"].apply(
    has_volcano_info
)

parsed_small_peakvisor["pv_trail_sentence"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_trail_sentence
)

parsed_small_peakvisor["pv_volcano_sentence"] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_volcano_sentence
)

# Unit-aware parsing
parsed_small_peakvisor[["pv_elev_from_page_v2", "pv_elev_unit_source"]] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_elevation_unit_aware
)

parsed_small_peakvisor[["pv_prom_from_page_v2", "pv_prom_unit_source"]] = parsed_small_peakvisor["detail_text_clean"].apply(
    extract_prominence_unit_aware
)

# Final PeakVisor values
parsed_small_peakvisor["pv_final_elev_v2"] = parsed_small_peakvisor["pv_elev_from_page_v2"].fillna(
    parsed_small_peakvisor["peakvisor_elev"]
)

parsed_small_peakvisor["pv_final_prom_v2"] = parsed_small_peakvisor["pv_prom_from_page_v2"].fillna(
    parsed_small_peakvisor["peakvisor_prom"]
)

parsed_small_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_has_trail_info,pv_has_volcano_info,pv_trail_sentence,pv_volcano_sentence,pv_elev_from_page_v2,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,pv_final_elev_v2,pv_final_prom_v2
0,Mount Saamong,500.0,NaN,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,513.0,m_from_title,401.0,m_from_about,513.0,401.0
1,Mount Simsiman,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,525.0,m_from_title,83.0,m_from_about,525.0,83.0
2,Mount Pigtuyuan,500.0,NaN,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,515.0,m_from_title,217.0,m_from_about,515.0,217.0
3,Mount Nacday,500.0,NaN,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,420.0,m_from_title,38.0,m_from_about,420.0,38.0
4,Mount Caypipili,500.0,NaN,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,518.0,m_from_title,81.0,m_from_about,518.0,81.0


In [187]:
# Validation flags for SMALL mountains

parsed_small_peakvisor["pv_elev_difference_v2"] = (
    parsed_small_peakvisor["pv_final_elev_v2"] - parsed_small_peakvisor["elev"]
)

parsed_small_peakvisor["pv_elev_validation_flag_v2"] = parsed_small_peakvisor.apply(
    elevation_validation_flag_v2,
    axis=1
)

parsed_small_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_trail_sentence,pv_volcano_sentence,pv_elev_from_page_v2,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,pv_final_elev_v2,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2
0,Mount Saamong,500.0,NaN,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,513.0,m_from_title,401.0,m_from_about,513.0,401.0,13.0,close_match
1,Mount Simsiman,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,525.0,m_from_title,83.0,m_from_about,525.0,83.0,25.0,close_match
2,Mount Pigtuyuan,500.0,NaN,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,515.0,m_from_title,217.0,m_from_about,515.0,217.0,15.0,close_match
3,Mount Nacday,500.0,NaN,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,420.0,m_from_title,38.0,m_from_about,420.0,38.0,-80.0,minor_difference
4,Mount Caypipili,500.0,NaN,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,518.0,m_from_title,81.0,m_from_about,518.0,81.0,18.0,close_match


In [188]:
print("Rows:", len(parsed_small_peakvisor))

print("\nElevation parsed from page:")
print(parsed_small_peakvisor["pv_elev_from_page"].notna().value_counts())

print("\nProminence parsed from page:")
print(parsed_small_peakvisor["pv_prom_from_page"].notna().value_counts())

print("\nAbout text parsed:")
print(parsed_small_peakvisor["pv_about_text"].notna().value_counts())

print("\nLocation parsed:")
print(parsed_small_peakvisor["pv_location_text"].notna().value_counts())

Rows: 98

Elevation parsed from page:
pv_elev_from_page
True    98
Name: count, dtype: int64

Prominence parsed from page:
pv_prom_from_page
True     96
False     2
Name: count, dtype: int64

About text parsed:
pv_about_text
True    98
Name: count, dtype: int64

Location parsed:
pv_location_text
True    98
Name: count, dtype: int64


In [189]:
# View small rows where unit-aware parsing still needs review

parse_small_review_v2 = parsed_small_peakvisor[
    (parsed_small_peakvisor["pv_final_elev_v2"].isna()) |
    (parsed_small_peakvisor["pv_final_prom_v2"].isna()) |
    (parsed_small_peakvisor["pv_about_text"].isna()) |
    (parsed_small_peakvisor["pv_elev_validation_flag_v2"] == "needs_review")
].copy()

parse_small_review_v2[[
    "name",
    "detail_url",
    "elev",
    "pv_elev_from_page_v2",
    "pv_elev_unit_source",
    "peakvisor_elev",
    "pv_final_elev_v2",
    "pv_prom_from_page_v2",
    "pv_prom_unit_source",
    "peakvisor_prom",
    "pv_final_prom_v2",
    "pv_elev_difference_v2",
    "pv_elev_validation_flag_v2",
    "pv_about_text",
    "detail_text"
]].head(10)

,name,detail_url,elev,pv_elev_from_page_v2,pv_elev_unit_source,peakvisor_elev,pv_final_elev_v2,pv_prom_from_page_v2,pv_prom_unit_source,peakvisor_prom,pv_final_prom_v2,pv_elev_difference_v2,pv_elev_validation_flag_v2,pv_about_text,detail_text
9,Mount Fitzhugh Lee,https://peakvisor.com/peak/mount-fitzhugh-lee....,498.0,714.0,m_from_title,NaN,714.0,3.0,m_from_about,NaN,3.0,216.0,needs_review,Mount Fitzhugh Lee (714m/2 343ft a.s.l.) is a ...,Mount Fitzhugh Lee (714 m) – Philippines | Pea...
15,Knob Peak,https://peakvisor.com/peak/knob-peak.html,490.0,100.0,m_from_about,NaN,100.0,51.0,m_from_about,NaN,51.0,-390.0,needs_review,Knob Peak (100m/328ft a.s.l.) is a hill in Aus...,Knob Peak (100 m) – Australia | PeakVisor We u...
22,Mount Mabilog,https://peakvisor.com/peak/mount-mabilog.html,484.0,311.0,m_from_title,NaN,311.0,134.0,m_from_text,NaN,134.0,-173.0,needs_review,"The Laguna Volcanic Field, also known as the S...",Mount Mabilog (311 m) – Philippines | PeakViso...
24,Mount Maria,https://peakvisor.com/peak/mount-maria.html,484.0,NaN,not_found,NaN,NaN,NaN,not_found,NaN,NaN,NaN,no_peakvisor_elevation,Mount Maria is a mountain of the Hornby Mounta...,Mount Maria (658 m) – Falkland Islands | PeakV...
32,Mount Tirad,https://peakvisor.com/peak/mount-tirad.html,480.0,1412.0,m_from_title,NaN,1412.0,374.0,m_from_text,NaN,374.0,932.0,needs_review,"Mount Tirad is a 1,154-metre (3,786 ft) mounta...","Mount Tirad (1,412 m) – Philippines | PeakViso..."
46,Mount Butong,https://peakvisor.com/peak/mount-butong.html,470.0,771.0,m_from_title,NaN,771.0,128.0,m_from_about,NaN,128.0,301.0,needs_review,Mount Butong (771m/2 530ft a.s.l.) is a mounta...,Mount Butong (771 m) – Philippines | PeakVisor...
93,Mount Difun,https://peakvisor.com/peak/mount-difun.html,430.0,836.0,m_from_title,NaN,836.0,120.0,m_from_about,NaN,120.0,406.0,needs_review,Mount Difun (836m/2 743ft a.s.l.) is a mountai...,Mount Difun (836 m) – Philippines | PeakVisor ...


In [190]:
# No match at all after all methods from the SMALL priority mountains

priority_small_not_found = priority_small_urls_final[
    priority_small_urls_final["final_match_status"] == "not_found_after_all_methods"
].copy()

priority_small_not_found.to_csv(
    "priority_small_mountains_not_found_in_peakvisor.csv",
    index=False
)

print("Small not found after all methods:", len(priority_small_not_found))

priority_small_not_found[[
    "name", "elev", "prov", "region", "isl_grp"
]].head(50)

Small not found after all methods: 2


,name,elev,prov,region,isl_grp
58,Mount Tugbok ti Apánluk,460.0,Rizal,Region IV-A,Luzon
69,Mount Megatong,450.0,Davao del Norte,Region XI,Mindanao


In [191]:
print("Small rows needing review:", len(parse_small_review_v2))

print("\nSmall elevation unit source:")
print(parsed_small_peakvisor["pv_elev_unit_source"].value_counts(dropna=False))

print("\nSmall prominence unit source:")
print(parsed_small_peakvisor["pv_prom_unit_source"].value_counts(dropna=False))

print("\nSmall validation flags:")
print(parsed_small_peakvisor["pv_elev_validation_flag_v2"].value_counts(dropna=False))

Small rows needing review: 7

Small elevation unit source:
pv_elev_unit_source
m_from_title    96
m_from_about     1
not_found        1
Name: count, dtype: int64

Small prominence unit source:
pv_prom_unit_source
m_from_about         94
m_from_text           2
not_found             1
ft_converted_to_m     1
Name: count, dtype: int64

Small validation flags:
pv_elev_validation_flag_v2
close_match               80
minor_difference          11
needs_review               6
no_peakvisor_elevation     1
Name: count, dtype: int64


In [192]:
# Rename SMALL PeakVisor parsed columns

parsed_small_peakvisor = parsed_small_peakvisor.rename(columns={
    "pv_final_elev_v2": "peakvisor_elev_m",
    "pv_final_prom_v2": "peakvisor_prom_m",
    "pv_elev_unit_source": "peakvisor_elev_source_unit",
    "pv_prom_unit_source": "peakvisor_prom_source_unit",
    "pv_elev_difference_v2": "peakvisor_elev_difference_m",
    "pv_elev_validation_flag_v2": "peakvisor_elev_validation_flag",
    "pv_lat": "peakvisor_lat",
    "pv_lon": "peakvisor_lon",
    "pv_location_text": "peakvisor_location_text",
    "pv_about_text": "peakvisor_about_text",
    "pv_has_trail_info": "has_trail_info",
    "pv_has_volcano_info": "has_volcano_info",
    "pv_trail_sentence": "trail_sentence",
    "pv_volcano_sentence": "volcano_sentence"
})

parsed_small_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,trail_sentence,volcano_sentence,pv_elev_from_page_v2,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,peakvisor_elev_difference_m,peakvisor_elev_validation_flag
0,Mount Saamong,500.0,NaN,NaN,Northern Samar,Region VIII,Visayas,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,513.0,m_from_title,401.0,m_from_about,513.0,401.0,13.0,close_match
1,Mount Simsiman,500.0,NaN,NaN,Davao del Norte,Region XI,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,525.0,m_from_title,83.0,m_from_about,525.0,83.0,25.0,close_match
2,Mount Pigtuyuan,500.0,NaN,NaN,Bukidnon,Region X,Mindanao,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,515.0,m_from_title,217.0,m_from_about,515.0,217.0,15.0,close_match
3,Mount Nacday,500.0,NaN,NaN,Zambales,Region III,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,420.0,m_from_title,38.0,m_from_about,420.0,38.0,-80.0,minor_difference
4,Mount Caypipili,500.0,NaN,NaN,Rizal,Region IV-A,Luzon,NaN,NaN,NaN,...,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,518.0,m_from_title,81.0,m_from_about,518.0,81.0,18.0,close_match


In [193]:
#Safe Checkpoint
parsed_small_peakvisor.to_csv(
    "priority_mountains_peakvisor_parsed_small_peakvisor.csv",
    index=False
)

print("Saved final PeakVisor parsed enrichment file.")

Saved final PeakVisor parsed enrichment file.


# Combine enriched priority groups: large + medium + small



In [194]:
all_priority_peakvisor_enriched = pd.concat(
    [
        parsed_peakvisor.assign(priority_group="large"),
        parsed_medium_peakvisor.assign(priority_group="medium"),
        parsed_small_peakvisor.assign(priority_group="small")
    ],
    ignore_index=True
)

print("Combined enriched priority rows:", len(all_priority_peakvisor_enriched))
print(all_priority_peakvisor_enriched["priority_group"].value_counts())

all_priority_peakvisor_enriched.head()

Combined enriched priority rows: 300
priority_group
medium    103
large      99
small      98
Name: count, dtype: int64


,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,peakvisor_elev_difference_m,peakvisor_elev_validation_flag,enrichment_source,peakvisor_enrichment_status,priority_group
0,Mother Peak,2954.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,24.0,m_from_about,2954.0,24.0,0.0,close_match,PeakVisor,strong_enrichment,large
1,Kidapawan Peak,2950.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,30.0,m_from_about,2950.0,30.0,0.0,close_match,PeakVisor,strong_enrichment,large
2,Digos Peak,2945.0,NaN,NaN,Davao del Sur,Region XI,Mindanao,NaN,NaN,NaN,...,m_from_title,34.0,m_from_about,2945.0,34.0,0.0,close_match,PeakVisor,strong_enrichment,large
3,Mount Dulang-Dulang,2938.0,NaN,NaN,Bukidnon,Region X,Mindanao,Mount Dulang-Dulang,2938.0,2444.0,...,m_from_title,2934.0,m_from_text,2938.0,2934.0,0.0,close_match,PeakVisor,strong_enrichment,large
4,Mount Pulag,2928.0,NaN,NaN,Benguet,CAR,Luzon,Mount Pulag,2928.0,2928.0,...,m_from_title,2928.0,m_from_text,2928.0,2928.0,0.0,close_match,PeakVisor,strong_enrichment,large


In [195]:
# Merge combined PeakVisor enrichment back to main dataframe

main_df = quickmap_validation.copy()
enrichment_df = all_priority_peakvisor_enriched.copy()

# Deduplicate enrichment rows before merging
enrichment_df = enrichment_df.sort_values(
    by=["priority_group", "peakvisor_elev_validation_flag"],
    na_position="last"
).drop_duplicates(
    subset=["name", "prov", "region", "isl_grp"],
    keep="first"
)

print("Main rows:", len(main_df))
print("Unique enrichment rows:", len(enrichment_df))

# Keep only useful PeakVisor enrichment columns
enrichment_keep_cols = [
    "name",
    "prov",
    "region",
    "isl_grp",
    "priority_group",
    "peakvisor_name",
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "peakvisor_elev_source_unit",
    "peakvisor_prom_source_unit",
    "peakvisor_elev_difference_m",
    "peakvisor_elev_validation_flag",
    "peakvisor_lat",
    "peakvisor_lon",
    "peakvisor_location_text",
    "peakvisor_about_text",
    "has_trail_info",
    "has_volcano_info",
    "trail_sentence",
    "volcano_sentence",
    "detail_url",
    "url_source",
    "detail_scrape_status"
]

enrichment_keep_cols = [
    col for col in enrichment_keep_cols
    if col in enrichment_df.columns
]

enrichment_df = enrichment_df[enrichment_keep_cols].copy()

# Merge back to main dataset
main_with_peakvisor = main_df.merge(
    enrichment_df,
    on=["name", "prov", "region", "isl_grp"],
    how="left",
    suffixes=("", "_peakvisor")
)

print("Rows after merge:", len(main_with_peakvisor))
print("Rows with PeakVisor enrichment:", main_with_peakvisor["peakvisor_elev_m"].notna().sum())

main_with_peakvisor.head()


Main rows: 2625
Unique enrichment rows: 294
Rows after merge: 2625
Rows with PeakVisor enrichment: 300


,mountain_id,name,name_original,source,name_clean,alt_names,elev,prom,marker_size,coord,...,peakvisor_lon,peakvisor_location_text,peakvisor_about_text,has_trail_info,has_volcano_info,trail_sentence,volcano_sentence,detail_url_peakvisor,url_source,detail_scrape_status
0,387_15.9003_120.9738,Mount 387,Mount 387,OpenStreetMap,387,[],777.0,None,medium,"15.900298, 120.9737706",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,OpenStreetMap,abao,[],2527.0,None,large,"16.8323069, 120.9148864",...,120.914709,Philippines Ifugao,Mount Abao is a mountain in the Philippines. I...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,https://peakvisor.com/peak/mount-abao.html,slug_guess,scraped_with_keywords
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,OpenStreetMap,abel's peak,[],1464.0,None,medium,"12.4311298, 122.5737552",...,122.573005,Philippines Romblon Cajidiocan,Abel's Peak (1 529m/5 016ft a.s.l.) is a mount...,True,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,https://peakvisor.com/peak/abel-s-peak.html,slug_guess,scraped_with_keywords
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,OpenStreetMap,abnataclayong,[],370.0,None,small,"5.675, 125.366944",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,OpenStreetMap,aboabo,[],NaN,None,None,"9.1475284, 118.060024",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Peakvisor Enrichment Batch 1

In [196]:
# Create Batch 1 target list: first 500 mountains still missing elevation

missing_elev_targets = main_with_peakvisor[
    main_with_peakvisor["elev"].isna()
    & main_with_peakvisor["name"].notna()
    & main_with_peakvisor["lat"].notna()
    & main_with_peakvisor["lon"].notna()
].copy()

print("Total missing-elevation targets:", len(missing_elev_targets))

missing_elev_batch_1 = (
    missing_elev_targets
    .sort_values(["name_clean", "prov", "region"])
    .head(500)
    .copy()
)

print("Batch 1 rows:", len(missing_elev_batch_1))

missing_elev_batch_1[
    [
        "name",
        "name_clean",
        "elev",
        "lat",
        "lon",
        "prov",
        "region",
        "isl_grp",
        "quickmap_confidence",
        "quickmap_distance_km"
    ]
].head(50)

Total missing-elevation targets: 1071
Batch 1 rows: 500


,name,name_clean,elev,lat,lon,prov,region,isl_grp,quickmap_confidence,quickmap_distance_km
4,Mount Aboabo,aboabo,NaN,9.147528,118.060024,Palawan,MIMAROPA,Luzon,No QuickMap match,NaN
6,Mount Abongabong,abongabong,NaN,13.395455,120.750376,Occidental Mindoro,MIMAROPA,Luzon,No QuickMap match,NaN
18,Mount Adiayan,adiayan,NaN,11.778136,121.949837,Antique,Region VI,Visayas,No QuickMap match,NaN
23,Mount Agao,agao,NaN,6.016667,121.016667,Sulu,Region IX,Mindanao,No QuickMap match,NaN
25,Mount Agapang,agapang,NaN,17.516667,120.716667,Abra,CAR,Luzon,Medium confidence,23.897772
29,Mount Agdulasan,agdulasan,NaN,11.099890,122.366662,Iloilo,Region VI,Visayas,No QuickMap match,NaN
30,Agkalasag Peak,agkalasag peak,NaN,11.063757,122.423837,Iloilo,Region VI,Visayas,No QuickMap match,NaN
31,Mount Agnato,agnato,NaN,11.340278,122.574722,Capiz,Region VI,Visayas,No QuickMap match,NaN
32,Mount Agnuwan,agnuwan,NaN,18.116667,121.183333,Apayao,CAR,Luzon,No QuickMap match,NaN
33,Mount Agpalali,agpalali,NaN,11.368056,122.416667,Capiz,Region VI,Visayas,No QuickMap match,NaN


In [197]:
#Setting up helpers
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PhilippinesMountainsPortfolioProject/1.0)"
}

PEAKVISOR_PH_URL = "https://peakvisor.com/adm/philippines.html"

KEYWORDS = [
    "elevation", "prominence", "mountain", "summit",
    "volcano", "volcanic", "stratovolcano",
    "hiking", "trail", "trek", "climb", "route",
    "range", "mountain range"
]

def clean_text(text):
    return (
        str(text)
        .replace("\u2060", " ")
        .replace("\xa0", " ")
        .replace("\n", " ")
        .replace("\t", " ")
        .strip()
    )

def numbers_only(text):
    return int(re.sub(r"[^\d]", "", str(text)))

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = name.replace("mount ", "").replace("mt. ", "").replace("mt ", "")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

def find_keywords(text):
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS if kw in text_lower]

In [198]:
#Generate possible links for those unmatched
def make_slug_candidates(name):
    raw = str(name).lower().strip()
    raw = re.sub(r"[^a-z0-9\s\-]", " ", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    no_mount = (
        raw.replace("mount ", "")
           .replace("mt. ", "")
           .replace("mt ", "")
           .strip()
    )

    candidates = [
        raw.replace(" ", "-"),
        no_mount.replace(" ", "-"),
        "mount-" + no_mount.replace(" ", "-"),
        "mt-" + no_mount.replace(" ", "-")
    ]

    return list(dict.fromkeys([c for c in candidates if c]))

def possible_peakvisor_urls(name):
    urls = []

    for slug in make_slug_candidates(name):
        urls.extend([
            f"https://peakvisor.com/peak/{slug}.html",
            f"https://peakvisor.com/mountain/{slug}.html"
        ])

    return urls

# Quick test
possible_peakvisor_urls("Mount Apo")

['https://peakvisor.com/peak/mount-apo.html',
 'https://peakvisor.com/mountain/mount-apo.html',
 'https://peakvisor.com/peak/apo.html',
 'https://peakvisor.com/mountain/apo.html',
 'https://peakvisor.com/peak/mt-apo.html',
 'https://peakvisor.com/mountain/mt-apo.html']

In [199]:
#Test unmatchedd mountains for guessed urls
def page_matches_mountain(text, mountain_name):
    text_lower = str(text).lower()

    name_clean = normalize_name(mountain_name)
    name_words = name_clean.split()

    found_keywords = find_keywords(text)

    # Good if page contains the mountain name
    name_in_page = name_clean in normalize_name(text)

    # Backup check: important name words appear
    important_words_found = sum(1 for word in name_words if len(word) > 2 and word in text_lower)

    is_useful = name_in_page or important_words_found >= 1 or len(found_keywords) >= 2

    return is_useful, found_keywords

def find_url_by_slug_guess(name):
    for test_url in possible_peakvisor_urls(name):
        try:
            r = requests.get(test_url, headers=headers, timeout=20)

            if r.status_code != 200:
                continue

            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)

            is_useful, found_keywords = page_matches_mountain(detail_text, name)

            if is_useful:
                return {
                    "guessed_detail_url": test_url,
                    "url_guess_status": "found_by_slug_guess",
                    "keywords_found": ", ".join(found_keywords),
                    "detail_text": detail_text
                }

            time.sleep(1)

        except Exception:
            continue

    return {
        "guessed_detail_url": None,
        "url_guess_status": "not_found_after_url_guess",
        "keywords_found": None,
        "detail_text": None
    }

In [200]:
# Apply URL guessing to missing-elevation batch 1 mountains

unmatched_missing_elev_batch_1 = missing_elev_batch_1.copy()

guess_missing_elev_batch_1_rows = []

print(
    "Missing-elevation batch 1 mountains to URL-guess:",
    len(unmatched_missing_elev_batch_1)
)

# Cache results so repeated names do not get requested again
slug_guess_cache = {}

for i, (_, row) in enumerate(unmatched_missing_elev_batch_1.iterrows(), start=1):
    name = row["name"]

    if name in slug_guess_cache:
        result = slug_guess_cache[name]
    else:
        result = find_url_by_slug_guess(name)
        slug_guess_cache[name] = result
        time.sleep(0.3)

    guess_missing_elev_batch_1_rows.append({
        "name": row["name"],
        "name_clean": row["name_clean"],
        "elev": row["elev"],
        "prom": row["prom"],
        "prov": row["prov"],
        "region": row["region"],
        "isl_grp": row["isl_grp"],
        "lat": row["lat"],
        "lon": row["lon"],
        "guessed_detail_url": result["guessed_detail_url"],
        "url_guess_status": result["url_guess_status"],
        "guess_keywords_found": result["keywords_found"],
        "guessed_detail_text": result["detail_text"]
    })

    if i % 25 == 0 or i == len(unmatched_missing_elev_batch_1):
        print(f"Processed {i}/{len(unmatched_missing_elev_batch_1)}")

url_guess_missing_elev_batch_1_df = pd.DataFrame(guess_missing_elev_batch_1_rows)

print("Done.")
print(url_guess_missing_elev_batch_1_df["url_guess_status"].value_counts(dropna=False))

url_guess_missing_elev_batch_1_df.head()

Missing-elevation batch 1 mountains to URL-guess: 500
Processed 25/500
Processed 50/500
Processed 75/500
Processed 100/500
Processed 125/500
Processed 150/500
Processed 175/500
Processed 200/500
Processed 225/500
Processed 250/500
Processed 275/500
Processed 300/500
Processed 325/500
Processed 350/500
Processed 375/500
Processed 400/500
Processed 425/500
Processed 450/500
Processed 475/500
Processed 500/500
Done.
url_guess_status
found_by_slug_guess          489
not_found_after_url_guess     11
Name: count, dtype: int64


,name,name_clean,elev,prom,prov,region,isl_grp,lat,lon,guessed_detail_url,url_guess_status,guess_keywords_found,guessed_detail_text
0,Mount Aboabo,aboabo,NaN,None,Palawan,MIMAROPA,Luzon,9.147528,118.060024,https://peakvisor.com/peak/mount-aboabo.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Aboabo (395 m) – Philippines | PeakVisor...
1,Mount Abongabong,abongabong,NaN,None,Occidental Mindoro,MIMAROPA,Luzon,13.395455,120.750376,https://peakvisor.com/peak/mount-abongabong.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Abongabong (792 m) – Philippines | PeakV...
2,Mount Adiayan,adiayan,NaN,None,Antique,Region VI,Visayas,11.778136,121.949837,https://peakvisor.com/peak/mount-adiayan.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking",Mount Adiayan (328 m) – Philippines | PeakViso...
3,Mount Agao,agao,NaN,None,Sulu,Region IX,Mindanao,6.016667,121.016667,https://peakvisor.com/peak/mount-agao.html,found_by_slug_guess,"elevation, prominence, mountain, hiking",Mount Agao (261 m) – Philippines | PeakVisor W...
4,Mount Agapang,agapang,NaN,None,Abra,CAR,Luzon,17.516667,120.716667,https://peakvisor.com/peak/mount-agapang.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hiking","Mount Agapang (1,504 m) – Philippines | PeakVi..."


In [201]:
# Add row IDs after URL guessing, based on current row order

missing_elev_batch_1 = missing_elev_batch_1.reset_index(drop=True).copy()
url_guess_missing_elev_batch_1_df = url_guess_missing_elev_batch_1_df.reset_index(drop=True).copy()

missing_elev_batch_1["batch_1_row_id"] = missing_elev_batch_1.index
url_guess_missing_elev_batch_1_df["batch_1_row_id"] = url_guess_missing_elev_batch_1_df.index

print("missing_elev_batch_1 rows:", len(missing_elev_batch_1))
print("url_guess_missing_elev_batch_1_df rows:", len(url_guess_missing_elev_batch_1_df))

missing_elev_batch_1 rows: 500
url_guess_missing_elev_batch_1_df rows: 500


In [202]:
# Merge safely by row ID, not name

missing_elev_batch_1_urls_final = missing_elev_batch_1.merge(
    url_guess_missing_elev_batch_1_df[
        [
            "batch_1_row_id",
            "guessed_detail_url",
            "url_guess_status",
            "guess_keywords_found",
            "guessed_detail_text"
        ]
    ],
    on="batch_1_row_id",
    how="left"
)

missing_elev_batch_1_urls_final["final_detail_url"] = missing_elev_batch_1_urls_final["guessed_detail_url"]

missing_elev_batch_1_urls_final["final_url_source"] = missing_elev_batch_1_urls_final["guessed_detail_url"].apply(
    lambda x: "slug_guess" if pd.notna(x) else "not_found"
)

missing_elev_batch_1_urls_final["final_match_status"] = missing_elev_batch_1_urls_final["final_detail_url"].apply(
    lambda x: "has_detail_url" if pd.notna(x) else "not_found_after_all_methods"
)

print("Expected batch 1 rows:", len(missing_elev_batch_1))
print("Final URL rows:", len(missing_elev_batch_1_urls_final))

print(missing_elev_batch_1_urls_final["final_url_source"].value_counts())
print(missing_elev_batch_1_urls_final["final_match_status"].value_counts())

Expected batch 1 rows: 500
Final URL rows: 500
final_url_source
slug_guess    489
not_found      11
Name: count, dtype: int64
final_match_status
has_detail_url                 489
not_found_after_all_methods     11
Name: count, dtype: int64


In [203]:
print("missing_elev_batch_1 rows:", len(missing_elev_batch_1))
print("url_guess_missing_elev_batch_1_df rows:", len(url_guess_missing_elev_batch_1_df))
print("merged rows:", len(missing_elev_batch_1_urls_final))

print("Duplicate names in missing_elev_batch_1:")
print(missing_elev_batch_1["name"].duplicated().sum())

print("Duplicate names in url_guess_missing_elev_batch_1_df:")
print(url_guess_missing_elev_batch_1_df["name"].duplicated().sum())

missing_elev_batch_1 rows: 500
url_guess_missing_elev_batch_1_df rows: 500
merged rows: 500
Duplicate names in missing_elev_batch_1:
13
Duplicate names in url_guess_missing_elev_batch_1_df:
13


In [204]:
# Clean duplicate columns after merge

if "name_clean_x" in missing_elev_batch_1_urls_final.columns:
    missing_elev_batch_1_urls_final["name_clean"] = missing_elev_batch_1_urls_final["name_clean_x"]

rename_back = {
    "elev_x": "elev",
    "prom_x": "prom",
    "prov_x": "prov",
    "region_x": "region",
    "isl_grp_x": "isl_grp",
    "lat_x": "lat",
    "lon_x": "lon",
    "source_x": "source"
}

missing_elev_batch_1_urls_final = missing_elev_batch_1_urls_final.rename(
    columns={k: v for k, v in rename_back.items() if k in missing_elev_batch_1_urls_final.columns}
)

cols_to_drop = [
    "name_clean_x", "name_clean_y",
    "elev_y", "prom_y", "prov_y", "region_y", "isl_grp_y",
    "lat_y", "lon_y", "source_y"
]

missing_elev_batch_1_urls_final = missing_elev_batch_1_urls_final.drop(
    columns=[col for col in cols_to_drop if col in missing_elev_batch_1_urls_final.columns],
    errors="ignore"
)

print("Cleaned columns.")
print("Rows:", len(missing_elev_batch_1_urls_final))
print(missing_elev_batch_1_urls_final["final_match_status"].value_counts(dropna=False))

Cleaned columns.
Rows: 500
final_match_status
has_detail_url                 489
not_found_after_all_methods     11
Name: count, dtype: int64


In [205]:
# Scrape final detail URLs one by one for missing-elevation batch 1 mountains
# raw scraped page text

detail_missing_elev_batch_1_rows = []

to_scrape_missing_elev_batch_1 = missing_elev_batch_1_urls_final[
    missing_elev_batch_1_urls_final["final_detail_url"].notna()
].copy()

print("Final missing-elevation batch 1 detail pages available:", len(to_scrape_missing_elev_batch_1))

for _, row in to_scrape_missing_elev_batch_1.iterrows():
    try:
        # If detail text already came from slug guessing, reuse it
        if row["final_url_source"] == "slug_guess" and pd.notna(row.get("guessed_detail_text")):
            detail_text = row["guessed_detail_text"]
        else:
            r = requests.get(row["final_detail_url"], headers=headers, timeout=30)
            r.raise_for_status()
            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)
            time.sleep(0.5)

        found_keywords = find_keywords(detail_text)

        detail_missing_elev_batch_1_rows.append({
            "name": row["name"],
            "elev": row["elev"],
            "prom": row["prom"],
            "is_volc": row["is_volc"],
            "prov": row["prov"],
            "region": row["region"],
            "isl_grp": row["isl_grp"],
            "peakvisor_name": row.get("peakvisor_name"),
            "peakvisor_elev": row.get("peakvisor_elev"),
            "peakvisor_prom": row.get("peakvisor_prom"),
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "keywords_found": ", ".join(found_keywords),
            "detail_scrape_status": "scraped_with_keywords" if found_keywords else "scraped_no_keywords",
            "detail_text": detail_text
        })

        print(row["name"], "→ scraped")

    except Exception as e:
        detail_missing_elev_batch_1_rows.append({
            "name": row["name"],
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "detail_scrape_status": "failed",
            "error": str(e)
        })

        print("Failed:", row["name"], e)

missing_elev_batch_1_detail_pages = pd.DataFrame(detail_missing_elev_batch_1_rows)

missing_elev_batch_1_detail_pages.head()

Final missing-elevation batch 1 detail pages available: 489
Mount Aboabo → scraped
Mount Abongabong → scraped
Mount Adiayan → scraped
Mount Agao → scraped
Mount Agapang → scraped
Mount Agdulasan → scraped
Agkalasag Peak → scraped
Mount Agnato → scraped
Mount Agnuwan → scraped
Mount Agpalali → scraped
Mount Agsing → scraped
Mount Agta → scraped
Agtalin Hill → scraped
Mount AguaFlora → scraped
Mount AguaViva → scraped
Mount Agudo → scraped
Mount Agudo → scraped
Mount Agumid → scraped
Mount Agustin → scraped
Mount Aharung → scraped
Aipes Hill → scraped
Mount Aja → scraped
Mount Aki → scraped
Alakak Hill → scraped
Mount Alapad → scraped
Mount Alapasco → scraped
Mount Alava → scraped
Mount Alayao → scraped
Mount Alegan → scraped
Mount Alibug → scraped
Mount Alihod → scraped
Mount Alimungao → scraped
Almirante Gil Hill → scraped
Mount Amaloi → scraped
Mount Amanapao → scraped
Mount Amantara → scraped
Mount Ambongan → scraped
Mount Ambubungan → scraped
Mount Ambulong → scraped
Mount Amicay → 

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,detail_url,url_source,keywords_found,detail_scrape_status,detail_text
0,Mount Aboabo,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,https://peakvisor.com/peak/mount-aboabo.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Aboabo (395 m) – Philippines | PeakVisor...
1,Mount Abongabong,NaN,None,None,Occidental Mindoro,MIMAROPA,Luzon,NaN,None,None,https://peakvisor.com/peak/mount-abongabong.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Abongabong (792 m) – Philippines | PeakV...
2,Mount Adiayan,NaN,None,None,Antique,Region VI,Visayas,NaN,None,None,https://peakvisor.com/peak/mount-adiayan.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,Mount Adiayan (328 m) – Philippines | PeakViso...
3,Mount Agao,NaN,None,None,Sulu,Region IX,Mindanao,NaN,None,None,https://peakvisor.com/peak/mount-agao.html,slug_guess,"elevation, prominence, mountain, hiking",scraped_with_keywords,Mount Agao (261 m) – Philippines | PeakVisor W...
4,Mount Agapang,NaN,None,None,Abra,CAR,Luzon,NaN,None,None,https://peakvisor.com/peak/mount-agapang.html,slug_guess,"elevation, prominence, mountain, summit, hiking",scraped_with_keywords,"Mount Agapang (1,504 m) – Philippines | PeakVi..."


In [206]:
# Applying Main parse for missing-elevation batch 1 PeakVisor detail pages

parsed_missing_elev_batch_1_peakvisor = missing_elev_batch_1_detail_pages.copy()

parsed_missing_elev_batch_1_peakvisor["detail_text_clean"] = parsed_missing_elev_batch_1_peakvisor["detail_text"].apply(
    clean_peakvisor_text
)

parsed_missing_elev_batch_1_peakvisor["pv_elev_from_page"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_elevation_from_peakvisor_text
)

parsed_missing_elev_batch_1_peakvisor["pv_prom_from_page"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_prominence_from_peakvisor_text
)

parsed_missing_elev_batch_1_peakvisor[["pv_lat", "pv_lon"]] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_coordinates_from_peakvisor_text
)

parsed_missing_elev_batch_1_peakvisor["pv_location_text"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_location_block
)

parsed_missing_elev_batch_1_peakvisor["pv_about_text"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_about_text
)

parsed_missing_elev_batch_1_peakvisor["pv_has_trail_info"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    has_trail_info
)

parsed_missing_elev_batch_1_peakvisor["pv_has_volcano_info"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    has_volcano_info
)

parsed_missing_elev_batch_1_peakvisor["pv_trail_sentence"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_trail_sentence
)

parsed_missing_elev_batch_1_peakvisor["pv_volcano_sentence"] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_volcano_sentence
)

# Unit-aware elevation/prominence parsing
parsed_missing_elev_batch_1_peakvisor[["pv_elev_from_page_v2", "pv_elev_unit_source"]] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_elevation_unit_aware
)

parsed_missing_elev_batch_1_peakvisor[["pv_prom_from_page_v2", "pv_prom_unit_source"]] = parsed_missing_elev_batch_1_peakvisor["detail_text_clean"].apply(
    extract_prominence_unit_aware
)

# Final PeakVisor values
parsed_missing_elev_batch_1_peakvisor["pv_final_elev_v2"] = parsed_missing_elev_batch_1_peakvisor["pv_elev_from_page_v2"].fillna(
    parsed_missing_elev_batch_1_peakvisor["peakvisor_elev"]
)

parsed_missing_elev_batch_1_peakvisor["pv_final_prom_v2"] = parsed_missing_elev_batch_1_peakvisor["pv_prom_from_page_v2"].fillna(
    parsed_missing_elev_batch_1_peakvisor["peakvisor_prom"]
)

# For missing-elevation batch 1, this is more useful than comparing difference
def missing_elev_fill_flag(row):
    if pd.notna(row["elev"]):
        return "already_had_original_elevation"
    elif pd.notna(row["pv_final_elev_v2"]):
        return "peakvisor_elevation_found"
    else:
        return "still_missing_elevation"

parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"] = parsed_missing_elev_batch_1_peakvisor.apply(
    missing_elev_fill_flag,
    axis=1
)

print(parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"].value_counts(dropna=False))

parsed_missing_elev_batch_1_peakvisor.head()

missing_elev_fill_flag
peakvisor_elevation_found    477
still_missing_elevation       12
Name: count, dtype: int64


/tmp/ipykernel_1326/2841701610.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  parsed_missing_elev_batch_1_peakvisor["pv_final_elev_v2"] = parsed_missing_elev_batch_1_peakvisor["pv_elev_from_page_v2"].fillna(


,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_has_volcano_info,pv_trail_sentence,pv_volcano_sentence,pv_elev_from_page_v2,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,pv_final_elev_v2,pv_final_prom_v2,missing_elev_fill_flag
0,Mount Aboabo,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,395.0,m_from_title,155,m_from_about,395.0,155,peakvisor_elevation_found
1,Mount Abongabong,NaN,None,None,Occidental Mindoro,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,792.0,m_from_title,15,m_from_about,792.0,15,peakvisor_elevation_found
2,Mount Adiayan,NaN,None,None,Antique,Region VI,Visayas,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,328.0,m_from_title,45,m_from_about,328.0,45,peakvisor_elevation_found
3,Mount Agao,NaN,None,None,Sulu,Region IX,Mindanao,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,261.0,m_from_title,0,m_from_about,261.0,0,peakvisor_elevation_found
4,Mount Agapang,NaN,None,None,Abra,CAR,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1504.0,m_from_title,7,m_from_about,1504.0,7,peakvisor_elevation_found


In [207]:
parsed_missing_elev_batch_1_peakvisor[[
    "name",
    "elev",
    "pv_final_elev_v2",
    "pv_elev_unit_source",
    "pv_final_prom_v2",
    "pv_prom_unit_source",
    "missing_elev_fill_flag",
    "detail_url",
    "url_source"
]].head(50)

,name,elev,pv_final_elev_v2,pv_elev_unit_source,pv_final_prom_v2,pv_prom_unit_source,missing_elev_fill_flag,detail_url,url_source
0,Mount Aboabo,NaN,395.0,m_from_title,155,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-aboabo.html,slug_guess
1,Mount Abongabong,NaN,792.0,m_from_title,15,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-abongabong.html,slug_guess
2,Mount Adiayan,NaN,328.0,m_from_title,45,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-adiayan.html,slug_guess
3,Mount Agao,NaN,261.0,m_from_title,0,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agao.html,slug_guess
4,Mount Agapang,NaN,1504.0,m_from_title,7,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agapang.html,slug_guess
5,Mount Agdulasan,NaN,819.0,m_from_title,315,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agdulasan.html,slug_guess
6,Agkalasag Peak,NaN,356.0,m_from_title,125,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/agkalasag-peak.html,slug_guess
7,Mount Agnato,NaN,384.0,m_from_title,316,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agnato.html,slug_guess
8,Mount Agnuwan,NaN,1077.0,m_from_title,277,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agnuwan.html,slug_guess
9,Mount Agpalali,NaN,417.0,m_from_title,63,m_from_about,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agpalali.html,slug_guess


In [208]:
parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"].value_counts(dropna=False)

,count
missing_elev_fill_flag,
peakvisor_elevation_found,477
still_missing_elevation,12


In [209]:
# Rename missing-elevation batch 1 PeakVisor parsed columns

parsed_missing_elev_batch_1_peakvisor = parsed_missing_elev_batch_1_peakvisor.rename(columns={
    "pv_final_elev_v2": "peakvisor_elev_m",
    "pv_final_prom_v2": "peakvisor_prom_m",
    "pv_elev_unit_source": "peakvisor_elev_source_unit",
    "pv_prom_unit_source": "peakvisor_prom_source_unit",
    "pv_elev_difference_v2": "peakvisor_elev_difference_m",
    "pv_elev_validation_flag_v2": "peakvisor_elev_validation_flag",
    "pv_lat": "peakvisor_lat",
    "pv_lon": "peakvisor_lon",
    "pv_location_text": "peakvisor_location_text",
    "pv_about_text": "peakvisor_about_text",
    "pv_has_trail_info": "has_trail_info",
    "pv_has_volcano_info": "has_volcano_info",
    "pv_trail_sentence": "trail_sentence",
    "pv_volcano_sentence": "volcano_sentence"
})

parsed_missing_elev_batch_1_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,has_volcano_info,trail_sentence,volcano_sentence,pv_elev_from_page_v2,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,missing_elev_fill_flag
0,Mount Aboabo,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,395.0,m_from_title,155,m_from_about,395.0,155,peakvisor_elevation_found
1,Mount Abongabong,NaN,None,None,Occidental Mindoro,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,792.0,m_from_title,15,m_from_about,792.0,15,peakvisor_elevation_found
2,Mount Adiayan,NaN,None,None,Antique,Region VI,Visayas,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,328.0,m_from_title,45,m_from_about,328.0,45,peakvisor_elevation_found
3,Mount Agao,NaN,None,None,Sulu,Region IX,Mindanao,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,261.0,m_from_title,0,m_from_about,261.0,0,peakvisor_elevation_found
4,Mount Agapang,NaN,None,None,Abra,CAR,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,1504.0,m_from_title,7,m_from_about,1504.0,7,peakvisor_elevation_found


In [210]:
parsed_missing_elev_batch_1_peakvisor[[
    "name",
    "elev",
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "missing_elev_fill_flag",
    "detail_url",
    "url_source"
]].head(50)

,name,elev,peakvisor_elev_m,peakvisor_prom_m,missing_elev_fill_flag,detail_url,url_source
0,Mount Aboabo,NaN,395.0,155,peakvisor_elevation_found,https://peakvisor.com/peak/mount-aboabo.html,slug_guess
1,Mount Abongabong,NaN,792.0,15,peakvisor_elevation_found,https://peakvisor.com/peak/mount-abongabong.html,slug_guess
2,Mount Adiayan,NaN,328.0,45,peakvisor_elevation_found,https://peakvisor.com/peak/mount-adiayan.html,slug_guess
3,Mount Agao,NaN,261.0,0,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agao.html,slug_guess
4,Mount Agapang,NaN,1504.0,7,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agapang.html,slug_guess
5,Mount Agdulasan,NaN,819.0,315,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agdulasan.html,slug_guess
6,Agkalasag Peak,NaN,356.0,125,peakvisor_elevation_found,https://peakvisor.com/peak/agkalasag-peak.html,slug_guess
7,Mount Agnato,NaN,384.0,316,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agnato.html,slug_guess
8,Mount Agnuwan,NaN,1077.0,277,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agnuwan.html,slug_guess
9,Mount Agpalali,NaN,417.0,63,peakvisor_elevation_found,https://peakvisor.com/peak/mount-agpalali.html,slug_guess


In [211]:
# Safe Checkpoint

parsed_missing_elev_batch_1_peakvisor.to_csv(
    "missing_elev_batch_1_peakvisor_parsed.csv",
    index=False
)

print("Saved final PeakVisor parsed missing-elevation batch 1 enrichment file.")

Saved final PeakVisor parsed missing-elevation batch 1 enrichment file.


In [212]:
# Check Batch 1 enrichment result

print(parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"].value_counts(dropna=False))

batch_1_found = (
    parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"]
    .eq("peakvisor_elevation_found")
    .sum()
)

batch_1_still_missing = (
    parsed_missing_elev_batch_1_peakvisor["missing_elev_fill_flag"]
    .eq("still_missing_elevation")
    .sum()
)

print("Batch 1 elevations found:", batch_1_found)
print("Batch 1 still missing:", batch_1_still_missing)

missing_elev_fill_flag
peakvisor_elevation_found    477
still_missing_elevation       12
Name: count, dtype: int64
Batch 1 elevations found: 477
Batch 1 still missing: 12


In [213]:
# Check columns in parsed batch 1
print(parsed_missing_elev_batch_1_peakvisor.columns.tolist())

['name', 'elev', 'prom', 'is_volc', 'prov', 'region', 'isl_grp', 'peakvisor_name', 'peakvisor_elev', 'peakvisor_prom', 'detail_url', 'url_source', 'keywords_found', 'detail_scrape_status', 'detail_text', 'detail_text_clean', 'pv_elev_from_page', 'pv_prom_from_page', 'peakvisor_lat', 'peakvisor_lon', 'peakvisor_location_text', 'peakvisor_about_text', 'has_trail_info', 'has_volcano_info', 'trail_sentence', 'volcano_sentence', 'pv_elev_from_page_v2', 'peakvisor_elev_source_unit', 'pv_prom_from_page_v2', 'peakvisor_prom_source_unit', 'peakvisor_elev_m', 'peakvisor_prom_m', 'missing_elev_fill_flag']


In [214]:
# Simulate applying Batch 1 PeakVisor elevations to main_with_peakvisor using name

main_after_missing_batch_1 = main_with_peakvisor.copy()

batch_1_elev_map = (
    parsed_missing_elev_batch_1_peakvisor
    .dropna(subset=["peakvisor_elev_m"])
    .drop_duplicates(subset=["name"])
    .set_index("name")["peakvisor_elev_m"]
)

main_after_missing_batch_1["elev_after_batch_1"] = main_after_missing_batch_1["elev"]

main_after_missing_batch_1.loc[
    main_after_missing_batch_1["elev_after_batch_1"].isna(),
    "elev_after_batch_1"
] = main_after_missing_batch_1.loc[
    main_after_missing_batch_1["elev_after_batch_1"].isna(),
    "name"
].map(batch_1_elev_map)

print("Original missing elevation count:", main_with_peakvisor["elev"].isna().sum())
print("Missing elevation count after Batch 1:", main_after_missing_batch_1["elev_after_batch_1"].isna().sum())
print(
    "Elevations filled by Batch 1:",
    main_with_peakvisor["elev"].isna().sum()
    - main_after_missing_batch_1["elev_after_batch_1"].isna().sum()
)

Original missing elevation count: 1071
Missing elevation count after Batch 1: 594
Elevations filled by Batch 1: 477


# Peakvisor Enrichment Batch 2

In [215]:
# Create remaining target list: ALL mountains still missing elevation after Batch 1

missing_elev_remaining_targets = main_after_missing_batch_1[
    main_after_missing_batch_1["elev_after_batch_1"].isna()
    & main_after_missing_batch_1["name"].notna()
    & main_after_missing_batch_1["lat"].notna()
    & main_after_missing_batch_1["lon"].notna()
].copy()

print("Total remaining missing-elevation targets:", len(missing_elev_remaining_targets))

missing_elev_remaining = (
    missing_elev_remaining_targets
    .sort_values(["name_clean", "prov", "region"])
    .copy()
)

# Make compatible with your previous workflow
missing_elev_remaining["elev"] = missing_elev_remaining["elev_after_batch_1"]

print("Remaining rows:", len(missing_elev_remaining))

missing_elev_remaining[
    [
        "name",
        "name_clean",
        "elev",
        "elev_after_batch_1",
        "lat",
        "lon",
        "prov",
        "region",
        "isl_grp",
        "quickmap_confidence",
        "quickmap_distance_km"
    ]
].head(50)

Total remaining missing-elevation targets: 594
Remaining rows: 594


,name,name_clean,elev,elev_after_batch_1,lat,lon,prov,region,isl_grp,quickmap_confidence,quickmap_distance_km
61,Mount Alava,alava,NaN,NaN,16.150008,120.500007,Pangasinan,Region I,Luzon,No QuickMap match,NaN
223,Balitbitan Hill,balitbitan hill,NaN,NaN,12.063652,120.187943,Palawan,MIMAROPA,Luzon,No QuickMap match,NaN
245,Banat-i Hill,banat-i hill,NaN,NaN,9.633214,123.882890,Bohol,Region VII,Visayas,No QuickMap match,NaN
337,Big Peak,big peak,NaN,NaN,10.583333,119.516667,Palawan,MIMAROPA,Luzon,Medium confidence,11.645386
383,Mount Boars Tusk,boars tusk,NaN,NaN,14.933333,120.333333,Zambales,Region III,Luzon,No QuickMap match,NaN
432,Bukagan Hill,bukagan hill,NaN,NaN,8.137117,123.820328,Misamis Occidental,Region X,Mindanao,No QuickMap match,NaN
532,Calbasaan Peak,calbasaan peak,NaN,NaN,10.304211,123.766802,Cebu,Region VII,Visayas,No QuickMap match,NaN
535,Mount Calo,calo,NaN,NaN,13.684942,121.208353,Batangas,Region IV-A,Luzon,No QuickMap match,NaN
551,Cambuang Hill,cambuang hill,NaN,NaN,10.025195,123.413637,Cebu,Region VII,Visayas,No QuickMap match,NaN
587,Cape Engaño Hill,cape engaño hill,NaN,NaN,18.579890,122.137484,Cagayan,Region II,Luzon,No QuickMap match,NaN


In [216]:
#Setting up helpers
import requests, re, time
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; PhilippinesMountainsPortfolioProject/1.0)"
}

PEAKVISOR_PH_URL = "https://peakvisor.com/adm/philippines.html"

KEYWORDS = [
    "elevation", "prominence", "mountain", "summit",
    "volcano", "volcanic", "stratovolcano",
    "hiking", "trail", "trek", "climb", "route",
    "range", "mountain range"
]

def clean_text(text):
    return (
        str(text)
        .replace("\u2060", " ")
        .replace("\xa0", " ")
        .replace("\n", " ")
        .replace("\t", " ")
        .strip()
    )

def numbers_only(text):
    return int(re.sub(r"[^\d]", "", str(text)))

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name).lower()
    name = name.replace("mount ", "").replace("mt. ", "").replace("mt ", "")
    name = re.sub(r"[^a-z0-9\s]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()

    return name

def find_keywords(text):
    text_lower = str(text).lower()
    return [kw for kw in KEYWORDS if kw in text_lower]

In [217]:
#Generate possible links for those unmatched
def make_slug_candidates(name):
    raw = str(name).lower().strip()
    raw = re.sub(r"[^a-z0-9\s\-]", " ", raw)
    raw = re.sub(r"\s+", " ", raw).strip()

    no_mount = (
        raw.replace("mount ", "")
           .replace("mt. ", "")
           .replace("mt ", "")
           .strip()
    )

    candidates = [
        raw.replace(" ", "-"),
        no_mount.replace(" ", "-"),
        "mount-" + no_mount.replace(" ", "-"),
        "mt-" + no_mount.replace(" ", "-")
    ]

    return list(dict.fromkeys([c for c in candidates if c]))

def possible_peakvisor_urls(name):
    urls = []

    for slug in make_slug_candidates(name):
        urls.extend([
            f"https://peakvisor.com/peak/{slug}.html",
            f"https://peakvisor.com/mountain/{slug}.html"
        ])

    return urls

# Quick test
possible_peakvisor_urls("Mount Apo")

['https://peakvisor.com/peak/mount-apo.html',
 'https://peakvisor.com/mountain/mount-apo.html',
 'https://peakvisor.com/peak/apo.html',
 'https://peakvisor.com/mountain/apo.html',
 'https://peakvisor.com/peak/mt-apo.html',
 'https://peakvisor.com/mountain/mt-apo.html']

In [218]:
#Test unmatchedd mountains for guessed urls
def page_matches_mountain(text, mountain_name):
    text_lower = str(text).lower()

    name_clean = normalize_name(mountain_name)
    name_words = name_clean.split()

    found_keywords = find_keywords(text)

    # Good if page contains the mountain name
    name_in_page = name_clean in normalize_name(text)

    # Backup check: important name words appear
    important_words_found = sum(1 for word in name_words if len(word) > 2 and word in text_lower)

    is_useful = name_in_page or important_words_found >= 1 or len(found_keywords) >= 2

    return is_useful, found_keywords

def find_url_by_slug_guess(name):
    for test_url in possible_peakvisor_urls(name):
        try:
            r = requests.get(test_url, headers=headers, timeout=20)

            if r.status_code != 200:
                continue

            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)

            is_useful, found_keywords = page_matches_mountain(detail_text, name)

            if is_useful:
                return {
                    "guessed_detail_url": test_url,
                    "url_guess_status": "found_by_slug_guess",
                    "keywords_found": ", ".join(found_keywords),
                    "detail_text": detail_text
                }

            time.sleep(1)

        except Exception:
            continue

    return {
        "guessed_detail_url": None,
        "url_guess_status": "not_found_after_url_guess",
        "keywords_found": None,
        "detail_text": None
    }

In [219]:
# Apply URL guessing to remaining missing-elevation mountains

unmatched_missing_elev_remaining = missing_elev_remaining.copy()

guess_missing_elev_remaining_rows = []

print(
    "Remaining missing-elevation mountains to URL-guess:",
    len(unmatched_missing_elev_remaining)
)

# Cache results so repeated names do not get requested again
slug_guess_cache = {}

for i, (_, row) in enumerate(unmatched_missing_elev_remaining.iterrows(), start=1):
    name = row["name"]

    if name in slug_guess_cache:
        result = slug_guess_cache[name]
    else:
        result = find_url_by_slug_guess(name)
        slug_guess_cache[name] = result
        time.sleep(0.3)

    guess_missing_elev_remaining_rows.append({
        "name": row["name"],
        "name_clean": row["name_clean"],
        "elev": row["elev"],
        "prom": row["prom"],
        "prov": row["prov"],
        "region": row["region"],
        "isl_grp": row["isl_grp"],
        "lat": row["lat"],
        "lon": row["lon"],
        "guessed_detail_url": result["guessed_detail_url"],
        "url_guess_status": result["url_guess_status"],
        "guess_keywords_found": result["keywords_found"],
        "guessed_detail_text": result["detail_text"]
    })

    if i % 25 == 0 or i == len(unmatched_missing_elev_remaining):
        print(f"Processed {i}/{len(unmatched_missing_elev_remaining)}")

url_guess_missing_elev_remaining_df = pd.DataFrame(guess_missing_elev_remaining_rows)

print("Done.")
print(url_guess_missing_elev_remaining_df["url_guess_status"].value_counts(dropna=False))

url_guess_missing_elev_remaining_df.head()

Remaining missing-elevation mountains to URL-guess: 594
Processed 25/594
Processed 50/594
Processed 75/594
Processed 100/594
Processed 125/594
Processed 150/594
Processed 175/594
Processed 200/594
Processed 225/594
Processed 250/594
Processed 275/594
Processed 300/594
Processed 325/594
Processed 350/594
Processed 375/594
Processed 400/594
Processed 425/594
Processed 450/594
Processed 475/594
Processed 500/594
Processed 525/594
Processed 550/594
Processed 575/594
Processed 594/594
Done.
url_guess_status
found_by_slug_guess          574
not_found_after_url_guess     20
Name: count, dtype: int64


,name,name_clean,elev,prom,prov,region,isl_grp,lat,lon,guessed_detail_url,url_guess_status,guess_keywords_found,guessed_detail_text
0,Mount Alava,alava,NaN,None,Pangasinan,Region I,Luzon,16.150008,120.500007,https://peakvisor.com/peak/mount-alava.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...","Mount Alava (1,621 m) – Vancouver Island Range..."
1,Balitbitan Hill,balitbitan hill,NaN,None,Palawan,MIMAROPA,Luzon,12.063652,120.187943,None,not_found_after_url_guess,None,None
2,Banat-i Hill,banat-i hill,NaN,None,Bohol,Region VII,Visayas,9.633214,123.882890,None,not_found_after_url_guess,None,None
3,Big Peak,big peak,NaN,None,Palawan,MIMAROPA,Luzon,10.583333,119.516667,https://peakvisor.com/peak/big-peak.html,found_by_slug_guess,"elevation, prominence, mountain, summit, hikin...","Big Peak (3,062 m) – Smoky Mountains, USA | Pe..."
4,Mount Boars Tusk,boars tusk,NaN,None,Zambales,Region III,Luzon,14.933333,120.333333,https://peakvisor.com/peak/boars-tusk.html,found_by_slug_guess,"elevation, prominence, mountain, summit, volca...","Boars Tusk (2,132 m) – Rocky Mountains, USA | ..."


In [220]:
# Add row IDs after URL guessing, based on current row order

missing_elev_remaining = missing_elev_remaining.reset_index(drop=True).copy()
url_guess_missing_elev_remaining_df = url_guess_missing_elev_remaining_df.reset_index(drop=True).copy()

missing_elev_remaining["remaining_row_id"] = missing_elev_remaining.index
url_guess_missing_elev_remaining_df["remaining_row_id"] = url_guess_missing_elev_remaining_df.index

print("missing_elev_remaining rows:", len(missing_elev_remaining))
print("url_guess_missing_elev_remaining_df rows:", len(url_guess_missing_elev_remaining_df))

missing_elev_remaining rows: 594
url_guess_missing_elev_remaining_df rows: 594


In [221]:
# Merge safely by row ID, not name

missing_elev_remaining_urls_final = missing_elev_remaining.merge(
    url_guess_missing_elev_remaining_df[
        [
            "remaining_row_id",
            "guessed_detail_url",
            "url_guess_status",
            "guess_keywords_found",
            "guessed_detail_text"
        ]
    ],
    on="remaining_row_id",
    how="left"
)

missing_elev_remaining_urls_final["final_detail_url"] = missing_elev_remaining_urls_final["guessed_detail_url"]

missing_elev_remaining_urls_final["final_url_source"] = missing_elev_remaining_urls_final["guessed_detail_url"].apply(
    lambda x: "slug_guess" if pd.notna(x) else "not_found"
)

missing_elev_remaining_urls_final["final_match_status"] = missing_elev_remaining_urls_final["final_detail_url"].apply(
    lambda x: "has_detail_url" if pd.notna(x) else "not_found_after_all_methods"
)

print("Expected remaining rows:", len(missing_elev_remaining))
print("Final URL rows:", len(missing_elev_remaining_urls_final))

print(missing_elev_remaining_urls_final["final_url_source"].value_counts())
print(missing_elev_remaining_urls_final["final_match_status"].value_counts())

Expected remaining rows: 594
Final URL rows: 594
final_url_source
slug_guess    574
not_found      20
Name: count, dtype: int64
final_match_status
has_detail_url                 574
not_found_after_all_methods     20
Name: count, dtype: int64


In [222]:
# Check available columns first
print(missing_elev_batch_1_urls_final.columns.tolist())

['mountain_id', 'name', 'name_original', 'source', 'name_clean', 'alt_names', 'elev', 'prom', 'marker_size', 'coord', 'lat', 'lon', 'is_volc', 'isl_grp', 'elev_raw', 'osm_id', 'osm_type', 'osm_tags', 'data_quality_notes', 'geometry', 'index_right', 'prov', 'prov_psgc_code', 'region', 'region_name', 'region_code', 'prov_gis_original', 'region_gis_original', 'source_file', 'source', 'coord_source', 'admin_note', 'detail_url', 'name_quickmap', 'lat_quickmap', 'lon_quickmap', 'elev_quickmap', 'quickmap_match', 'quickmap_distance_km', 'quickmap_validation_note', 'quickmap_status', 'quickmap_confidence', 'priority_group', 'peakvisor_name', 'peakvisor_elev_m', 'peakvisor_prom_m', 'peakvisor_elev_source_unit', 'peakvisor_prom_source_unit', 'peakvisor_elev_difference_m', 'peakvisor_elev_validation_flag', 'peakvisor_lat', 'peakvisor_lon', 'peakvisor_location_text', 'peakvisor_about_text', 'has_trail_info', 'has_volcano_info', 'trail_sentence', 'volcano_sentence', 'detail_url_peakvisor', 'url_sou

In [223]:
print("missing_elev_remaining rows:", len(missing_elev_remaining))
print("url_guess_missing_elev_remaining_df rows:", len(url_guess_missing_elev_remaining_df))
print("merged rows:", len(missing_elev_remaining_urls_final))

print("Duplicate names in missing_elev_remaining:")
print(missing_elev_remaining["name"].duplicated().sum())

print("Duplicate names in url_guess_missing_elev_remaining_df:")
print(url_guess_missing_elev_remaining_df["name"].duplicated().sum())

missing_elev_remaining rows: 594
url_guess_missing_elev_remaining_df rows: 594
merged rows: 594
Duplicate names in missing_elev_remaining:
89
Duplicate names in url_guess_missing_elev_remaining_df:
89


In [224]:
# Clean duplicate columns after merge

if "name_clean_x" in missing_elev_remaining_urls_final.columns:
    missing_elev_remaining_urls_final["name_clean"] = missing_elev_remaining_urls_final["name_clean_x"]

rename_back = {
    "elev_x": "elev",
    "prom_x": "prom",
    "prov_x": "prov",
    "region_x": "region",
    "isl_grp_x": "isl_grp",
    "lat_x": "lat",
    "lon_x": "lon",
    "source_x": "source"
}

missing_elev_remaining_urls_final = missing_elev_remaining_urls_final.rename(
    columns={k: v for k, v in rename_back.items() if k in missing_elev_remaining_urls_final.columns}
)

cols_to_drop = [
    "name_clean_x", "name_clean_y",
    "elev_y", "prom_y", "prov_y", "region_y", "isl_grp_y",
    "lat_y", "lon_y", "source_y"
]

missing_elev_remaining_urls_final = missing_elev_remaining_urls_final.drop(
    columns=[col for col in cols_to_drop if col in missing_elev_remaining_urls_final.columns],
    errors="ignore"
)

print("Cleaned columns.")
print("Rows:", len(missing_elev_remaining_urls_final))
print(missing_elev_remaining_urls_final["final_match_status"].value_counts(dropna=False))

Cleaned columns.
Rows: 594
final_match_status
has_detail_url                 574
not_found_after_all_methods     20
Name: count, dtype: int64


In [225]:
# Scrape final detail URLs one by one for remaining missing-elevation mountains
# raw scraped page text

detail_missing_elev_remaining_rows = []

to_scrape_missing_elev_remaining = missing_elev_remaining_urls_final[
    missing_elev_remaining_urls_final["final_detail_url"].notna()
].copy()

print("Final remaining missing-elevation detail pages available:", len(to_scrape_missing_elev_remaining))

for _, row in to_scrape_missing_elev_remaining.iterrows():
    try:
        # If detail text already came from slug guessing, reuse it
        if row["final_url_source"] == "slug_guess" and pd.notna(row.get("guessed_detail_text")):
            detail_text = row["guessed_detail_text"]
        else:
            r = requests.get(row["final_detail_url"], headers=headers, timeout=30)
            r.raise_for_status()
            detail_soup = BeautifulSoup(r.text, "html.parser")
            detail_text = detail_soup.get_text(" ", strip=True)
            time.sleep(0.5)

        found_keywords = find_keywords(detail_text)

        detail_missing_elev_remaining_rows.append({
            "name": row["name"],
            "elev": row["elev"],
            "prom": row["prom"],
            "is_volc": row["is_volc"],
            "prov": row["prov"],
            "region": row["region"],
            "isl_grp": row["isl_grp"],
            "peakvisor_name": row.get("peakvisor_name"),
            "peakvisor_elev": row.get("peakvisor_elev"),
            "peakvisor_prom": row.get("peakvisor_prom"),
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "keywords_found": ", ".join(found_keywords),
            "detail_scrape_status": "scraped_with_keywords" if found_keywords else "scraped_no_keywords",
            "detail_text": detail_text
        })

        print(row["name"], "→ scraped")

    except Exception as e:
        detail_missing_elev_remaining_rows.append({
            "name": row["name"],
            "detail_url": row["final_detail_url"],
            "url_source": row["final_url_source"],
            "detail_scrape_status": "failed",
            "error": str(e)
        })

        print("Failed:", row["name"], e)

missing_elev_remaining_detail_pages = pd.DataFrame(detail_missing_elev_remaining_rows)

missing_elev_remaining_detail_pages.head()

Final remaining missing-elevation detail pages available: 574
Mount Alava → scraped
Big Peak → scraped
Mount Boars Tusk → scraped
Crawford Hill → scraped
Mount Davidson → scraped
Dome Peak → scraped
Dome Peak → scraped
Dome Peak → scraped
Double Peak → scraped
Green Hill → scraped
High Peak → scraped
Mount Knob Mountain → scraped
Lipa Hill → scraped
Mount Lipata → scraped
Mount Lirut → scraped
Mount Llorente → scraped
Mount Lobo → scraped
Mount Loggerhead Peaks → scraped
Mount Lomboy → scraped
Mount Lomintu → scraped
Lone Tree Hill → scraped
Lone Tree Peak → scraped
Long Point Hill → scraped
Mount Longolung → scraped
Mount Lonhi → scraped
Mount Lubiranan → scraped
Lucbuan Hill → scraped
Mount Luho → scraped
Mount Lumandong → scraped
Lumbian Hill → scraped
Mount Lusong → scraped
Mount Lusong → scraped
Luyo Hill → scraped
Mount Ma-angit → scraped
Mount Mabittayon → scraped
Mount Mabukok → scraped
Mount Mabukok → scraped
Mount Mac Gregor → scraped
Mount Macanacanabbang → scraped
Mount Mac

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,detail_url,url_source,keywords_found,detail_scrape_status,detail_text
0,Mount Alava,NaN,None,None,Pangasinan,Region I,Luzon,NaN,None,None,https://peakvisor.com/peak/mount-alava.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Mount Alava (1,621 m) – Vancouver Island Range..."
1,Big Peak,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,https://peakvisor.com/peak/big-peak.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Big Peak (3,062 m) – Smoky Mountains, USA | Pe..."
2,Mount Boars Tusk,NaN,None,None,Zambales,Region III,Luzon,NaN,None,None,https://peakvisor.com/peak/boars-tusk.html,slug_guess,"elevation, prominence, mountain, summit, volca...",scraped_with_keywords,"Boars Tusk (2,132 m) – Rocky Mountains, USA | ..."
3,Crawford Hill,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,https://peakvisor.com/peak/crawford-hill.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,"Crawford Hill (120 m) – Appalachian Mountains,..."
4,Mount Davidson,NaN,None,None,Oriental Mindoro,MIMAROPA,Luzon,NaN,None,None,https://peakvisor.com/peak/mount-davidson.html,slug_guess,"elevation, prominence, mountain, summit, hikin...",scraped_with_keywords,Mount Davidson (286 m) – USA | PeakVisor We us...


In [226]:
# Applying Main parse for remaining missing-elevation PeakVisor detail pages

parsed_missing_elev_remaining_peakvisor = missing_elev_remaining_detail_pages.copy()

parsed_missing_elev_remaining_peakvisor["detail_text_clean"] = parsed_missing_elev_remaining_peakvisor["detail_text"].apply(
    clean_peakvisor_text
)

parsed_missing_elev_remaining_peakvisor["pv_elev_from_page"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_elevation_from_peakvisor_text
)

parsed_missing_elev_remaining_peakvisor["pv_prom_from_page"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_prominence_from_peakvisor_text
)

parsed_missing_elev_remaining_peakvisor[["pv_lat", "pv_lon"]] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_coordinates_from_peakvisor_text
)

parsed_missing_elev_remaining_peakvisor["pv_location_text"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_location_block
)

parsed_missing_elev_remaining_peakvisor["pv_about_text"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_about_text
)

parsed_missing_elev_remaining_peakvisor["pv_has_trail_info"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    has_trail_info
)

parsed_missing_elev_remaining_peakvisor["pv_has_volcano_info"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    has_volcano_info
)

parsed_missing_elev_remaining_peakvisor["pv_trail_sentence"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_trail_sentence
)

parsed_missing_elev_remaining_peakvisor["pv_volcano_sentence"] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_volcano_sentence
)

# Unit-aware elevation/prominence parsing
parsed_missing_elev_remaining_peakvisor[["pv_elev_from_page_v2", "pv_elev_unit_source"]] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_elevation_unit_aware
)

parsed_missing_elev_remaining_peakvisor[["pv_prom_from_page_v2", "pv_prom_unit_source"]] = parsed_missing_elev_remaining_peakvisor["detail_text_clean"].apply(
    extract_prominence_unit_aware
)

# Final PeakVisor values
parsed_missing_elev_remaining_peakvisor["pv_final_elev_v2"] = parsed_missing_elev_remaining_peakvisor["pv_elev_from_page_v2"].fillna(
    parsed_missing_elev_remaining_peakvisor["peakvisor_elev"]
)

parsed_missing_elev_remaining_peakvisor["pv_final_prom_v2"] = parsed_missing_elev_remaining_peakvisor["pv_prom_from_page_v2"].fillna(
    parsed_missing_elev_remaining_peakvisor["peakvisor_prom"]
)

# For missing-elevation remaining, this is more useful than comparing difference
def missing_elev_fill_flag(row):
    if pd.notna(row["elev"]):
        return "already_had_original_elevation"
    elif pd.notna(row["pv_final_elev_v2"]):
        return "peakvisor_elevation_found"
    else:
        return "still_missing_elevation"

parsed_missing_elev_remaining_peakvisor["missing_elev_fill_flag"] = parsed_missing_elev_remaining_peakvisor.apply(
    missing_elev_fill_flag,
    axis=1
)

print(parsed_missing_elev_remaining_peakvisor["missing_elev_fill_flag"].value_counts(dropna=False))

parsed_missing_elev_remaining_peakvisor.head()

missing_elev_fill_flag
peakvisor_elevation_found    481
still_missing_elevation       93
Name: count, dtype: int64


/tmp/ipykernel_1326/2786298213.py:55: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  parsed_missing_elev_remaining_peakvisor["pv_final_elev_v2"] = parsed_missing_elev_remaining_peakvisor["pv_elev_from_page_v2"].fillna(


,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,pv_has_volcano_info,pv_trail_sentence,pv_volcano_sentence,pv_elev_from_page_v2,pv_elev_unit_source,pv_prom_from_page_v2,pv_prom_unit_source,pv_final_elev_v2,pv_final_prom_v2,missing_elev_fill_flag
0,Mount Alava,NaN,None,None,Pangasinan,Region I,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,1267,m_from_text,NaN,1267,still_missing_elevation
1,Big Peak,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,1020,m_from_text,NaN,1020,still_missing_elevation
2,Mount Boars Tusk,NaN,None,None,Zambales,Region III,Luzon,NaN,None,None,...,True,Accept Reject PeakVisor Panorama 3D Hiking Map...,Proportional Prominence 149 m Location USA Wyo...,NaN,not_found,149,m_from_text,NaN,149,still_missing_elevation
3,Crawford Hill,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,104,m_from_text,NaN,104,still_missing_elevation
4,Mount Davidson,NaN,None,None,Oriental Mindoro,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,219,m_from_text,NaN,219,still_missing_elevation


In [227]:
parsed_missing_elev_remaining_peakvisor[[
    "name",
    "elev",
    "pv_final_elev_v2",
    "pv_elev_unit_source",
    "pv_final_prom_v2",
    "pv_prom_unit_source",
    "missing_elev_fill_flag",
    "detail_url",
    "url_source"
]].head(50)

,name,elev,pv_final_elev_v2,pv_elev_unit_source,pv_final_prom_v2,pv_prom_unit_source,missing_elev_fill_flag,detail_url,url_source
0,Mount Alava,NaN,NaN,not_found,1267,m_from_text,still_missing_elevation,https://peakvisor.com/peak/mount-alava.html,slug_guess
1,Big Peak,NaN,NaN,not_found,1020,m_from_text,still_missing_elevation,https://peakvisor.com/peak/big-peak.html,slug_guess
2,Mount Boars Tusk,NaN,NaN,not_found,149,m_from_text,still_missing_elevation,https://peakvisor.com/peak/boars-tusk.html,slug_guess
3,Crawford Hill,NaN,NaN,not_found,104,m_from_text,still_missing_elevation,https://peakvisor.com/peak/crawford-hill.html,slug_guess
4,Mount Davidson,NaN,NaN,not_found,219,m_from_text,still_missing_elevation,https://peakvisor.com/peak/mount-davidson.html,slug_guess
5,Dome Peak,NaN,NaN,not_found,1865,m_from_text,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
6,Dome Peak,NaN,NaN,not_found,1865,m_from_text,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
7,Dome Peak,NaN,NaN,not_found,1865,m_from_text,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
8,Double Peak,NaN,NaN,not_found,267,m_from_text,still_missing_elevation,https://peakvisor.com/peak/double-peak.html,slug_guess
9,Green Hill,NaN,NaN,not_found,179,m_from_text,still_missing_elevation,https://peakvisor.com/peak/green-hill.html,slug_guess


In [228]:
parsed_missing_elev_remaining_peakvisor["missing_elev_fill_flag"].value_counts(dropna=False)

,count
missing_elev_fill_flag,
peakvisor_elevation_found,481
still_missing_elevation,93


In [229]:
# Rename remaining missing-elevation PeakVisor parsed columns

parsed_missing_elev_remaining_peakvisor = parsed_missing_elev_remaining_peakvisor.rename(columns={
    "pv_final_elev_v2": "peakvisor_elev_m",
    "pv_final_prom_v2": "peakvisor_prom_m",
    "pv_elev_unit_source": "peakvisor_elev_source_unit",
    "pv_prom_unit_source": "peakvisor_prom_source_unit",
    "pv_elev_difference_v2": "peakvisor_elev_difference_m",
    "pv_elev_validation_flag_v2": "peakvisor_elev_validation_flag",
    "pv_lat": "peakvisor_lat",
    "pv_lon": "peakvisor_lon",
    "pv_location_text": "peakvisor_location_text",
    "pv_about_text": "peakvisor_about_text",
    "pv_has_trail_info": "has_trail_info",
    "pv_has_volcano_info": "has_volcano_info",
    "pv_trail_sentence": "trail_sentence",
    "pv_volcano_sentence": "volcano_sentence"
})

parsed_missing_elev_remaining_peakvisor.head()

,name,elev,prom,is_volc,prov,region,isl_grp,peakvisor_name,peakvisor_elev,peakvisor_prom,...,has_volcano_info,trail_sentence,volcano_sentence,pv_elev_from_page_v2,peakvisor_elev_source_unit,pv_prom_from_page_v2,peakvisor_prom_source_unit,peakvisor_elev_m,peakvisor_prom_m,missing_elev_fill_flag
0,Mount Alava,NaN,None,None,Pangasinan,Region I,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,1267,m_from_text,NaN,1267,still_missing_elevation
1,Big Peak,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,1020,m_from_text,NaN,1020,still_missing_elevation
2,Mount Boars Tusk,NaN,None,None,Zambales,Region III,Luzon,NaN,None,None,...,True,Accept Reject PeakVisor Panorama 3D Hiking Map...,Proportional Prominence 149 m Location USA Wyo...,NaN,not_found,149,m_from_text,NaN,149,still_missing_elevation
3,Crawford Hill,NaN,None,None,Palawan,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,104,m_from_text,NaN,104,still_missing_elevation
4,Mount Davidson,NaN,None,None,Oriental Mindoro,MIMAROPA,Luzon,NaN,None,None,...,False,Accept Reject PeakVisor Panorama 3D Hiking Map...,None,NaN,not_found,219,m_from_text,NaN,219,still_missing_elevation


In [230]:
parsed_missing_elev_remaining_peakvisor[[
    "name",
    "elev",
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "missing_elev_fill_flag",
    "detail_url",
    "url_source"
]].head(50)

,name,elev,peakvisor_elev_m,peakvisor_prom_m,missing_elev_fill_flag,detail_url,url_source
0,Mount Alava,NaN,NaN,1267,still_missing_elevation,https://peakvisor.com/peak/mount-alava.html,slug_guess
1,Big Peak,NaN,NaN,1020,still_missing_elevation,https://peakvisor.com/peak/big-peak.html,slug_guess
2,Mount Boars Tusk,NaN,NaN,149,still_missing_elevation,https://peakvisor.com/peak/boars-tusk.html,slug_guess
3,Crawford Hill,NaN,NaN,104,still_missing_elevation,https://peakvisor.com/peak/crawford-hill.html,slug_guess
4,Mount Davidson,NaN,NaN,219,still_missing_elevation,https://peakvisor.com/peak/mount-davidson.html,slug_guess
5,Dome Peak,NaN,NaN,1865,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
6,Dome Peak,NaN,NaN,1865,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
7,Dome Peak,NaN,NaN,1865,still_missing_elevation,https://peakvisor.com/peak/dome-peak.html,slug_guess
8,Double Peak,NaN,NaN,267,still_missing_elevation,https://peakvisor.com/peak/double-peak.html,slug_guess
9,Green Hill,NaN,NaN,179,still_missing_elevation,https://peakvisor.com/peak/green-hill.html,slug_guess


In [231]:
# Safe Checkpoint

parsed_missing_elev_remaining_peakvisor.to_csv(
    "missing_elev_remaining_peakvisor_parsed.csv",
    index=False
)

print("Saved final PeakVisor parsed remaining missing-elevation enrichment file.")

Saved final PeakVisor parsed remaining missing-elevation enrichment file.


# Combining Peakvisor Elev Enrichment Remaining all aside from Priority Mountains

In [232]:
# ============================================================
# Apply Batch 1 + Remaining PeakVisor elevations to main dataset
# Then check final missing elevation count and recalculate marker_size
# ============================================================

import numpy as np
import pandas as pd

main_final_elev = main_with_peakvisor.copy()

# Start with original elevation
main_final_elev["elev_final_m"] = main_final_elev["elev"]


# ------------------------------------------------------------
# Helper: safer merge key using name + province + region
# ------------------------------------------------------------

def make_peak_key(df):
    return (
        df["name"].astype(str).str.strip().str.lower()
        + " | " +
        df["prov"].astype(str).str.strip().str.lower()
        + " | " +
        df["region"].astype(str).str.strip().str.lower()
    )


main_final_elev["peak_key"] = make_peak_key(main_final_elev)


# ------------------------------------------------------------
# 1. Apply Batch 1 PeakVisor elevations
# ------------------------------------------------------------

batch_1_for_merge = parsed_missing_elev_batch_1_peakvisor.copy()
batch_1_for_merge["peak_key"] = make_peak_key(batch_1_for_merge)

batch_1_elev_map = (
    batch_1_for_merge
    .dropna(subset=["peakvisor_elev_m"])
    .drop_duplicates(subset=["peak_key"])
    .set_index("peak_key")["peakvisor_elev_m"]
)

main_final_elev.loc[
    main_final_elev["elev_final_m"].isna(),
    "elev_final_m"
] = main_final_elev.loc[
    main_final_elev["elev_final_m"].isna(),
    "peak_key"
].map(batch_1_elev_map)


# ------------------------------------------------------------
# 2. Apply Remaining PeakVisor elevations
# ------------------------------------------------------------

remaining_for_merge = parsed_missing_elev_remaining_peakvisor.copy()
remaining_for_merge["peak_key"] = make_peak_key(remaining_for_merge)

remaining_elev_map = (
    remaining_for_merge
    .dropna(subset=["peakvisor_elev_m"])
    .drop_duplicates(subset=["peak_key"])
    .set_index("peak_key")["peakvisor_elev_m"]
)

main_final_elev.loc[
    main_final_elev["elev_final_m"].isna(),
    "elev_final_m"
] = main_final_elev.loc[
    main_final_elev["elev_final_m"].isna(),
    "peak_key"
].map(remaining_elev_map)


# ------------------------------------------------------------
# 3. Check final missing elevation count
# ------------------------------------------------------------

original_missing = main_with_peakvisor["elev"].isna().sum()
final_missing = main_final_elev["elev_final_m"].isna().sum()
filled_total = original_missing - final_missing

print("Original missing elevation count:", original_missing)
print("Final missing elevation count after all PeakVisor missing-elev enrichment:", final_missing)
print("Total elevations filled:", filled_total)


# ------------------------------------------------------------
# 4. Add elevation source flag
# ------------------------------------------------------------

main_final_elev["elev_final_source"] = np.where(
    main_final_elev["elev"].notna(),
    "original_elev",
    np.where(
        main_final_elev["elev_final_m"].notna(),
        "peakvisor_missing_elev_enrichment",
        "still_missing"
    )
)

print("\nElevation source counts:")
print(main_final_elev["elev_final_source"].value_counts(dropna=False))


# ------------------------------------------------------------
# 5. Recalculate marker_size as category
# ------------------------------------------------------------

def classify_marker_size(elev):
    if pd.isna(elev):
        return "unknown"
    elif elev <= 500:
        return "small"
    elif elev < 1500:
        return "medium"
    else:
        return "large"


main_final_elev["marker_size"] = main_final_elev["elev_final_m"].apply(classify_marker_size)

print("\nFinal marker_size counts:")
print(main_final_elev["marker_size"].value_counts(dropna=False))


# ------------------------------------------------------------
# 6. Preview final elevation + marker result
# ------------------------------------------------------------

main_final_elev[[
    "name",
    "elev",
    "elev_final_m",
    "elev_final_source",
    "marker_size",
    "prov",
    "region",
    "isl_grp"
]].head(50)

Original missing elevation count: 1071
Final missing elevation count after all PeakVisor missing-elev enrichment: 113
Total elevations filled: 958

Elevation source counts:
elev_final_source
original_elev                        1554
peakvisor_missing_elev_enrichment     958
still_missing                         113
Name: count, dtype: int64

Final marker_size counts:
marker_size
small      1146
medium     1085
large       281
unknown     113
Name: count, dtype: int64


,name,elev,elev_final_m,elev_final_source,marker_size,prov,region,isl_grp
0,Mount 387,777.0,777.0,original_elev,medium,Nueva Ecija,Region III,Luzon
1,Mount Abao,2527.0,2527.0,original_elev,large,Ifugao,CAR,Luzon
2,Abel's Peak,1464.0,1464.0,original_elev,medium,Romblon,MIMAROPA,Luzon
3,Mount Abnataclayong,370.0,370.0,original_elev,small,Sarangani,Region XII,Mindanao
4,Mount Aboabo,NaN,395.0,peakvisor_missing_elev_enrichment,small,Palawan,MIMAROPA,Luzon
5,Abong-abong Peak,885.0,885.0,original_elev,medium,Basilan,BARMM,Mindanao
6,Mount Abongabong,NaN,792.0,peakvisor_missing_elev_enrichment,medium,Occidental Mindoro,MIMAROPA,Luzon
7,Mount Aborlan,745.0,745.0,original_elev,medium,Palawan,MIMAROPA,Luzon
8,Mount Abra de Ilog,1018.0,1018.0,original_elev,medium,Occidental Mindoro,MIMAROPA,Luzon
9,Mount Abriringan,659.0,659.0,original_elev,medium,Cagayan,Region II,Luzon


In [233]:
# ============================================================
# Marker size sanity check
# ============================================================

def expected_marker_size(elev):
    if pd.isna(elev):
        return "unknown"
    elif elev <= 500:
        return "small"
    elif elev < 1500:
        return "medium"
    else:
        return "large"


main_final_elev["expected_marker_size"] = main_final_elev["elev_final_m"].apply(expected_marker_size)

main_final_elev["marker_size_check"] = np.where(
    main_final_elev["marker_size"] == main_final_elev["expected_marker_size"],
    "correct",
    "needs_review"
)

print("Marker size sanity check:")
print(main_final_elev["marker_size_check"].value_counts(dropna=False))

print("\nMarker size vs expected:")
print(pd.crosstab(
    main_final_elev["marker_size"],
    main_final_elev["expected_marker_size"],
    dropna=False
))

Marker size sanity check:
marker_size_check
correct    2625
Name: count, dtype: int64

Marker size vs expected:
expected_marker_size  large  medium  small  unknown
marker_size                                        
large                   281       0      0        0
medium                    0    1085      0        0
small                     0       0   1146        0
unknown                   0       0      0      113


In [234]:
# Show rows where marker_size does not match elevation rule

marker_size_issues = main_final_elev[
    main_final_elev["marker_size_check"] == "needs_review"
].copy()

print("Rows needing marker_size review:", len(marker_size_issues))

marker_size_issues[[
    "name",
    "elev",
    "elev_final_m",
    "marker_size",
    "expected_marker_size",
    "elev_final_source",
    "prov",
    "region",
    "isl_grp"
]].head(50)

Rows needing marker_size review: 0


,name,elev,elev_final_m,marker_size,expected_marker_size,elev_final_source,prov,region,isl_grp


In [235]:
main_final_elev.to_csv(
    "philippines_mountains_final_elevation_enriched.csv",
    index=False
)

print("Saved final elevation-enriched mountains dataset.")

Saved final elevation-enriched mountains dataset.


# Enrich/validate with GeoJson Files

In [236]:
#Clone the Repo
!git clone https://github.com/j4ckofalltrades/phl-mountains

Cloning into 'phl-mountains'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (237/237), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 237 (delta 174), reused 224 (delta 167), pack-reused 0 (from 0)
Receiving objects: 100% (237/237), 233.47 KiB | 5.31 MiB/s, done.
Resolving deltas: 100% (174/174), done.


In [237]:
# Inspect repo structure

import os

for root, dirs, files in os.walk("phl-mountains"):
    level = root.replace("phl-mountains", "").count(os.sep)
    indent = "  " * level

    if level <= 3:
        print(f"{indent}{os.path.basename(root)}/")

        for file in files[:10]:
            print(f"{indent}  {file}")

phl-mountains/
  LICENSE
  .nvmrc
  package-lock.json
  CONTRIBUTING.md
  Makefile
  README.md
  .gitignore
  CODE_OF_CONDUCT.md
  package.json
  tsconfig.json
  .husky/
    .gitignore
    pre-commit
  .git/
    config
    description
    HEAD
    packed-refs
    index
    refs/
      tags/
      heads/
        main
      remotes/
    branches/
    info/
      exclude
    objects/
      info/
      pack/
        pack-48a62cd99b91a72fc35d14e885b53cbd500ab8fe.idx
        pack-48a62cd99b91a72fc35d14e885b53cbd500ab8fe.pack
    logs/
      HEAD
      refs/
    hooks/
      pre-receive.sample
      pre-merge-commit.sample
      pre-commit.sample
      fsmonitor-watchman.sample
      update.sample
      post-update.sample
      applypatch-msg.sample
      commit-msg.sample
      prepare-commit-msg.sample
      pre-applypatch.sample
  scripts/
    constants.ts
    topojson_gen.ts
    geojson_gen.ts
  data/
    geojson/
      _index.geojson
      island-group/
        mindanao.geojson
        v

In [238]:
repo_geojson_folder = "phl-mountains/data/geojson/province"

In [239]:
# Find province GeoJSON files in the repo

import glob

repo_geojson_files = glob.glob(
    "phl-mountains/data/geojson/province/*.geojson"
)

print("Province mountain GeoJSON files found:", len(repo_geojson_files))
repo_geojson_files[:10]

Province mountain GeoJSON files found: 81


['phl-mountains/data/geojson/province/leyte.geojson',
 'phl-mountains/data/geojson/province/cagayan.geojson',
 'phl-mountains/data/geojson/province/camarines_norte.geojson',
 'phl-mountains/data/geojson/province/zamboanga_del_sur.geojson',
 'phl-mountains/data/geojson/province/quirino.geojson',
 'phl-mountains/data/geojson/province/laguna.geojson',
 'phl-mountains/data/geojson/province/negros_occidental.geojson',
 'phl-mountains/data/geojson/province/davao_occidental.geojson',
 'phl-mountains/data/geojson/province/negros_oriental.geojson',
 'phl-mountains/data/geojson/province/guimaras.geojson']

In [240]:
# Read and combine all province mountain GeoJSON files

import geopandas as gpd
import pandas as pd
import os

repo_mountain_gdfs = []

for file in repo_geojson_files:
    gdf = gpd.read_file(file)
    gdf["source_file"] = os.path.basename(file)
    repo_mountain_gdfs.append(gdf)

repo_mountains_gdf = pd.concat(repo_mountain_gdfs, ignore_index=True)

print("Combined repo mountain rows:", len(repo_mountains_gdf))
repo_mountains_gdf.head()

Combined repo mountain rows: 519


,name,elev,prom,coord,is_volc,prov,region,isl_grp,alt_names,marker-color,marker-size,marker-symbol,geometry,source_file
0,Mount Aminduen,1325.0,1325.0,11°6′14″N 124°44′33″E,NaN,[Leyte],[Region VIII],Visayas,[Alto Peak],#259346,medium,mountain,POINT (124.7425 11.10382),leyte.geojson
1,Mount Busay,476.0,456.0,11°23′58″N 124°49′56″E,NaN,[Leyte],[Region VIII],Visayas,[],#259346,small,mountain,POINT (124.83229 11.39931),leyte.geojson
2,Mount Gumdalitan,865.0,396.0,10°52′53″N 124°53′15″E,NaN,[Leyte],[Region VIII],Visayas,[],#259346,medium,mountain,POINT (124.88757 10.88142),leyte.geojson
3,Mount Hingatungan,745.0,386.0,10°35′10″N 125°6′54″E,NaN,"[Leyte, Southern Leyte]",[Region VIII],Visayas,[],#259346,medium,mountain,POINT (125.11495 10.58613),leyte.geojson
4,Mount Lobi,1319.0,710.0,11°0′44″N 124°48′44″E,1.0,[Leyte],[Region VIII],Visayas,[],#259346,medium,mountain,POINT (124.81222 11.0123),leyte.geojson


In [241]:
# Inspect repo mountain columns

repo_mountains_gdf.columns.tolist()


['name',
 'elev',
 'prom',
 'coord',
 'is_volc',
 'prov',
 'region',
 'isl_grp',
 'alt_names',
 'marker-color',
 'marker-size',
 'marker-symbol',
 'geometry',
 'source_file']

In [242]:
repo_mountains_gdf = repo_mountains_gdf.rename(columns={
    "marker-size": "marker_size",
    "marker-color": "marker_color",
    "marker-symbol": "marker_symbol"
})

In [243]:
repo_mountains_gdf[[
    "name",
    "elev",
    "prom",
    "coord",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "alt_names",
    "marker_size",
    "geometry",
    "source_file"
]].head(20)

,name,elev,prom,coord,is_volc,prov,region,isl_grp,alt_names,marker_size,geometry,source_file
0,Mount Aminduen,1325.0,1325.0,11°6′14″N 124°44′33″E,NaN,[Leyte],[Region VIII],Visayas,[Alto Peak],medium,POINT (124.7425 11.10382),leyte.geojson
1,Mount Busay,476.0,456.0,11°23′58″N 124°49′56″E,NaN,[Leyte],[Region VIII],Visayas,[],small,POINT (124.83229 11.39931),leyte.geojson
2,Mount Gumdalitan,865.0,396.0,10°52′53″N 124°53′15″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.88757 10.88142),leyte.geojson
3,Mount Hingatungan,745.0,386.0,10°35′10″N 125°6′54″E,NaN,"[Leyte, Southern Leyte]",[Region VIII],Visayas,[],medium,POINT (125.11495 10.58613),leyte.geojson
4,Mount Lobi,1319.0,710.0,11°0′44″N 124°48′44″E,1.0,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.81222 11.0123),leyte.geojson
5,Mount Magsanga,572.0,500.0,10°57′15″N 124°29′36″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.49342 10.95415),leyte.geojson
6,Mount Mahagnao,860.0,318.0,10°52′49″N 124°51′53″E,1.0,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.86474 10.88029),leyte.geojson
7,Mount Naburac,864.0,386.0,10°51′54″N 124°52′36″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.87662 10.86504),leyte.geojson
8,Mount Pangasugan,1150.0,403.0,10°45′35″N 124°50′22″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.8395 10.75972),leyte.geojson
9,Mount Sacripante,924.0,524.0,10°31′56″N 124°48′17″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,POINT (124.80482 10.53225),leyte.geojson


In [244]:
# Create clean repo validation dataframe

repo_validation = repo_mountains_gdf.copy()

# Make numeric fields numeric
repo_validation["elev"] = pd.to_numeric(repo_validation["elev"], errors="coerce")
repo_validation["prom"] = pd.to_numeric(repo_validation["prom"], errors="coerce")

# Create clean name for matching
repo_validation["name_clean"] = repo_validation["name"].apply(normalize_name)

repo_validation[[
    "name",
    "name_clean",
    "elev",
    "prom",
    "coord",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "alt_names",
    "marker_size",
    "source_file"
]].head(20)

,name,name_clean,elev,prom,coord,is_volc,prov,region,isl_grp,alt_names,marker_size,source_file
0,Mount Aminduen,aminduen,1325.0,1325.0,11°6′14″N 124°44′33″E,NaN,[Leyte],[Region VIII],Visayas,[Alto Peak],medium,leyte.geojson
1,Mount Busay,busay,476.0,456.0,11°23′58″N 124°49′56″E,NaN,[Leyte],[Region VIII],Visayas,[],small,leyte.geojson
2,Mount Gumdalitan,gumdalitan,865.0,396.0,10°52′53″N 124°53′15″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
3,Mount Hingatungan,hingatungan,745.0,386.0,10°35′10″N 125°6′54″E,NaN,"[Leyte, Southern Leyte]",[Region VIII],Visayas,[],medium,leyte.geojson
4,Mount Lobi,lobi,1319.0,710.0,11°0′44″N 124°48′44″E,1.0,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
5,Mount Magsanga,magsanga,572.0,500.0,10°57′15″N 124°29′36″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
6,Mount Mahagnao,mahagnao,860.0,318.0,10°52′49″N 124°51′53″E,1.0,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
7,Mount Naburac,naburac,864.0,386.0,10°51′54″N 124°52′36″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
8,Mount Pangasugan,pangasugan,1150.0,403.0,10°45′35″N 124°50′22″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson
9,Mount Sacripante,sacripante,924.0,524.0,10°31′56″N 124°48′17″E,NaN,[Leyte],[Region VIII],Visayas,[],medium,leyte.geojson


In [245]:
# Compare main dataset columns with repo validation columns

print("Main dataset rows:", len(main_final_elev))
print("Repo validation rows:", len(repo_validation))

print("\nColumns in repo_validation:")
print(repo_validation.columns.tolist())

Main dataset rows: 2625
Repo validation rows: 519

Columns in repo_validation:
['name', 'elev', 'prom', 'coord', 'is_volc', 'prov', 'region', 'isl_grp', 'alt_names', 'marker_color', 'marker_size', 'marker_symbol', 'geometry', 'source_file', 'name_clean']


In [246]:
# Match main dataset with repo validation using clean mountain name

repo_match_cols = [
    "name_clean",
    "name",
    "elev",
    "prom",
    "coord",
    "is_volc",
    "prov",
    "region",
    "isl_grp",
    "alt_names",
    "marker_size",
    "source_file"
]

repo_match_cols = [col for col in repo_match_cols if col in repo_validation.columns]

repo_validation_match = repo_validation[repo_match_cols].copy()

# Rename repo columns before merging
repo_validation_match = repo_validation_match.rename(columns={
    "name": "repo_name",
    "elev": "repo_elev",
    "prom": "repo_prom",
    "coord": "repo_coord",
    "is_volc": "repo_is_volc",
    "prov": "repo_prov",
    "region": "repo_region",
    "isl_grp": "repo_isl_grp",
    "alt_names": "repo_alt_names",
    "marker_size": "repo_marker_size",
    "source_file": "repo_source_file"
})

main_repo_validation = main_final_elev.merge(
    repo_validation_match,
    on="name_clean",
    how="left"
)

print("Rows after repo validation merge:", len(main_repo_validation))
print("Rows matched with repo:", main_repo_validation["repo_name"].notna().sum())

main_repo_validation[[
    "name",
    "repo_name",
    "elev",
    "repo_elev",
    "prom",
    "repo_prom",
    "prov",
    "repo_prov",
    "region",
    "repo_region",
    "isl_grp",
    "repo_isl_grp"
]].head(20)

Rows after repo validation merge: 2677
Rows matched with repo: 560


,name,repo_name,elev,repo_elev,prom,repo_prom,prov,repo_prov,region,repo_region,isl_grp,repo_isl_grp
0,Mount 387,NaN,777.0,NaN,None,NaN,Nueva Ecija,NaN,Region III,NaN,Luzon,NaN
1,Mount Abao,NaN,2527.0,NaN,None,NaN,Ifugao,NaN,CAR,NaN,Luzon,NaN
2,Abel's Peak,NaN,1464.0,NaN,None,NaN,Romblon,NaN,MIMAROPA,NaN,Luzon,NaN
3,Mount Abnataclayong,NaN,370.0,NaN,None,NaN,Sarangani,NaN,Region XII,NaN,Mindanao,NaN
4,Mount Aboabo,NaN,NaN,NaN,None,NaN,Palawan,NaN,MIMAROPA,NaN,Luzon,NaN
5,Abong-abong Peak,NaN,885.0,NaN,None,NaN,Basilan,NaN,BARMM,NaN,Mindanao,NaN
6,Mount Abongabong,NaN,NaN,NaN,None,NaN,Occidental Mindoro,NaN,MIMAROPA,NaN,Luzon,NaN
7,Mount Aborlan,Mount Aborlan,745.0,745.0,None,408.0,Palawan,[Palawan],MIMAROPA,[Mimaropa],Luzon,Luzon
8,Mount Abra de Ilog,Mount Abra de Ilog,1018.0,1076.0,None,537.0,Occidental Mindoro,[Occidental Mindoro],MIMAROPA,[Mimaropa],Luzon,Luzon
9,Mount Abriringan,NaN,659.0,NaN,None,NaN,Cagayan,NaN,Region II,NaN,Luzon,NaN


In [247]:
# Create repo validation flags for elevation

main_repo_validation["repo_match_status"] = main_repo_validation["repo_name"].apply(
    lambda x: "matched_repo_geojson" if pd.notna(x) else "not_found_in_repo_geojson"
)

main_repo_validation["repo_elev_difference"] = (
    main_repo_validation["repo_elev"] - main_repo_validation["elev"]
)

def repo_elevation_validation_flag(row):
    if pd.isna(row["repo_elev"]):
        return "no_repo_elevation"

    if pd.isna(row["elev"]):
        return "no_original_elevation"

    diff = row["repo_elev"] - row["elev"]

    if abs(diff) <= 30:
        return "close_match"
    elif abs(diff) <= 100:
        return "minor_difference"
    else:
        return "needs_review"

main_repo_validation["repo_elev_validation_flag"] = main_repo_validation.apply(
    repo_elevation_validation_flag,
    axis=1
)

print(main_repo_validation["repo_match_status"].value_counts())
print(main_repo_validation["repo_elev_validation_flag"].value_counts())

main_repo_validation[[
    "name",
    "repo_name",
    "elev",
    "repo_elev",
    "repo_elev_difference",
    "repo_elev_validation_flag",
    "prov",
    "repo_prov",
    "region",
    "repo_region",
    "isl_grp",
    "repo_isl_grp"
]].head(30)

repo_match_status
not_found_in_repo_geojson    2117
matched_repo_geojson          560
Name: count, dtype: int64
repo_elev_validation_flag
no_repo_elevation        2117
close_match               316
no_original_elevation     150
needs_review               51
minor_difference           43
Name: count, dtype: int64


,name,repo_name,elev,repo_elev,repo_elev_difference,repo_elev_validation_flag,prov,repo_prov,region,repo_region,isl_grp,repo_isl_grp
0,Mount 387,NaN,777.0,NaN,NaN,no_repo_elevation,Nueva Ecija,NaN,Region III,NaN,Luzon,NaN
1,Mount Abao,NaN,2527.0,NaN,NaN,no_repo_elevation,Ifugao,NaN,CAR,NaN,Luzon,NaN
2,Abel's Peak,NaN,1464.0,NaN,NaN,no_repo_elevation,Romblon,NaN,MIMAROPA,NaN,Luzon,NaN
3,Mount Abnataclayong,NaN,370.0,NaN,NaN,no_repo_elevation,Sarangani,NaN,Region XII,NaN,Mindanao,NaN
4,Mount Aboabo,NaN,NaN,NaN,NaN,no_repo_elevation,Palawan,NaN,MIMAROPA,NaN,Luzon,NaN
5,Abong-abong Peak,NaN,885.0,NaN,NaN,no_repo_elevation,Basilan,NaN,BARMM,NaN,Mindanao,NaN
6,Mount Abongabong,NaN,NaN,NaN,NaN,no_repo_elevation,Occidental Mindoro,NaN,MIMAROPA,NaN,Luzon,NaN
7,Mount Aborlan,Mount Aborlan,745.0,745.0,0.0,close_match,Palawan,[Palawan],MIMAROPA,[Mimaropa],Luzon,Luzon
8,Mount Abra de Ilog,Mount Abra de Ilog,1018.0,1076.0,58.0,minor_difference,Occidental Mindoro,[Occidental Mindoro],MIMAROPA,[Mimaropa],Luzon,Luzon
9,Mount Abriringan,NaN,659.0,NaN,NaN,no_repo_elevation,Cagayan,NaN,Region II,NaN,Luzon,NaN


In [248]:
# Enrich prominence and volcanic status from repo GeoJSON
#If original value exists → keep original.
#If original missing and repo has value → enrich from repo.
#If both missing → keep missing.

# Prominence: use repo_prom only when original prom is missing
main_repo_validation["prom_enriched"] = main_repo_validation["prom"].fillna(
    main_repo_validation["repo_prom"]
)

main_repo_validation["prom_enrichment_source"] = main_repo_validation.apply(
    lambda row:
        "original"
        if pd.notna(row["prom"])
        else ("repo_geojson" if pd.notna(row["repo_prom"]) else "missing"),
    axis=1
)

# Volcanic status: use repo_is_volc only when original is_volc is missing
main_repo_validation["is_volc_enriched"] = main_repo_validation["is_volc"].fillna(
    main_repo_validation["repo_is_volc"]
)

main_repo_validation["is_volc_enrichment_source"] = main_repo_validation.apply(
    lambda row:
        "original"
        if pd.notna(row["is_volc"])
        else ("repo_geojson" if pd.notna(row["repo_is_volc"]) else "missing"),
    axis=1
)

/tmp/ipykernel_1326/2456598997.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  main_repo_validation["prom_enriched"] = main_repo_validation["prom"].fillna(


In [249]:
print("Prominence enrichment source:")
print(main_repo_validation["prom_enrichment_source"].value_counts(dropna=False))

print("\nVolcanic status enrichment source:")
print(main_repo_validation["is_volc_enrichment_source"].value_counts(dropna=False))



Prominence enrichment source:
prom_enrichment_source
missing         2117
repo_geojson     560
Name: count, dtype: int64

Volcanic status enrichment source:
is_volc_enrichment_source
missing         2643
repo_geojson      34
Name: count, dtype: int64


In [250]:
print("Original prom missing:", main_repo_validation["prom"].isna().sum())
print("Enriched prom missing:", main_repo_validation["prom_enriched"].isna().sum())

print("\nOriginal is_volc missing:", main_repo_validation["is_volc"].isna().sum())
print("Enriched is_volc missing:", main_repo_validation["is_volc_enriched"].isna().sum())

Original prom missing: 2677
Enriched prom missing: 2117

Original is_volc missing: 2677
Enriched is_volc missing: 2643


In [251]:
main_repo_validation.to_csv(
    "philippines_mountains_with_peakvisor_repo_enrichment.csv",
    index=False
)

print("Saved: philippines_mountains_with_peakvisor_repo_enrichment.csv")
print("Rows:", len(main_repo_validation))
print("Columns:", len(main_repo_validation.columns))

Saved: philippines_mountains_with_peakvisor_repo_enrichment.csv
Rows: 2677
Columns: 84


# Final consolidation step

In [252]:
#Load your current masterfile
import pandas as pd

maindataframe = pd.read_csv("philippines_mountains_with_peakvisor_repo_enrichment.csv")
print("Rows, Columns:", maindataframe.shape)
maindataframe.columns.tolist()


Rows, Columns: (2677, 84)


['mountain_id',
 'name',
 'name_original',
 'source',
 'name_clean',
 'alt_names',
 'elev',
 'prom',
 'marker_size',
 'coord',
 'lat',
 'lon',
 'is_volc',
 'isl_grp',
 'elev_raw',
 'osm_id',
 'osm_type',
 'osm_tags',
 'data_quality_notes',
 'geometry',
 'index_right',
 'prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'prov_gis_original',
 'region_gis_original',
 'source_file',
 'source.1',
 'coord_source',
 'admin_note',
 'detail_url',
 'name_quickmap',
 'lat_quickmap',
 'lon_quickmap',
 'elev_quickmap',
 'quickmap_match',
 'quickmap_distance_km',
 'quickmap_validation_note',
 'quickmap_status',
 'quickmap_confidence',
 'priority_group',
 'peakvisor_name',
 'peakvisor_elev_m',
 'peakvisor_prom_m',
 'peakvisor_elev_source_unit',
 'peakvisor_prom_source_unit',
 'peakvisor_elev_difference_m',
 'peakvisor_elev_validation_flag',
 'peakvisor_lat',
 'peakvisor_lon',
 'peakvisor_location_text',
 'peakvisor_about_text',
 'has_trail_info',
 'has_volcano_info',
 'trail_sente

In [253]:
# ============================================================
# Clean / reorder final dataframe columns
# ============================================================

# Use your final working dataframe
final_df = maindataframe.copy()

# Optional: clean accidental spaces in column names
final_df.columns = final_df.columns.str.strip()

# Preferred column order
preferred_cols = [
    # IDs and names
    "mountain_id",
    "name",
    "name_original",
    "name_clean",
    "alt_names",

    # Final BI-ready elevation / marker fields
    "elev_final_m",
    "elev_final_source",
    "marker_size",
    "expected_marker_size",
    "marker_size_check",

    # Original elevation/prominence
    "elev",
    "prom",

    # PeakVisor elevation/prominence
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "peakvisor_elev_source_unit",
    "peakvisor_prom_source_unit",
    "peakvisor_elev_difference_m",
    "peakvisor_elev_validation_flag",

    # GeoJSON / repo enrichment
    "repo_name",
    "repo_elev",
    "repo_prom",
    "repo_coord",
    "repo_is_volc",
    "repo_prov",
    "repo_region",
    "repo_isl_grp",
    "repo_alt_names",
    "repo_marker_size",
    "repo_source_file",
    "repo_match_status",
    "repo_elev_difference",
    "repo_elev_validation_flag",

    # Final enriched fields
    "prom_enriched",
    "prom_enrichment_source",
    "is_volc_enriched",
    "is_volc_enrichment_source",

    # Coordinates
    "coord",
    "lat",
    "lon",
    "coord_source",
    "lat_quickmap",
    "lon_quickmap",
    "peakvisor_lat",
    "peakvisor_lon",

    # Location / admin fields
    "prov",
    "prov_psgc_code",
    "region",
    "region_name",
    "region_code",
    "isl_grp",
    "prov_gis_original",
    "region_gis_original",
    "admin_note",

    # Volcano fields
    "is_volc",
    "has_volcano_info",
    "volcano_sentence",

    # Trail / descriptive fields
    "has_trail_info",
    "trail_sentence",
    "peakvisor_location_text",
    "peakvisor_about_text",

    # Source and quality fields
    "source",
    "source1",
    "source_file",
    "data_quality_notes",
    "quickmap_match",
    "quickmap_distance_km",
    "quickmap_validation_note",
    "quickmap_status",
    "quickmap_confidence",
    "priority_group",
    "detail_scrape_status",
    "url_source",

    # URLs
    "detail_url",
    "detail_url_peakvisor",

    # OSM / geometry fields
    "osm_id",
    "osm_type",
    "osm_tags",
    "geometry",
    "index_right",

    # Helper keys
    "peak_key"
]

# Keep only preferred columns that actually exist
preferred_existing = [
    col for col in preferred_cols
    if col in final_df.columns
]

# Keep all other columns at the end, so nothing is accidentally deleted
remaining_cols = [
    col for col in final_df.columns
    if col not in preferred_existing
]

# Create cleaned/reordered dataframe
final_df_cleaned = final_df[preferred_existing + remaining_cols].copy()

print("Original shape:", final_df.shape)
print("Cleaned column-order shape:", final_df_cleaned.shape)
print("Preferred columns found:", len(preferred_existing))
print("Extra columns kept at end:", len(remaining_cols))
print("Total columns kept:", len(final_df_cleaned.columns))

print("\nMissing preferred columns:")
print([col for col in preferred_cols if col not in final_df.columns])

final_df_cleaned.head()

Original shape: (2677, 84)
Cleaned column-order shape: (2677, 84)
Preferred columns found: 79
Extra columns kept at end: 5
Total columns kept: 84

Missing preferred columns:
['source1']


,mountain_id,name,name_original,name_clean,alt_names,elev_final_m,elev_final_source,marker_size,expected_marker_size,marker_size_check,...,osm_type,osm_tags,geometry,index_right,peak_key,elev_raw,source.1,name_quickmap,elev_quickmap,peakvisor_name
0,387_15.9003_120.9738,Mount 387,Mount 387,387,[],777.0,original_elev,medium,medium,correct,...,node,"{'alt_name': 'Mount Batong Amat', 'ele': '777'...",POINT (120.9737706 15.900298),78.0,mount 387 | nueva ecija | region iii,777,OpenStreetMap,NaN,NaN,NaN
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,abao,[],2527.0,original_elev,large,large,correct,...,node,"{'ele': '2527', 'gns:dsg': 'MT', 'gns:uni': '-...",POINT (120.9148864 16.8323069),10.0,mount abao | ifugao | car,2527,OpenStreetMap,Mount Abao,2524.0,NaN
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,abel's peak,[],1464.0,original_elev,medium,medium,correct,...,node,"{'ele': '1464', 'name': ""Abel's Peak"", 'natura...",POINT (122.5737552 12.4311298),27.0,abel's peak | romblon | mimaropa,1464,OpenStreetMap,NaN,NaN,NaN
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,[],370.0,original_elev,small,small,correct,...,node,"{'ele': '370', 'gns:dsg': 'MT', 'gns:uni': '-3...",POINT (125.366944 5.675),55.0,mount abnataclayong | sarangani | region xii,370,OpenStreetMap,Mount Abnataclayong,370.0,NaN
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,aboabo,[],395.0,peakvisor_missing_elev_enrichment,small,small,correct,...,node,"{'gns:dsg': 'MT', 'gns:uni': '-3321795', 'is_i...",POINT (118.060024 9.1475284),26.0,mount aboabo | palawan | mimaropa,NaN,OpenStreetMap,NaN,NaN,NaN


In [254]:
final_df_cleaned.columns.tolist()

['mountain_id',
 'name',
 'name_original',
 'name_clean',
 'alt_names',
 'elev_final_m',
 'elev_final_source',
 'marker_size',
 'expected_marker_size',
 'marker_size_check',
 'elev',
 'prom',
 'peakvisor_elev_m',
 'peakvisor_prom_m',
 'peakvisor_elev_source_unit',
 'peakvisor_prom_source_unit',
 'peakvisor_elev_difference_m',
 'peakvisor_elev_validation_flag',
 'repo_name',
 'repo_elev',
 'repo_prom',
 'repo_coord',
 'repo_is_volc',
 'repo_prov',
 'repo_region',
 'repo_isl_grp',
 'repo_alt_names',
 'repo_marker_size',
 'repo_source_file',
 'repo_match_status',
 'repo_elev_difference',
 'repo_elev_validation_flag',
 'prom_enriched',
 'prom_enrichment_source',
 'is_volc_enriched',
 'is_volc_enrichment_source',
 'coord',
 'lat',
 'lon',
 'coord_source',
 'lat_quickmap',
 'lon_quickmap',
 'peakvisor_lat',
 'peakvisor_lon',
 'prov',
 'prov_psgc_code',
 'region',
 'region_name',
 'region_code',
 'isl_grp',
 'prov_gis_original',
 'region_gis_original',
 'admin_note',
 'is_volc',
 'has_volcano

In [255]:
#Duplicate Consolidation Rules

1. Look only at duplicated rows, probably the same mountain
2. Find rows where elev is missing
3. look for another row with elev available
4. Prefer the elevation from the row with:
   - High QuickMap confidence first
   - then good PeakVisor validation
   - then good GeoJSON/repo validation
5. Use that value to fill the missing elev within the duplicate group

Probably the same mountain
- same/similar name_clean
- coordinates very close
- same province/region or same island group

In [256]:
# 1. Check missing identity fields

identity_cols = [
    "mountain_id",
    "name",
    "name_original",
    "name_clean",
    "alt_names"
]

identity_cols_existing = [
    col for col in identity_cols
    if col in final_df_cleaned.columns
]

final_df_cleaned[identity_cols_existing].isna().sum()

,0
mountain_id,0
name,0
name_original,0
name_clean,0
alt_names,0


In [257]:
print("Existing identity columns:", identity_cols_existing)
print("Missing identity columns:", [col for col in identity_cols if col not in final_df_cleaned.columns])

final_df_cleaned[identity_cols_existing].isna().sum()

Existing identity columns: ['mountain_id', 'name', 'name_original', 'name_clean', 'alt_names']
Missing identity columns: []


,0
mountain_id,0
name,0
name_original,0
name_clean,0
alt_names,0


In [258]:
# 3. Check same-name mountain groups with missing final elevation

same_name_groups = final_df_cleaned[
    final_df_cleaned["name_clean"].duplicated(keep=False)
].copy()

same_name_elev_check = same_name_groups.groupby("name_clean").agg(
    row_count=("name_clean", "size"),
    missing_elev_count=("elev_final_m", lambda x: x.isna().sum()),
    available_elev_count=("elev_final_m", lambda x: x.notna().sum()),
    provinces=("prov", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
    regions=("region", lambda x: ", ".join(sorted(x.dropna().astype(str).unique()))),
    island_groups=("isl_grp", lambda x: ", ".join(sorted(x.dropna().astype(str).unique())))
).reset_index()

same_name_groups_with_missing_elev = same_name_elev_check[
    (same_name_elev_check["missing_elev_count"] > 0)
    & (same_name_elev_check["available_elev_count"] > 0)
].copy()

print("Same-name groups:", same_name_elev_check.shape[0])
print("Same-name groups with missing elev and available elev:", same_name_groups_with_missing_elev.shape[0])

same_name_groups_with_missing_elev.head(50)

Same-name groups: 113
Same-name groups with missing elev and available elev: 5


,name_clean,row_count,missing_elev_count,available_elev_count,provinces,regions,island_groups
16,big peak,2,1,1,Palawan,MIMAROPA,Luzon
38,dome peak,4,3,1,"Negros Oriental, Palawan, Unassigned","MIMAROPA, NIR, Unassigned","Luzon, Unassigned, Visayas"
39,double peak,6,2,4,"Palawan, Surigao del Norte","MIMAROPA, Region XIII","Luzon, Mindanao"
88,round hill,2,1,1,Palawan,MIMAROPA,Luzon
93,sharp peak,88,76,12,"Catanduanes, Davao del Sur, Lanao del Sur, Pal...","BARMM, MIMAROPA, Region IX, Region V, Region V...","Luzon, Mindanao, Visayas"


96 same-name groups -> only 5 rows passed close-coordinate/admin validation -> those 5 are safe to fill automatically

In [259]:
# View actual rows inside those same-name groups

same_name_missing_elev_rows = final_df_cleaned[
    final_df_cleaned["name_clean"].isin(
        same_name_groups_with_missing_elev["name_clean"]
    )
].copy()

same_name_missing_elev_rows[
    [
        "name_clean",
        "name",
        "name_original",
        "mountain_id",
        "elev",
        "elev_final_m",
        "lat",
        "lon",
        "prov",
        "region",
        "isl_grp"
    ]
].sort_values(["name_clean", "elev_final_m"])

,name_clean,name,name_original,mountain_id,elev,elev_final_m,lat,lon,prov,region,isl_grp
341,big peak,Big Peak,Big Peak,big peak_10.5725_119.6228,389.0,389.0,10.572500,119.622778,Palawan,MIMAROPA,Luzon
342,big peak,Big Peak,Big Peak,big peak_10.5833_119.5167,NaN,NaN,10.583333,119.516667,Palawan,MIMAROPA,Luzon
799,dome peak,Dome Peak,Dome Peak,dome peak_9.21_122.9958,846.0,846.0,9.210000,122.995833,Negros Oriental,NIR,Visayas
800,dome peak,Dome Peak,Dome Peak,dome peak_5.6167_125.3333,NaN,NaN,5.616667,125.333333,Unassigned,Unassigned,Unassigned
801,dome peak,Dome Peak,Dome Peak,dome peak_11.0383_119.5949,NaN,NaN,11.038261,119.594872,Palawan,MIMAROPA,Luzon
...,...,...,...,...,...,...,...,...,...,...,...
2227,sharp peak,Sharp Peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,7.436111,122.254167,Zamboanga del Sur,Region IX,Mindanao
2228,sharp peak,Sharp Peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,7.436111,122.254167,Zamboanga del Sur,Region IX,Mindanao
2229,sharp peak,Sharp Peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,7.436111,122.254167,Zamboanga del Sur,Region IX,Mindanao
2230,sharp peak,Sharp Peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,7.436111,122.254167,Zamboanga del Sur,Region IX,Mindanao


In [260]:
# Score rows for best elevation donor

elev_candidate_rows = same_name_missing_elev_rows.copy()

def quickmap_score(row):
    confidence = str(row.get("quickmap_confidence", "")).lower()

    if "high confidence" in confidence:
        return 3
    elif "medium confidence" in confidence:
        return 2
    elif "low confidence" in confidence:
        return 1
    else:
        return 0


def peakvisor_score(row):
    flag = str(row.get("peakvisor_elev_validation_flag", "")).lower()

    if flag == "close_match":
        return 3
    elif flag == "minor_difference":
        return 2
    elif flag == "needs_review":
        return 1
    else:
        return 0


def repo_score(row):
    flag = str(row.get("repo_elev_validation_flag", "")).lower()

    if flag == "close_match":
        return 3
    elif flag == "minor_difference":
        return 2
    elif flag == "needs_review":
        return 1
    else:
        return 0


elev_candidate_rows["quickmap_score"] = elev_candidate_rows.apply(quickmap_score, axis=1)
elev_candidate_rows["peakvisor_score"] = elev_candidate_rows.apply(peakvisor_score, axis=1)
elev_candidate_rows["repo_score"] = elev_candidate_rows.apply(repo_score, axis=1)

elev_candidate_rows["elev_donor_score"] = (
    elev_candidate_rows["elev_final_m"].notna().astype(int) * 100
    + elev_candidate_rows["quickmap_score"] * 10
    + elev_candidate_rows["peakvisor_score"] * 5
    + elev_candidate_rows["repo_score"] * 3
)

elev_candidate_rows[
    [
        "name_clean",
        "name",
        "mountain_id",
        "elev",
        "elev_final_m",
        "elev_final_source",
        "quickmap_confidence",
        "peakvisor_elev_validation_flag",
        "repo_elev_validation_flag",
        "quickmap_score",
        "peakvisor_score",
        "repo_score",
        "elev_donor_score"
    ]
].sort_values(["name_clean", "elev_donor_score"], ascending=[True, False]).head(100)

,name_clean,name,mountain_id,elev,elev_final_m,elev_final_source,quickmap_confidence,peakvisor_elev_validation_flag,repo_elev_validation_flag,quickmap_score,peakvisor_score,repo_score,elev_donor_score
341,big peak,Big Peak,big peak_10.5725_119.6228,389.0,389.0,original_elev,High confidence,NaN,no_repo_elevation,3,0,0,130
342,big peak,Big Peak,big peak_10.5833_119.5167,NaN,NaN,still_missing,Medium confidence,NaN,no_repo_elevation,2,0,0,20
799,dome peak,Dome Peak,dome peak_9.21_122.9958,846.0,846.0,original_elev,High confidence,NaN,minor_difference,3,0,2,136
800,dome peak,Dome Peak,dome peak_5.6167_125.3333,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10
801,dome peak,Dome Peak,dome peak_11.0383_119.5949,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2224,sharp peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10
2225,sharp peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10
2226,sharp peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10
2227,sharp peak,Sharp Peak,sharp peak_7.4361_122.2542,NaN,NaN,still_missing,Low confidence - likely wrong name match,NaN,no_original_elevation,1,0,0,10


In [261]:
# Pick best available elevation donor per name_clean group

best_elev_donor_by_name = (
    elev_candidate_rows[elev_candidate_rows["elev"].notna()]
    .sort_values(
        ["name_clean", "elev_donor_score"],
        ascending=[True, False]
    )
    .groupby("name_clean", as_index=False)
    .first()[
        [
            "name_clean",
            "name",
            "elev",
            "lat",
            "lon",
            "prov",
            "region",
            "isl_grp",
            "quickmap_confidence",
            "peakvisor_elev_validation_flag",
            "repo_elev_validation_flag",
            "elev_donor_score"
        ]
    ]
    .rename(columns={
        "name": "elev_donor_name",
        "elev": "best_group_elev",
        "lat": "elev_donor_lat",
        "lon": "elev_donor_lon",
        "prov": "elev_donor_prov",
        "region": "elev_donor_region",
        "isl_grp": "elev_donor_isl_grp",
        "quickmap_confidence": "elev_donor_quickmap_confidence",
        "peakvisor_elev_validation_flag": "elev_donor_peakvisor_flag",
        "repo_elev_validation_flag": "elev_donor_repo_flag"
    })
)

best_elev_donor_by_name.head(50)

,name_clean,elev_donor_name,best_group_elev,elev_donor_lat,elev_donor_lon,elev_donor_prov,elev_donor_region,elev_donor_isl_grp,elev_donor_quickmap_confidence,elev_donor_peakvisor_flag,elev_donor_repo_flag,elev_donor_score
0,big peak,Big Peak,389.0,10.572500,119.622778,Palawan,MIMAROPA,Luzon,High confidence,None,no_repo_elevation,130
1,dome peak,Dome Peak,846.0,9.210000,122.995833,Negros Oriental,NIR,Visayas,High confidence,None,minor_difference,136
2,double peak,Double Peak,544.0,8.723333,117.424722,Palawan,MIMAROPA,Luzon,High confidence,None,close_match,139
3,round hill,Round Hill,126.0,7.927500,116.965556,Palawan,MIMAROPA,Luzon,No QuickMap match,None,no_repo_elevation,100
4,sharp peak,Sharp Peak,775.0,8.914256,117.663814,Palawan,MIMAROPA,Luzon,High confidence,None,minor_difference,136


In [262]:
# Merge best elevation donor back to the same-name rows

elev_fill_review = same_name_missing_elev_rows.merge(
    best_elev_donor_by_name,
    on="name_clean",
    how="left"
)

elev_fill_review.head()

,mountain_id,name,name_original,name_clean,alt_names,elev_final_m,elev_final_source,marker_size,expected_marker_size,marker_size_check,...,best_group_elev,elev_donor_lat,elev_donor_lon,elev_donor_prov,elev_donor_region,elev_donor_isl_grp,elev_donor_quickmap_confidence,elev_donor_peakvisor_flag,elev_donor_repo_flag,elev_donor_score
0,big peak_10.5725_119.6228,Big Peak,Big Peak,big peak,[],389.0,original_elev,small,small,correct,...,389.0,10.5725,119.622778,Palawan,MIMAROPA,Luzon,High confidence,None,no_repo_elevation,130
1,big peak_10.5833_119.5167,Big Peak,Big Peak,big peak,[],NaN,still_missing,unknown,unknown,correct,...,389.0,10.5725,119.622778,Palawan,MIMAROPA,Luzon,High confidence,None,no_repo_elevation,130
2,dome peak_9.21_122.9958,Dome Peak,Dome Peak,dome peak,[],846.0,original_elev,medium,medium,correct,...,846.0,9.2100,122.995833,Negros Oriental,NIR,Visayas,High confidence,None,minor_difference,136
3,dome peak_5.6167_125.3333,Dome Peak,Dome Peak,dome peak,[],NaN,still_missing,unknown,unknown,correct,...,846.0,9.2100,122.995833,Negros Oriental,NIR,Visayas,High confidence,None,minor_difference,136
4,dome peak_11.0383_119.5949,Dome Peak,Dome Peak,dome peak,[],NaN,still_missing,unknown,unknown,correct,...,846.0,9.2100,122.995833,Negros Oriental,NIR,Visayas,High confidence,None,minor_difference,136


In [263]:
# Calculate distance from each row to its best elevation donor

import numpy as np

def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


elev_fill_review["distance_to_elev_donor_km"] = elev_fill_review.apply(
    lambda row: haversine_km(
        row["lat"],
        row["lon"],
        row["elev_donor_lat"],
        row["elev_donor_lon"]
    ),
    axis=1
)

In [264]:
# Confirm which missing-elevation rows are safe to fill

elev_fill_review["same_admin_area"] = (
    (elev_fill_review["prov"] == elev_fill_review["elev_donor_prov"])
    | (elev_fill_review["region"] == elev_fill_review["elev_donor_region"])
    | (elev_fill_review["isl_grp"] == elev_fill_review["elev_donor_isl_grp"])
)

elev_fill_review["safe_to_fill_elev"] = (
    elev_fill_review["elev"].isna()
    & elev_fill_review["best_group_elev"].notna()
    & (
        (elev_fill_review["distance_to_elev_donor_km"] <= 5)
        | (
            elev_fill_review["distance_to_elev_donor_km"].isna()
            & elev_fill_review["same_admin_area"]
        )
    )
)

print("Rows safe to fill elevation:", elev_fill_review["safe_to_fill_elev"].sum())

elev_fill_review[
    [
        "name_clean",
        "name",
        "elev",
        "best_group_elev",
        "distance_to_elev_donor_km",
        "same_admin_area",
        "safe_to_fill_elev",
        "quickmap_confidence",
        "elev_donor_quickmap_confidence",
        "elev_donor_peakvisor_flag",
        "elev_donor_repo_flag"
    ]
].sort_values(["safe_to_fill_elev", "name_clean"], ascending=[False, True]).head(50)

Rows safe to fill elevation: 0


,name_clean,name,elev,best_group_elev,distance_to_elev_donor_km,same_admin_area,safe_to_fill_elev,quickmap_confidence,elev_donor_quickmap_confidence,elev_donor_peakvisor_flag,elev_donor_repo_flag
0,big peak,Big Peak,389.0,389.0,0.000000,True,False,High confidence,High confidence,None,no_repo_elevation
1,big peak,Big Peak,NaN,389.0,11.660878,True,False,Medium confidence,High confidence,None,no_repo_elevation
2,dome peak,Dome Peak,846.0,846.0,0.000000,True,False,High confidence,High confidence,None,minor_difference
3,dome peak,Dome Peak,NaN,846.0,475.455816,False,False,Low confidence - likely wrong name match,High confidence,None,minor_difference
4,dome peak,Dome Peak,NaN,846.0,424.154912,False,False,Low confidence - likely wrong name match,High confidence,None,minor_difference
5,dome peak,Dome Peak,NaN,846.0,432.397161,False,False,Low confidence - likely wrong name match,High confidence,None,minor_difference
6,double peak,Double Peak,930.0,544.0,896.771772,False,False,Low confidence - likely wrong name match,High confidence,None,close_match
7,double peak,Double Peak,930.0,544.0,896.771772,False,False,Low confidence - likely wrong name match,High confidence,None,close_match
8,double peak,Double Peak,544.0,544.0,0.000000,True,False,High confidence,High confidence,None,close_match
9,double peak,Double Peak,544.0,544.0,0.000000,True,False,High confidence,High confidence,None,close_match


In [265]:
# Fill missing elevation only for safe same-mountain matches

maindataframe_elev_filled = final_df_cleaned.copy()

safe_elev_fills = elev_fill_review[
    elev_fill_review["safe_to_fill_elev"]
][["name_clean", "name", "lat", "lon", "best_group_elev"]].copy()

maindataframe_elev_filled = maindataframe_elev_filled.merge(
    safe_elev_fills,
    on=["name_clean", "name", "lat", "lon"],
    how="left"
)

maindataframe_elev_filled["elev_filled_from_same_mountain"] = (
    maindataframe_elev_filled["elev"].isna()
    & maindataframe_elev_filled["best_group_elev"].notna()
)

maindataframe_elev_filled["elev_after_same_mountain_fill"] = maindataframe_elev_filled["elev"].fillna(
    maindataframe_elev_filled["best_group_elev"]
)

print("Original missing elev:", final_df_cleaned["elev"].isna().sum())
print("After same-mountain fill missing elev:", maindataframe_elev_filled["elev_after_same_mountain_fill"].isna().sum())
print("Rows filled:", maindataframe_elev_filled["elev_filled_from_same_mountain"].sum())

Original missing elev: 1078
After same-mountain fill missing elev: 1078
Rows filled: 0


In [266]:
maindataframe_elev_filled[
    maindataframe_elev_filled["elev_filled_from_same_mountain"]
][
    [
        "name_clean",
        "name",
        "elev",
        "best_group_elev",
        "elev_after_same_mountain_fill",
        "lat",
        "lon",
        "prov",
        "region",
        "isl_grp"
    ]
]

,name_clean,name,elev,best_group_elev,elev_after_same_mountain_fill,lat,lon,prov,region,isl_grp


---

Missing elev row
↓
Find same-name candidate
↓
Confirm close coordinates/admin area
↓
Copy elevation from best donor row

---

For the remaining missing elevation rows,
try to fill from external/enrichment columns
but only when the validation is strong enough.



For remaining missing elev:

1. Use PeakVisor if:
   peakvisor_elev_validation_flag = close_match
   or PeakVisor and repo elevation are close to each other

2. Else use repo if:
   repo_elev_validation_flag = close_match
   and PeakVisor is missing

3. Else use minor_difference only if:
   there is no close_match source

4. Else leave blank / review

In [267]:
# Fill remaining missing elevation using validated PeakVisor / repo sources

elev_source_fill = maindataframe_elev_filled.copy()

# Make sure elevation columns are numeric
for col in [
    "elev_after_same_mountain_fill",
    "peakvisor_elev_m",
    "repo_elev"
]:
    if col in elev_source_fill.columns:
        elev_source_fill[col] = pd.to_numeric(elev_source_fill[col], errors="coerce")


def choose_external_elev(row):
    current_elev = row.get("elev_after_same_mountain_fill")
    pv_elev = row.get("peakvisor_elev_m")
    repo_elev = row.get("repo_elev")

    pv_flag = str(row.get("peakvisor_elev_validation_flag", "")).lower()
    repo_flag = str(row.get("repo_elev_validation_flag", "")).lower()

    # If already has elevation, keep it
    if pd.notna(current_elev):
        return current_elev, "existing_elev", "kept existing elevation"

    pv_good = pv_flag in ["close_match", "minor_difference"]
    repo_good = repo_flag in ["close_match", "minor_difference"]

    # Case 1: both PeakVisor and repo are good
    if pd.notna(pv_elev) and pd.notna(repo_elev) and pv_good and repo_good:
        diff = abs(pv_elev - repo_elev)

        if diff <= 30:
            return pv_elev, "peakvisor_repo_agree", "PeakVisor and repo agree within 30m"

        elif diff <= 100:
            if pv_flag == "close_match" and repo_flag != "close_match":
                return pv_elev, "peakvisor_close_match", "PeakVisor stronger than repo"
            elif repo_flag == "close_match" and pv_flag != "close_match":
                return repo_elev, "repo_close_match", "repo stronger than PeakVisor"
            else:
                return pd.NA, "review", "PeakVisor and repo differ by 31-100m"

        else:
            return pd.NA, "review", "PeakVisor and repo differ by more than 100m"

    # Case 2: only PeakVisor is good
    if pd.notna(pv_elev) and pv_good:
        return pv_elev, "peakvisor", "filled from validated PeakVisor elevation"

    # Case 3: only repo is good
    if pd.notna(repo_elev) and repo_good:
        return repo_elev, "repo_geojson", "filled from validated repo elevation"

    # Case 4: no safe source
    return pd.NA, "missing", "no safe elevation source"


elev_source_fill[
    [
        "elev_after_external_fill",
        "elev_external_source",
        "elev_external_note"
    ]
] = elev_source_fill.apply(
    lambda row: pd.Series(choose_external_elev(row)),
    axis=1
)

print("Missing after same-mountain fill:", elev_source_fill["elev_after_same_mountain_fill"].isna().sum())
print("Missing after external fill:", elev_source_fill["elev_after_external_fill"].isna().sum())

elev_source_fill["elev_external_source"].value_counts(dropna=False)

Missing after same-mountain fill: 1078
Missing after external fill: 1075


,count
elev_external_source,
existing_elev,1599
missing,1075
peakvisor,3


In [268]:
elev_source_fill[
    elev_source_fill["elev_after_same_mountain_fill"].isna()
    & elev_source_fill["elev_after_external_fill"].notna()
][
    [
        "name",
        "elev_after_same_mountain_fill",
        "elev_after_external_fill",
        "elev_external_source",
        "elev_external_note",
        "peakvisor_elev_m",
        "peakvisor_elev_validation_flag",
        "repo_elev",
        "repo_elev_validation_flag"
    ]
].head(100)

,name,elev_after_same_mountain_fill,elev_after_external_fill,elev_external_source,elev_external_note,peakvisor_elev_m,peakvisor_elev_validation_flag,repo_elev,repo_elev_validation_flag
362,Mount Bingo,NaN,555.0,peakvisor,filled from validated PeakVisor elevation,555.0,minor_difference,NaN,no_repo_elevation
1259,Mount Lobo,NaN,2121.0,peakvisor,filled from validated PeakVisor elevation,2121.0,close_match,974.0,no_original_elevation
1260,Mount Lobo,NaN,2121.0,peakvisor,filled from validated PeakVisor elevation,2121.0,close_match,2121.0,no_original_elevation


In [269]:
# Check why no external elevations were filled

remaining_missing_elev = elev_source_fill[
    elev_source_fill["elev_after_same_mountain_fill"].isna()
].copy()

print("Remaining missing elevation rows:", len(remaining_missing_elev))

print("\nPeakVisor elevation availability:")
print(remaining_missing_elev["peakvisor_elev_m"].notna().value_counts(dropna=False))

print("\nPeakVisor validation flags:")
print(remaining_missing_elev["peakvisor_elev_validation_flag"].value_counts(dropna=False))

print("\nRepo elevation availability:")
print(remaining_missing_elev["repo_elev"].notna().value_counts(dropna=False))

print("\nRepo validation flags:")
print(remaining_missing_elev["repo_elev_validation_flag"].value_counts(dropna=False))

print("\nExternal source decision:")
print(remaining_missing_elev["elev_external_source"].value_counts(dropna=False))

Remaining missing elevation rows: 1078

PeakVisor elevation availability:
peakvisor_elev_m
False    1074
True        4
Name: count, dtype: int64

PeakVisor validation flags:
peakvisor_elev_validation_flag
NaN                 1074
close_match            2
needs_review           1
minor_difference       1
Name: count, dtype: int64

Repo elevation availability:
repo_elev
False    928
True     150
Name: count, dtype: int64

Repo validation flags:
repo_elev_validation_flag
no_repo_elevation        928
no_original_elevation    150
Name: count, dtype: int64

External source decision:
elev_external_source
missing      1075
peakvisor       3
Name: count, dtype: int64


In [270]:
# Rows with PeakVisor or repo elevation available but not used

external_elev_available_but_not_used = remaining_missing_elev[
    remaining_missing_elev["peakvisor_elev_m"].notna()
    | remaining_missing_elev["repo_elev"].notna()
].copy()

print("Rows with external elevation available but not used:", len(external_elev_available_but_not_used))

external_elev_available_but_not_used[
    [
        "name",
        "elev_after_same_mountain_fill",
        "elev_after_external_fill",
        "elev_external_source",
        "elev_external_note",
        "peakvisor_elev_m",
        "peakvisor_elev_validation_flag",
        "repo_elev",
        "repo_elev_validation_flag"
    ]
].head(100)

Rows with external elevation available but not used: 151


,name,elev_after_same_mountain_fill,elev_after_external_fill,elev_external_source,elev_external_note,peakvisor_elev_m,peakvisor_elev_validation_flag,repo_elev,repo_elev_validation_flag
29,Mount Agdulasan,NaN,<NA>,missing,no safe elevation source,NaN,NaN,819.0,no_original_elevation
46,Mount Agudo,NaN,<NA>,missing,no safe elevation source,NaN,NaN,812.0,no_original_elevation
47,Mount Agudo,NaN,<NA>,missing,no safe elevation source,NaN,NaN,812.0,no_original_elevation
48,Mount Agudo,NaN,<NA>,missing,no safe elevation source,NaN,NaN,812.0,no_original_elevation
49,Mount Agudo,NaN,<NA>,missing,no safe elevation source,NaN,NaN,812.0,no_original_elevation
...,...,...,...,...,...,...,...,...,...
2188,Sharp Peak,NaN,<NA>,missing,no safe elevation source,NaN,NaN,684.0,no_original_elevation
2189,Sharp Peak,NaN,<NA>,missing,no safe elevation source,NaN,NaN,684.0,no_original_elevation
2190,Sharp Peak,NaN,<NA>,missing,no safe elevation source,NaN,NaN,684.0,no_original_elevation
2191,Sharp Peak,NaN,<NA>,missing,no safe elevation source,NaN,NaN,684.0,no_original_elevation


checkpoint: realized that a lot has no elevation and validation

cta: compare more sources

In [271]:
# ============================================================
# Fill remaining missing elevation using validated PeakVisor / repo sources
# ============================================================

maindataframe_elev_external_filled = maindataframe_elev_filled.copy()

# Make sure elevation columns are numeric
for col in [
    "elev_after_same_mountain_fill",
    "peakvisor_elev_m",
    "repo_elev"
]:
    if col in maindataframe_elev_external_filled.columns:
        maindataframe_elev_external_filled[col] = pd.to_numeric(
            maindataframe_elev_external_filled[col],
            errors="coerce"
        )


def choose_external_elev(row):
    current_elev = row.get("elev_after_same_mountain_fill")
    pv_elev = row.get("peakvisor_elev_m")
    repo_elev = row.get("repo_elev")

    pv_flag = str(row.get("peakvisor_elev_validation_flag", "")).lower()
    repo_flag = str(row.get("repo_elev_validation_flag", "")).lower()

    # If already has elevation, keep it
    if pd.notna(current_elev):
        return pd.Series({
            "elev_after_external_fill": current_elev,
            "elev_external_source": "already_filled_or_original",
            "elev_external_note": "kept existing elevation"
        })

    # Use PeakVisor only if validated well enough
    if pd.notna(pv_elev) and pv_flag in ["close_match", "minor_difference"]:
        return pd.Series({
            "elev_after_external_fill": pv_elev,
            "elev_external_source": "peakvisor",
            "elev_external_note": pv_flag
        })

    # Use repo only if validated well enough
    if pd.notna(repo_elev) and repo_flag in ["close_match", "minor_difference"]:
        return pd.Series({
            "elev_after_external_fill": repo_elev,
            "elev_external_source": "repo",
            "elev_external_note": repo_flag
        })

    # Otherwise keep missing
    return pd.Series({
        "elev_after_external_fill": pd.NA,
        "elev_external_source": "missing",
        "elev_external_note": "no safe elevation source"
    })


maindataframe_elev_external_filled[
    [
        "elev_after_external_fill",
        "elev_external_source",
        "elev_external_note"
    ]
] = maindataframe_elev_external_filled.apply(
    choose_external_elev,
    axis=1
)


print("Rows:", len(maindataframe_elev_external_filled))

print("\nMissing after same-mountain fill:")
print(maindataframe_elev_external_filled["elev_after_same_mountain_fill"].isna().sum())

print("\nMissing after external fill:")
print(maindataframe_elev_external_filled["elev_after_external_fill"].isna().sum())

print("\nExternal elevation source counts:")
print(maindataframe_elev_external_filled["elev_external_source"].value_counts(dropna=False))

maindataframe_elev_external_filled[
    [
        "name",
        "elev_after_same_mountain_fill",
        "elev_after_external_fill",
        "elev_external_source",
        "elev_external_note",
        "peakvisor_elev_m",
        "peakvisor_elev_validation_flag",
        "repo_elev",
        "repo_elev_validation_flag"
    ]
].head(100)

Rows: 2677

Missing after same-mountain fill:
1078

Missing after external fill:
1075

External elevation source counts:
elev_external_source
already_filled_or_original    1599
missing                       1075
peakvisor                        3
Name: count, dtype: int64


,name,elev_after_same_mountain_fill,elev_after_external_fill,elev_external_source,elev_external_note,peakvisor_elev_m,peakvisor_elev_validation_flag,repo_elev,repo_elev_validation_flag
0,Mount 387,777.0,777.0,already_filled_or_original,kept existing elevation,NaN,NaN,NaN,no_repo_elevation
1,Mount Abao,2527.0,2527.0,already_filled_or_original,kept existing elevation,2596.0,minor_difference,NaN,no_repo_elevation
2,Abel's Peak,1464.0,1464.0,already_filled_or_original,kept existing elevation,1529.0,minor_difference,NaN,no_repo_elevation
3,Mount Abnataclayong,370.0,370.0,already_filled_or_original,kept existing elevation,NaN,NaN,NaN,no_repo_elevation
4,Mount Aboabo,NaN,<NA>,missing,no safe elevation source,NaN,NaN,NaN,no_repo_elevation
...,...,...,...,...,...,...,...,...,...
95,Mount Ampakaw,1889.0,1889.0,already_filled_or_original,kept existing elevation,NaN,NaN,NaN,no_repo_elevation
96,Mount Ampalauag,1659.0,1659.0,already_filled_or_original,kept existing elevation,NaN,NaN,1697.0,minor_difference
97,Mount Ampawid,844.0,844.0,already_filled_or_original,kept existing elevation,NaN,NaN,844.0,close_match
98,Mount Ampayao,905.0,905.0,already_filled_or_original,kept existing elevation,NaN,NaN,905.0,close_match


In [272]:
# ============================================================
# Fill missing prom and is_volc within safe duplicate groups
# ============================================================

import numpy as np
import pandas as pd

final_df_attr_filled = maindataframe_elev_external_filled.copy()


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def first_existing_col(df, possible_cols):
    for col in possible_cols:
        if col in df.columns:
            return col
    return None


def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def clean_bool_value(x):
    if pd.isna(x):
        return np.nan

    x_str = str(x).strip().lower()

    if x_str in ["true", "1", "yes", "y", "volcano", "volcanic"]:
        return True
    elif x_str in ["false", "0", "no", "n", "non-volcano", "not volcanic"]:
        return False
    else:
        return np.nan


def same_admin_area(row, donor):
    return (
        (pd.notna(row.get("prov")) and row.get("prov") == donor.get("prov"))
        or (pd.notna(row.get("region")) and row.get("region") == donor.get("region"))
        or (pd.notna(row.get("isl_grp")) and row.get("isl_grp") == donor.get("isl_grp"))
    )


# ------------------------------------------------------------
# 2. Create final working fields
# ------------------------------------------------------------

# Prominence: start with best already-existing value on the same row
prom_source_cols = [
    "prom_enriched",
    "prom",
    "peakvisor_prom_m",
    "repo_prom"
]

existing_prom_cols = [col for col in prom_source_cols if col in final_df_attr_filled.columns]

final_df_attr_filled["prom_final_m"] = np.nan

for col in existing_prom_cols:
    final_df_attr_filled["prom_final_m"] = final_df_attr_filled["prom_final_m"].fillna(
        pd.to_numeric(final_df_attr_filled[col], errors="coerce")
    )


# Volcano: start with best already-existing value on the same row
is_volc_source_cols = [
    "is_volc_enriched",
    "is_volc",
    "repo_is_volc"
]

existing_is_volc_cols = [col for col in is_volc_source_cols if col in final_df_attr_filled.columns]

final_df_attr_filled["is_volc_final"] = np.nan

for col in existing_is_volc_cols:
    cleaned = final_df_attr_filled[col].apply(clean_bool_value)
    final_df_attr_filled["is_volc_final"] = final_df_attr_filled["is_volc_final"].fillna(cleaned)


# Optional: if row itself has volcano info text, mark as True only when still missing
if "has_volcano_info" in final_df_attr_filled.columns:
    volcano_info_clean = final_df_attr_filled["has_volcano_info"].apply(clean_bool_value)

    final_df_attr_filled.loc[
        final_df_attr_filled["is_volc_final"].isna() & (volcano_info_clean == True),
        "is_volc_final"
    ] = True


# Track original/fill source
final_df_attr_filled["prom_final_source"] = np.where(
    final_df_attr_filled["prom_final_m"].notna(),
    "existing_row_value",
    "missing"
)

final_df_attr_filled["is_volc_final_source"] = np.where(
    final_df_attr_filled["is_volc_final"].notna(),
    "existing_row_value",
    "missing"
)


# ------------------------------------------------------------
# 3. Score possible donor rows
# ------------------------------------------------------------

def quickmap_score(row):
    confidence = str(row.get("quickmap_confidence", "")).lower()

    if "high confidence" in confidence:
        return 3
    elif "medium confidence" in confidence:
        return 2
    elif "low confidence" in confidence:
        return 1
    else:
        return 0


def peakvisor_score(row):
    flag = str(row.get("peakvisor_elev_validation_flag", "")).lower()

    if flag == "close_match":
        return 3
    elif flag == "minor_difference":
        return 2
    elif flag == "needs_review":
        return 1
    else:
        return 0


def repo_score(row):
    flag = str(row.get("repo_elev_validation_flag", "")).lower()

    if flag == "close_match":
        return 3
    elif flag == "minor_difference":
        return 2
    elif flag == "needs_review":
        return 1
    elif flag == "no_original_elevation":
        return 1
    else:
        return 0


final_df_attr_filled["duplicate_donor_score"] = (
    final_df_attr_filled["quickmap_score"] if "quickmap_score" in final_df_attr_filled.columns else 0
)

final_df_attr_filled["quickmap_score_attr"] = final_df_attr_filled.apply(quickmap_score, axis=1)
final_df_attr_filled["peakvisor_score_attr"] = final_df_attr_filled.apply(peakvisor_score, axis=1)
final_df_attr_filled["repo_score_attr"] = final_df_attr_filled.apply(repo_score, axis=1)

final_df_attr_filled["attr_donor_score"] = (
    final_df_attr_filled["quickmap_score_attr"] * 10
    + final_df_attr_filled["peakvisor_score_attr"] * 5
    + final_df_attr_filled["repo_score_attr"] * 3
)


# ------------------------------------------------------------
# 4. Fill prom and is_volc inside safe duplicate groups
# ------------------------------------------------------------

prom_filled_count = 0
is_volc_filled_count = 0

prom_fill_notes = []
is_volc_fill_notes = []

duplicate_name_groups = final_df_attr_filled[
    final_df_attr_filled["name_clean"].duplicated(keep=False)
].copy()

for name_clean, group in duplicate_name_groups.groupby("name_clean"):
    group = group.copy()

    prom_donors = group[group["prom_final_m"].notna()].copy()
    is_volc_donors = group[group["is_volc_final"].notna()].copy()

    prom_donors = prom_donors.sort_values("attr_donor_score", ascending=False)
    is_volc_donors = is_volc_donors.sort_values("attr_donor_score", ascending=False)

    for idx, row in group.iterrows():

        # ------------------------------
        # Fill missing prominence
        # ------------------------------
        if pd.isna(final_df_attr_filled.loc[idx, "prom_final_m"]) and len(prom_donors) > 0:
            best_donor = None
            best_distance = np.nan

            for _, donor in prom_donors.iterrows():
                distance = haversine_km(
                    row.get("lat"), row.get("lon"),
                    donor.get("lat"), donor.get("lon")
                )

                admin_match = same_admin_area(row, donor)

                safe_to_fill = (
                    (pd.notna(distance) and distance <= 5)
                    or (pd.isna(distance) and admin_match)
                )

                if safe_to_fill:
                    best_donor = donor
                    best_distance = distance
                    break

            if best_donor is not None:
                final_df_attr_filled.loc[idx, "prom_final_m"] = best_donor["prom_final_m"]
                final_df_attr_filled.loc[idx, "prom_final_source"] = "duplicate_group_safe_fill"
                prom_filled_count += 1

                prom_fill_notes.append({
                    "target_index": idx,
                    "name_clean": name_clean,
                    "name": row.get("name"),
                    "filled_prom": best_donor["prom_final_m"],
                    "donor_name": best_donor.get("name"),
                    "donor_mountain_id": best_donor.get("mountain_id"),
                    "distance_to_donor_km": best_distance,
                    "donor_score": best_donor.get("attr_donor_score")
                })

        # ------------------------------
        # Fill missing volcano flag
        # ------------------------------
        if pd.isna(final_df_attr_filled.loc[idx, "is_volc_final"]) and len(is_volc_donors) > 0:
            best_donor = None
            best_distance = np.nan

            for _, donor in is_volc_donors.iterrows():
                distance = haversine_km(
                    row.get("lat"), row.get("lon"),
                    donor.get("lat"), donor.get("lon")
                )

                admin_match = same_admin_area(row, donor)

                safe_to_fill = (
                    (pd.notna(distance) and distance <= 5)
                    or (pd.isna(distance) and admin_match)
                )

                if safe_to_fill:
                    best_donor = donor
                    best_distance = distance
                    break

            if best_donor is not None:
                final_df_attr_filled.loc[idx, "is_volc_final"] = best_donor["is_volc_final"]
                final_df_attr_filled.loc[idx, "is_volc_final_source"] = "duplicate_group_safe_fill"
                is_volc_filled_count += 1

                is_volc_fill_notes.append({
                    "target_index": idx,
                    "name_clean": name_clean,
                    "name": row.get("name"),
                    "filled_is_volc": best_donor["is_volc_final"],
                    "donor_name": best_donor.get("name"),
                    "donor_mountain_id": best_donor.get("mountain_id"),
                    "distance_to_donor_km": best_distance,
                    "donor_score": best_donor.get("attr_donor_score")
                })


prom_fill_review = pd.DataFrame(prom_fill_notes)
is_volc_fill_review = pd.DataFrame(is_volc_fill_notes)


# ------------------------------------------------------------
# 5. Results
# ------------------------------------------------------------

print("Prominence filled from safe duplicate groups:", prom_filled_count)
print("Volcano flags filled from safe duplicate groups:", is_volc_filled_count)

print("\nProm final source counts:")
print(final_df_attr_filled["prom_final_source"].value_counts(dropna=False))

print("\nIs_volc final source counts:")
print(final_df_attr_filled["is_volc_final_source"].value_counts(dropna=False))

print("\nProm final missing count:", final_df_attr_filled["prom_final_m"].isna().sum())
print("Is_volc final missing count:", final_df_attr_filled["is_volc_final"].isna().sum())

final_df_attr_filled[[
    "name",
    "name_clean",
    "prom",
    "prom_final_m",
    "prom_final_source",
    "is_volc",
    "is_volc_final",
    "is_volc_final_source",
    "lat",
    "lon",
    "prov",
    "region",
    "isl_grp"
]].head(50)

Prominence filled from safe duplicate groups: 0
Volcano flags filled from safe duplicate groups: 0

Prom final source counts:
prom_final_source
missing               1918
existing_row_value     759
Name: count, dtype: int64

Is_volc final source counts:
is_volc_final_source
missing               2663
existing_row_value      14
Name: count, dtype: int64

Prom final missing count: 1918
Is_volc final missing count: 2663


,name,name_clean,prom,prom_final_m,prom_final_source,is_volc,is_volc_final,is_volc_final_source,lat,lon,prov,region,isl_grp
0,Mount 387,387,NaN,NaN,missing,NaN,NaN,missing,15.900298,120.973771,Nueva Ecija,Region III,Luzon
1,Mount Abao,abao,NaN,1095.0,existing_row_value,NaN,NaN,missing,16.832307,120.914886,Ifugao,CAR,Luzon
2,Abel's Peak,abel's peak,NaN,100.0,existing_row_value,NaN,NaN,missing,12.431130,122.573755,Romblon,MIMAROPA,Luzon
3,Mount Abnataclayong,abnataclayong,NaN,NaN,missing,NaN,NaN,missing,5.675000,125.366944,Sarangani,Region XII,Mindanao
4,Mount Aboabo,aboabo,NaN,NaN,missing,NaN,NaN,missing,9.147528,118.060024,Palawan,MIMAROPA,Luzon
5,Abong-abong Peak,abong-abong peak,NaN,NaN,missing,NaN,NaN,missing,6.501684,121.984999,Basilan,BARMM,Mindanao
6,Mount Abongabong,abongabong,NaN,NaN,missing,NaN,NaN,missing,13.395455,120.750376,Occidental Mindoro,MIMAROPA,Luzon
7,Mount Aborlan,aborlan,NaN,408.0,existing_row_value,NaN,NaN,missing,9.510556,118.467222,Palawan,MIMAROPA,Luzon
8,Mount Abra de Ilog,abra de ilog,NaN,537.0,existing_row_value,NaN,NaN,missing,13.454526,120.673642,Occidental Mindoro,MIMAROPA,Luzon
9,Mount Abriringan,abriringan,NaN,NaN,missing,NaN,NaN,missing,18.299000,122.270372,Cagayan,Region II,Luzon


# Final Data Quality Check

Review possible duplicates by name + distance
Best row:
final elevation
final prominence
final volcano classification
high QuickMap confidence
coordinates
source/detail evidence
more complete fields overall

1. Picked the best row to keep
2. Filled missing useful values into that kept row
3. Combined alt_names / repo_alt_names
4. Marked the other rows as duplicate rows to drop
5. Removed only those safe duplicate rows

In [273]:
# ============================================================
# Deduplicate same mountains safely
# Input: final_df_attr_filled
# Output: final_df_deduped
# ============================================================

import numpy as np
import pandas as pd

dedup_input = final_df_attr_filled.copy()
dedup_input.columns = dedup_input.columns.str.strip()

# ------------------------------------------------------------
# 1. Settings
# ------------------------------------------------------------

DEDUP_DISTANCE_KM = 1.0   # strict dedup threshold
REQUIRE_SAME_ADMIN = True

print("Starting rows:", len(dedup_input))


# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    return 2 * R * np.arcsin(np.sqrt(a))


def same_admin(row1, row2):
    prov_same = (
        pd.notna(row1.get("prov"))
        and pd.notna(row2.get("prov"))
        and row1.get("prov") == row2.get("prov")
    )

    region_same = (
        pd.notna(row1.get("region"))
        and pd.notna(row2.get("region"))
        and row1.get("region") == row2.get("region")
    )

    island_same = (
        pd.notna(row1.get("isl_grp"))
        and pd.notna(row2.get("isl_grp"))
        and row1.get("isl_grp") == row2.get("isl_grp")
    )

    return prov_same or region_same or island_same


def source_score(row):
    score = 0

    # Prefer rows with final elevation
    if pd.notna(row.get("elev_final_m")):
        score += 100

    # Prefer rows with final prominence
    if pd.notna(row.get("prom_final_m")):
        score += 40
    elif pd.notna(row.get("prom_enriched")):
        score += 30
    elif pd.notna(row.get("prom")):
        score += 20

    # Prefer rows with volcano info
    if pd.notna(row.get("is_volc_final")):
        score += 30
    elif pd.notna(row.get("is_volc_enriched")):
        score += 25
    elif pd.notna(row.get("is_volc")):
        score += 20

    # Prefer high confidence coordinate/admin match
    quickmap_conf = str(row.get("quickmap_confidence", "")).lower()
    if "high confidence" in quickmap_conf:
        score += 50
    elif "medium confidence" in quickmap_conf:
        score += 30
    elif "low confidence" in quickmap_conf:
        score += 5

    # Prefer rows with coordinates
    if pd.notna(row.get("lat")) and pd.notna(row.get("lon")):
        score += 30

    # Prefer rows with source/detail evidence
    if pd.notna(row.get("detail_url")) or pd.notna(row.get("detail_url_peakvisor")):
        score += 10
    if pd.notna(row.get("repo_source_file")):
        score += 8
    if pd.notna(row.get("osm_id")):
        score += 5

    # Prefer richer rows overall
    score += row.notna().sum() * 0.1

    return score


def first_non_null_by_score(group, col):
    if col not in group.columns:
        return np.nan

    available = group[group[col].notna()].copy()

    if available.empty:
        return np.nan

    available = available.sort_values("dedup_row_score", ascending=False)
    return available.iloc[0][col]


def combine_unique_text(group, cols):
    values = []

    for col in cols:
        if col in group.columns:
            for val in group[col].dropna():
                val = str(val).strip()
                if val and val.lower() not in ["nan", "none", "<na>"]:
                    values.extend([v.strip() for v in val.split(";") if v.strip()])

    unique_values = sorted(set(values))

    if len(unique_values) == 0:
        return np.nan

    return "; ".join(unique_values)


# ------------------------------------------------------------
# 3. Build safe duplicate groups
# ------------------------------------------------------------

dedup_input = dedup_input.reset_index(drop=True).copy()
dedup_input["dedup_original_index"] = dedup_input.index
dedup_input["dedup_row_score"] = dedup_input.apply(source_score, axis=1)

dedup_input["dedup_group_id"] = pd.NA
dedup_input["dedup_action"] = "keep_unique"

group_counter = 1

candidate_names = (
    dedup_input.loc[
        dedup_input["name_clean"].notna()
        & dedup_input["name_clean"].duplicated(keep=False),
        "name_clean"
    ]
    .dropna()
    .unique()
)

for name_clean in candidate_names:
    group = dedup_input[dedup_input["name_clean"] == name_clean].copy()

    if len(group) < 2:
        continue

    used_indices = set()

    for idx, row in group.iterrows():
        if idx in used_indices:
            continue

        current_cluster = [idx]
        used_indices.add(idx)

        for idx2, row2 in group.iterrows():
            if idx2 == idx or idx2 in used_indices:
                continue

            distance = haversine_km(
                row.get("lat"), row.get("lon"),
                row2.get("lat"), row2.get("lon")
            )

            admin_match = same_admin(row, row2)

            safe_duplicate = (
                pd.notna(distance)
                and distance <= DEDUP_DISTANCE_KM
                and (admin_match if REQUIRE_SAME_ADMIN else True)
            )

            if safe_duplicate:
                current_cluster.append(idx2)
                used_indices.add(idx2)

        if len(current_cluster) > 1:
            group_id = f"dedup_{group_counter:05d}"
            dedup_input.loc[current_cluster, "dedup_group_id"] = group_id
            group_counter += 1


print("Safe duplicate groups found:", dedup_input["dedup_group_id"].notna().sum())
print("Number of duplicate groups:", dedup_input["dedup_group_id"].nunique(dropna=True))

Starting rows: 2677
Safe duplicate groups found: 245
Number of duplicate groups: 73


In [274]:
# ============================================================
# Consolidate values inside safe duplicate groups, then deduplicate
# ============================================================

dedup_work = dedup_input.copy()

# Columns where we want to preserve the best available value before dropping duplicates
value_cols_to_consolidate = [
    # Final fields
    "elev_final_m",
    "elev_final_source",
    "prom_final_m",
    "prom_final_source",
    "is_volc_final",
    "is_volc_final_source",

    # Original/enriched values
    "elev",
    "prom",
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "repo_elev",
    "repo_prom",
    "prom_enriched",
    "prom_enrichment_source",
    "is_volc",
    "repo_is_volc",
    "is_volc_enriched",
    "is_volc_enrichment_source",

    # Evidence/source fields
    "detail_url",
    "detail_url_peakvisor",
    "url_source",
    "detail_scrape_status",
    "repo_source_file",
    "repo_match_status",
    "quickmap_confidence",
    "quickmap_status",
    "quickmap_validation_note",

    # Descriptive fields
    "peakvisor_location_text",
    "peakvisor_about_text",
    "has_trail_info",
    "trail_sentence",
    "has_volcano_info",
    "volcano_sentence",

    # Admin notes
    "data_quality_notes",
    "admin_note"
]

text_merge_cols = [
    "alt_names",
    "repo_alt_names"
]

duplicate_group_summaries = []
rows_to_drop = []

for group_id, group in dedup_work[dedup_work["dedup_group_id"].notna()].groupby("dedup_group_id"):
    group = group.copy()

    # Pick best row to keep
    group = group.sort_values("dedup_row_score", ascending=False)
    keep_idx = group.index[0]
    drop_indices = group.index[1:].tolist()

    # Consolidate best non-null values into kept row
    for col in value_cols_to_consolidate:
        if col in dedup_work.columns:
            best_val = first_non_null_by_score(group, col)

            if pd.notna(best_val):
                # Only fill if kept row is missing
                if pd.isna(dedup_work.loc[keep_idx, col]):
                    dedup_work.loc[keep_idx, col] = best_val

    # Merge alternative names
    merged_alt_names = combine_unique_text(group, text_merge_cols)

    if pd.notna(merged_alt_names):
        if "alt_names" in dedup_work.columns:
            dedup_work.loc[keep_idx, "alt_names"] = merged_alt_names

    dedup_work.loc[keep_idx, "dedup_action"] = "kept_best_consolidated"
    dedup_work.loc[drop_indices, "dedup_action"] = "drop_duplicate"

    rows_to_drop.extend(drop_indices)

    duplicate_group_summaries.append({
        "dedup_group_id": group_id,
        "name_clean": group["name_clean"].iloc[0],
        "group_size": len(group),
        "kept_mountain_id": dedup_work.loc[keep_idx, "mountain_id"] if "mountain_id" in dedup_work.columns else keep_idx,
        "kept_name": dedup_work.loc[keep_idx, "name"] if "name" in dedup_work.columns else None,
        "rows_dropped": len(drop_indices),
        "max_distance_within_group_km": np.nan,
        "kept_score": dedup_work.loc[keep_idx, "dedup_row_score"]
    })


duplicate_group_review = pd.DataFrame(duplicate_group_summaries)

duplicate_rows_removed_review = dedup_work.loc[rows_to_drop].copy()

final_df_deduped = dedup_work.drop(index=rows_to_drop).copy()
final_df_deduped = final_df_deduped.reset_index(drop=True)

print("Rows before dedup:", len(dedup_input))
print("Rows removed:", len(rows_to_drop))
print("Rows after dedup:", len(final_df_deduped))

print("\nDedup action counts:")
print(final_df_deduped["dedup_action"].value_counts(dropna=False))

duplicate_group_review.head(50)

Rows before dedup: 2677
Rows removed: 172
Rows after dedup: 2505

Dedup action counts:
dedup_action
keep_unique               2432
kept_best_consolidated      73
Name: count, dtype: int64


,dedup_group_id,name_clean,group_size,kept_mountain_id,kept_name,rows_dropped,max_distance_within_group_km,kept_score
0,dedup_00001,agamomata,2,agamomata_18.1548_120.9678,Mount Agamomata,1,NaN,190.9
1,dedup_00002,agtuuganon,2,agtuuganon_7.7978_126.2035,Mount Agtuuganon,1,NaN,240.2
2,dedup_00003,agudo,2,agudo_14.8917_120.1098,Mount Agudo,1,NaN,189.7
3,dedup_00004,agudo,2,agudo_9.6_125.45,Mount Agudo,1,NaN,189.2
4,dedup_00005,agudo,2,agudo_11.386_123.0036,Mount Agudo,1,NaN,189.2
5,dedup_00006,bagsak,2,bagsak_5.8893_125.568,Mount Bagsak,1,NaN,190.9
6,dedup_00007,bagsak,2,bagsak_6.0281_121.1394,Mount Bagsak,1,NaN,190.9
7,dedup_00008,balabac,2,balabac_11.6013_122.1473,Mount Balabac,1,NaN,190.9
8,dedup_00009,bantay lakay,2,bantay lakay_16.4207_121.4112,Mount Bantay Lakay,1,NaN,201.4
9,dedup_00010,bener,2,bener_18.1464_120.9611,Mount Bener,1,NaN,192.2


In [275]:
# ============================================================
# Deduplication QA checks
# ============================================================

print("Final deduped shape:", final_df_deduped.shape)

print("\nRows removed review shape:", duplicate_rows_removed_review.shape)

print("\nRemaining duplicated name_clean groups:")
remaining_same_name_groups = final_df_deduped[
    final_df_deduped["name_clean"].duplicated(keep=False)
].copy()

print("Rows still sharing name_clean:", len(remaining_same_name_groups))
print("Same-name groups still remaining:", remaining_same_name_groups["name_clean"].nunique())

print("\nImportant missing counts:")
for col in [
    "elev_final_m",
    "prom_final_m",
    "is_volc_final",
    "lat",
    "lon",
    "prov",
    "region",
    "isl_grp"
]:
    if col in final_df_deduped.columns:
        print(col, "missing:", final_df_deduped[col].isna().sum())

remaining_same_name_groups[
    [
        col for col in [
            "name_clean",
            "name",
            "mountain_id",
            "elev_final_m",
            "prom_final_m",
            "is_volc_final",
            "lat",
            "lon",
            "prov",
            "region",
            "isl_grp",
            "dedup_action"
        ]
        if col in remaining_same_name_groups.columns
    ]
].sort_values(["name_clean", "name"]).head(100)

Final deduped shape: (2505, 103)

Rows removed review shape: (172, 103)

Remaining duplicated name_clean groups:
Rows still sharing name_clean: 182
Same-name groups still remaining: 82

Important missing counts:
elev_final_m missing: 43
prom_final_m missing: 1903
is_volc_final missing: 2493
lat missing: 0
lon missing: 0
prov missing: 0
region missing: 0
isl_grp missing: 0


,name_clean,name,mountain_id,elev_final_m,prom_final_m,is_volc_final,lat,lon,prov,region,isl_grp,dedup_action
23,agapang,Mount Agapang,agapang_17.5206_120.942,503.0,NaN,NaN,17.520600,120.942000,Abra,CAR,Luzon,keep_unique
24,agapang,Mount Agapang,agapang_17.5167_120.7167,1504.0,NaN,NaN,17.516667,120.716667,Abra,CAR,Luzon,keep_unique
42,agudo,Mount Agudo,agudo_14.8917_120.1098,990.0,738.0,NaN,14.891656,120.109819,Zambales,Region III,Luzon,kept_best_consolidated
43,agudo,Mount Agudo,agudo_9.6_125.45,783.0,738.0,NaN,9.600000,125.450000,Surigao del Norte,Region XIII,Mindanao,kept_best_consolidated
44,agudo,Mount Agudo,agudo_11.386_123.0036,783.0,738.0,NaN,11.385983,123.003570,Capiz,Region VI,Visayas,kept_best_consolidated
...,...,...,...,...,...,...,...,...,...,...,...,...
1266,lusong,Mount Lusong,lusong_16.8667_120.4667,1827.0,NaN,NaN,16.866667,120.466667,La Union,Region I,Luzon,keep_unique
1285,mabukok,Mount Mabukok,mabukok_17.4333_121.05,1437.0,NaN,NaN,17.433333,121.050000,Kalinga,CAR,Luzon,keep_unique
1286,mabukok,Mount Mabukok,mabukok_17.4723_121.0519,1437.0,NaN,NaN,17.472300,121.051900,Kalinga,CAR,Luzon,keep_unique
1323,magsingan,Mount Magsingan,magsingan_18.4081_120.9975,1138.0,NaN,NaN,18.408056,120.997500,Apayao,CAR,Luzon,keep_unique


In [276]:
# ============================================================
# Safe checkpoint
# ============================================================

final_df_deduped.to_csv(
    "philippines_mountains_final_deduped.csv",
    index=False
)

duplicate_rows_removed_review.to_csv(
    "philippines_mountains_duplicate_rows_removed_review.csv",
    index=False
)

duplicate_group_review.to_csv(
    "philippines_mountains_duplicate_group_review.csv",
    index=False
)

print("Saved:")
print("philippines_mountains_final_deduped.csv")
print("philippines_mountains_duplicate_rows_removed_review.csv")
print("philippines_mountains_duplicate_group_review.csv")

Saved:
philippines_mountains_final_deduped.csv
philippines_mountains_duplicate_rows_removed_review.csv
philippines_mountains_duplicate_group_review.csv


# Export final Tableau-ready dataset

In [277]:
# ============================================================
# Final deduplication QA checks
# ============================================================

print("Final deduped dataframe shape:", final_df_deduped.shape)

print("\nRows removed during deduplication:")
print(len(duplicate_rows_removed_review))

print("\nDuplicate group review shape:")
print(duplicate_group_review.shape)

print("\nDedup action counts:")
if "dedup_action" in final_df_deduped.columns:
    print(final_df_deduped["dedup_action"].value_counts(dropna=False))

print("\nImportant missing counts:")
for col in [
    "elev_final_m",
    "prom_final_m",
    "is_volc_final",
    "lat",
    "lon",
    "prov",
    "region",
    "isl_grp"
]:
    if col in final_df_deduped.columns:
        print(col, "missing:", final_df_deduped[col].isna().sum())

print("\nRemaining same-name rows:")
remaining_same_name_rows = final_df_deduped[
    final_df_deduped["name_clean"].duplicated(keep=False)
].copy()

print("Rows still sharing name_clean:", len(remaining_same_name_rows))
print("Same-name groups still remaining:", remaining_same_name_rows["name_clean"].nunique())

remaining_same_name_rows[
    [
        col for col in [
            "name_clean",
            "name",
            "mountain_id",
            "elev_final_m",
            "prom_final_m",
            "is_volc_final",
            "lat",
            "lon",
            "prov",
            "region",
            "isl_grp",
            "dedup_action"
        ]
        if col in remaining_same_name_rows.columns
    ]
].sort_values(["name_clean", "name"]).head(100)

Final deduped dataframe shape: (2505, 103)

Rows removed during deduplication:
172

Duplicate group review shape:
(73, 8)

Dedup action counts:
dedup_action
keep_unique               2432
kept_best_consolidated      73
Name: count, dtype: int64

Important missing counts:
elev_final_m missing: 43
prom_final_m missing: 1903
is_volc_final missing: 2493
lat missing: 0
lon missing: 0
prov missing: 0
region missing: 0
isl_grp missing: 0

Remaining same-name rows:
Rows still sharing name_clean: 182
Same-name groups still remaining: 82


,name_clean,name,mountain_id,elev_final_m,prom_final_m,is_volc_final,lat,lon,prov,region,isl_grp,dedup_action
23,agapang,Mount Agapang,agapang_17.5206_120.942,503.0,NaN,NaN,17.520600,120.942000,Abra,CAR,Luzon,keep_unique
24,agapang,Mount Agapang,agapang_17.5167_120.7167,1504.0,NaN,NaN,17.516667,120.716667,Abra,CAR,Luzon,keep_unique
42,agudo,Mount Agudo,agudo_14.8917_120.1098,990.0,738.0,NaN,14.891656,120.109819,Zambales,Region III,Luzon,kept_best_consolidated
43,agudo,Mount Agudo,agudo_9.6_125.45,783.0,738.0,NaN,9.600000,125.450000,Surigao del Norte,Region XIII,Mindanao,kept_best_consolidated
44,agudo,Mount Agudo,agudo_11.386_123.0036,783.0,738.0,NaN,11.385983,123.003570,Capiz,Region VI,Visayas,kept_best_consolidated
...,...,...,...,...,...,...,...,...,...,...,...,...
1266,lusong,Mount Lusong,lusong_16.8667_120.4667,1827.0,NaN,NaN,16.866667,120.466667,La Union,Region I,Luzon,keep_unique
1285,mabukok,Mount Mabukok,mabukok_17.4333_121.05,1437.0,NaN,NaN,17.433333,121.050000,Kalinga,CAR,Luzon,keep_unique
1286,mabukok,Mount Mabukok,mabukok_17.4723_121.0519,1437.0,NaN,NaN,17.472300,121.051900,Kalinga,CAR,Luzon,keep_unique
1323,magsingan,Mount Magsingan,magsingan_18.4081_120.9975,1138.0,NaN,NaN,18.408056,120.997500,Apayao,CAR,Luzon,keep_unique


In [278]:
# ============================================================
# Review removed duplicate rows
# ============================================================

duplicate_rows_removed_review[
    [
        col for col in [
            "dedup_group_id",
            "dedup_action",
            "name_clean",
            "name",
            "mountain_id",
            "elev_final_m",
            "prom_final_m",
            "is_volc_final",
            "lat",
            "lon",
            "prov",
            "region",
            "isl_grp",
            "quickmap_confidence",
            "dedup_row_score"
        ]
        if col in duplicate_rows_removed_review.columns
    ]
].sort_values(["dedup_group_id", "name_clean"]).head(100)

,dedup_group_id,dedup_action,name_clean,name,mountain_id,elev_final_m,prom_final_m,is_volc_final,lat,lon,prov,region,isl_grp,quickmap_confidence,dedup_row_score
21,dedup_00001,drop_duplicate,agamomata,Mount Agamomata,agamomata_18.1548_120.9678,1649.0,NaN,NaN,18.154763,120.967806,Apayao,CAR,Luzon,High confidence,190.9
41,dedup_00002,drop_duplicate,agtuuganon,Mount Agtuuganon,agtuuganon_7.7978_126.2035,1660.0,1195.0,NaN,7.797766,126.203465,Davao de Oro,Region XI,Mindanao,High confidence,240.2
45,dedup_00003,drop_duplicate,agudo,Mount Agudo,agudo_14.8917_120.1098,990.0,738.0,NaN,14.891656,120.109819,Zambales,Region III,Luzon,No QuickMap match,189.7
47,dedup_00004,drop_duplicate,agudo,Mount Agudo,agudo_9.6_125.45,783.0,738.0,NaN,9.600000,125.450000,Surigao del Norte,Region XIII,Mindanao,No QuickMap match,189.2
49,dedup_00005,drop_duplicate,agudo,Mount Agudo,agudo_11.386_123.0036,783.0,738.0,NaN,11.385983,123.003570,Capiz,Region VI,Visayas,No QuickMap match,189.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2164,dedup_00056,drop_duplicate,sharp peak,Sharp Peak,sharp peak_7.4833_124.1833,NaN,445.0,NaN,7.483333,124.183333,Lanao del Sur,BARMM,Mindanao,Low confidence - likely wrong name match,94.5
2165,dedup_00056,drop_duplicate,sharp peak,Sharp Peak,sharp peak_7.4833_124.1833,NaN,445.0,NaN,7.483333,124.183333,Lanao del Sur,BARMM,Mindanao,Low confidence - likely wrong name match,94.5
2166,dedup_00057,drop_duplicate,sharp peak,Sharp Peak,sharp peak_13.6333_124.15,NaN,445.0,NaN,13.633333,124.150000,Catanduanes,Region V,Luzon,Low confidence - likely wrong name match,94.6
2167,dedup_00057,drop_duplicate,sharp peak,Sharp Peak,sharp peak_13.6333_124.15,NaN,445.0,NaN,13.633333,124.150000,Catanduanes,Region V,Luzon,Low confidence - likely wrong name match,94.6


In [279]:
# ============================================================
# Create final full export dataframe
# ============================================================

final_df_export = final_df_deduped.copy()
final_df_export.columns = final_df_export.columns.str.strip()

# Final preferred column order
final_export_cols = [
    # IDs and names
    "mountain_id",
    "name",
    "name_original",
    "name_clean",
    "alt_names",

    # Final BI-ready fields
    "elev_final_m",
    "elev_final_source",
    "prom_final_m",
    "prom_final_source",
    "is_volc_final",
    "is_volc_final_source",
    "marker_size",
    "expected_marker_size",
    "marker_size_check",

    # Original fields
    "elev",
    "prom",
    "is_volc",

    # Coordinates
    "coord",
    "lat",
    "lon",
    "coord_source",

    # Admin/location
    "prov",
    "prov_psgc_code",
    "region",
    "region_name",
    "region_code",
    "isl_grp",
    "prov_gis_original",
    "region_gis_original",
    "admin_note",

    # PeakVisor fields
    "peakvisor_name",
    "peakvisor_elev_m",
    "peakvisor_prom_m",
    "peakvisor_elev_source_unit",
    "peakvisor_prom_source_unit",
    "peakvisor_elev_difference_m",
    "peakvisor_elev_validation_flag",
    "peakvisor_lat",
    "peakvisor_lon",
    "peakvisor_location_text",
    "peakvisor_about_text",
    "detail_url_peakvisor",
    "url_source",
    "detail_scrape_status",

    # QuickMap fields
    "name_quickmap",
    "lat_quickmap",
    "lon_quickmap",
    "elev_quickmap",
    "quickmap_match",
    "quickmap_distance_km",
    "quickmap_validation_note",
    "quickmap_status",
    "quickmap_confidence",

    # Repo / GeoJSON fields
    "repo_name",
    "repo_elev",
    "repo_prom",
    "repo_coord",
    "repo_is_volc",
    "repo_prov",
    "repo_region",
    "repo_isl_grp",
    "repo_alt_names",
    "repo_marker_size",
    "repo_source_file",
    "repo_match_status",
    "repo_elev_difference",
    "repo_elev_validation_flag",

    # Enriched final helper fields
    "prom_enriched",
    "prom_enrichment_source",
    "is_volc_enriched",
    "is_volc_enrichment_source",

    # Trail / volcano text
    "has_trail_info",
    "trail_sentence",
    "has_volcano_info",
    "volcano_sentence",

    # Source / raw / quality
    "source",
    "source1",
    "source_file",
    "data_quality_notes",
    "detail_url",

    # OSM / geometry
    "osm_id",
    "osm_type",
    "osm_tags",
    "geometry",
    "index_right",

    # Dedup fields
    "dedup_group_id",
    "dedup_action",
    "dedup_original_index",
    "dedup_row_score",

    # Helper keys
    "peak_key"
]

existing_final_export_cols = [
    col for col in final_export_cols
    if col in final_df_export.columns
]

remaining_cols = [
    col for col in final_df_export.columns
    if col not in existing_final_export_cols
]

final_df_export = final_df_export[existing_final_export_cols + remaining_cols].copy()

print("Final export shape:", final_df_export.shape)
print("Columns:", len(final_df_export.columns))

print("\nMissing preferred columns:")
print([col for col in final_export_cols if col not in final_df_export.columns])

final_df_export.head()

Final export shape: (2505, 103)
Columns: 103

Missing preferred columns:
['source1']


,mountain_id,name,name_original,name_clean,alt_names,elev_final_m,elev_final_source,prom_final_m,prom_final_source,is_volc_final,...,elev_filled_from_same_mountain,elev_after_same_mountain_fill,elev_after_external_fill,elev_external_source,elev_external_note,duplicate_donor_score,quickmap_score_attr,peakvisor_score_attr,repo_score_attr,attr_donor_score
0,387_15.9003_120.9738,Mount 387,Mount 387,387,[],777.0,original_elev,NaN,missing,NaN,...,False,777.0,777.0,already_filled_or_original,kept existing elevation,0,0,0,0,0
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,abao,[],2527.0,original_elev,1095.0,existing_row_value,NaN,...,False,2527.0,2527.0,already_filled_or_original,kept existing elevation,0,3,2,0,40
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,abel's peak,[],1464.0,original_elev,100.0,existing_row_value,NaN,...,False,1464.0,1464.0,already_filled_or_original,kept existing elevation,0,0,2,0,10
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,[],370.0,original_elev,NaN,missing,NaN,...,False,370.0,370.0,already_filled_or_original,kept existing elevation,0,3,0,0,30
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,aboabo,[],395.0,peakvisor_missing_elev_enrichment,NaN,missing,NaN,...,False,NaN,<NA>,missing,no safe elevation source,0,0,0,0,0


In [280]:
# ============================================================
# Create BI-ready dataframe
# ============================================================

bi_ready_cols = [
    "mountain_id",
    "name",
    "name_original",
    "name_clean",
    "alt_names",

    "elev_final_m",
    "elev_final_source",
    "prom_final_m",
    "prom_final_source",
    "is_volc_final",
    "is_volc_final_source",
    "marker_size",

    "lat",
    "lon",
    "coord_source",

    "prov",
    "prov_psgc_code",
    "region",
    "region_name",
    "region_code",
    "isl_grp",

    "quickmap_confidence",
    "quickmap_distance_km",
    "peakvisor_elev_validation_flag",
    "repo_match_status",
    "repo_elev_validation_flag",

    "source",
    "source_file",
    "detail_url",
    "detail_url_peakvisor",

    "dedup_action"
]

bi_ready_existing_cols = [
    col for col in bi_ready_cols
    if col in final_df_export.columns
]

final_df_bi_ready = final_df_export[bi_ready_existing_cols].copy()

print("BI-ready shape:", final_df_bi_ready.shape)
print("BI-ready columns:", len(final_df_bi_ready.columns))

final_df_bi_ready.head()

BI-ready shape: (2505, 31)
BI-ready columns: 31


,mountain_id,name,name_original,name_clean,alt_names,elev_final_m,elev_final_source,prom_final_m,prom_final_source,is_volc_final,...,quickmap_confidence,quickmap_distance_km,peakvisor_elev_validation_flag,repo_match_status,repo_elev_validation_flag,source,source_file,detail_url,detail_url_peakvisor,dedup_action
0,387_15.9003_120.9738,Mount 387,Mount 387,387,[],777.0,original_elev,NaN,missing,NaN,...,No QuickMap match,NaN,NaN,not_found_in_repo_geojson,no_repo_elevation,OpenStreetMap,philippines-json-maps/2011/geojson/provinces/m...,NaN,NaN,keep_unique
1,abao_16.8323_120.9149,Mount Abao,Mount Abao,abao,[],2527.0,original_elev,1095.0,existing_row_value,NaN,...,High confidence,0.222378,minor_difference,not_found_in_repo_geojson,no_repo_elevation,OpenStreetMap,philippines-json-maps/2011/geojson/provinces/m...,NaN,https://peakvisor.com/peak/mount-abao.html,keep_unique
2,abel's peak_12.4311_122.5738,Abel's Peak,Abel's Peak,abel's peak,[],1464.0,original_elev,100.0,existing_row_value,NaN,...,No QuickMap match,NaN,minor_difference,not_found_in_repo_geojson,no_repo_elevation,OpenStreetMap,philippines-json-maps/2011/geojson/provinces/m...,NaN,https://peakvisor.com/peak/abel-s-peak.html,keep_unique
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,Mount Abnataclayong,abnataclayong,[],370.0,original_elev,NaN,missing,NaN,...,High confidence,0.000443,NaN,not_found_in_repo_geojson,no_repo_elevation,OpenStreetMap,philippines-json-maps/2011/geojson/provinces/m...,NaN,NaN,keep_unique
4,aboabo_9.1475_118.06,Mount Aboabo,Mount Aboabo,aboabo,[],395.0,peakvisor_missing_elev_enrichment,NaN,missing,NaN,...,No QuickMap match,NaN,NaN,not_found_in_repo_geojson,no_repo_elevation,OpenStreetMap,philippines-json-maps/2011/geojson/provinces/m...,NaN,NaN,keep_unique


In [281]:
# ============================================================
# Final sanity checks
# ============================================================

print("FULL EXPORT")
print("Rows:", len(final_df_export))
print("Columns:", len(final_df_export.columns))

print("\nBI READY")
print("Rows:", len(final_df_bi_ready))
print("Columns:", len(final_df_bi_ready.columns))

print("\nElevation source counts:")
if "elev_final_source" in final_df_export.columns:
    print(final_df_export["elev_final_source"].value_counts(dropna=False))

print("\nMarker size counts:")
if "marker_size" in final_df_export.columns:
    print(final_df_export["marker_size"].value_counts(dropna=False))

print("\nVolcano final counts:")
if "is_volc_final" in final_df_export.columns:
    print(final_df_export["is_volc_final"].value_counts(dropna=False))

print("\nProminence missing:")
if "prom_final_m" in final_df_export.columns:
    print(final_df_export["prom_final_m"].isna().sum())

print("\nElevation missing:")
if "elev_final_m" in final_df_export.columns:
    print(final_df_export["elev_final_m"].isna().sum())

print("\nCoordinate missing:")
for col in ["lat", "lon"]:
    if col in final_df_export.columns:
        print(col, final_df_export[col].isna().sum())

FULL EXPORT
Rows: 2505
Columns: 103

BI READY
Rows: 2505
Columns: 31

Elevation source counts:
elev_final_source
original_elev                        1510
peakvisor_missing_elev_enrichment     952
still_missing                          43
Name: count, dtype: int64

Marker size counts:
marker_size
small      1140
medium     1049
large       273
unknown      43
Name: count, dtype: int64

Volcano final counts:
is_volc_final
NaN     2493
True      12
Name: count, dtype: int64

Prominence missing:
1903

Elevation missing:
43

Coordinate missing:
lat 0
lon 0


In [282]:
# ============================================================
# Save final files
# ============================================================

final_df_export.to_csv(
    "philippines_mountains_final_full_deduped.csv",
    index=False
)

final_df_bi_ready.to_csv(
    "philippines_mountains_bi_ready.csv",
    index=False
)

duplicate_rows_removed_review.to_csv(
    "philippines_mountains_duplicate_rows_removed_review.csv",
    index=False
)

duplicate_group_review.to_csv(
    "philippines_mountains_duplicate_group_review.csv",
    index=False
)

print("Saved final files:")
print("philippines_mountains_final_full_deduped.csv")
print("philippines_mountains_bi_ready.csv")
print("philippines_mountains_duplicate_rows_removed_review.csv")
print("philippines_mountains_duplicate_group_review.csv")

Saved final files:
philippines_mountains_final_full_deduped.csv
philippines_mountains_bi_ready.csv
philippines_mountains_duplicate_rows_removed_review.csv
philippines_mountains_duplicate_group_review.csv


In [287]:
# ============================================================
# Create SIMPLE BI-ready export with clean column names
# ============================================================

# Use your final full dataframe after deduplication
simple_source = final_df_export.copy()

# Create simple BI dataframe manually from final/validated columns
final_df_bi_simple = pd.DataFrame()

final_df_bi_simple["mountain_id"] = simple_source["mountain_id"]
final_df_bi_simple["name"] = simple_source["name"]

# Rename final validated fields into simple names
final_df_bi_simple["elev"] = simple_source["elev_final_m"]
final_df_bi_simple["prom"] = simple_source["prom_final_m"]
final_df_bi_simple["lat"] = simple_source["lat"]
final_df_bi_simple["lon"] = simple_source["lon"]

# Coordinate pair
final_df_bi_simple["coord"] = simple_source["coord"]

# Simple volcano field
final_df_bi_simple["is_volc"] = simple_source["is_volc_final"]

# Admin fields
final_df_bi_simple["province"] = simple_source["prov"]
final_df_bi_simple["region"] = simple_source["region"]
final_df_bi_simple["region_name"] = simple_source["region_name"]
final_df_bi_simple["isl_grp"] = simple_source["isl_grp"]

# Map category
final_df_bi_simple["marker_size"] = simple_source["marker_size"]

# Source / notes
final_df_bi_simple["source"] = simple_source["source"]

final_df_bi_simple["data_quality_notes"] = (
    "elev_source=" + simple_source["elev_final_source"].astype(str)
    + "; prom_source=" + simple_source["prom_final_source"].astype(str)
    + "; is_volc_source=" + simple_source["is_volc_final_source"].astype(str)
    + "; dedup_action=" + simple_source["dedup_action"].astype(str)
)

print("Simple BI export shape:", final_df_bi_simple.shape)
final_df_bi_simple.head()

Simple BI export shape: (2505, 15)


,mountain_id,name,elev,prom,lat,lon,coord,is_volc,province,region,region_name,isl_grp,marker_size,source,data_quality_notes
0,387_15.9003_120.9738,Mount 387,777.0,NaN,15.900298,120.973771,"15.900298, 120.9737706",NaN,Nueva Ecija,Region III,Region III (Central Luzon),Luzon,medium,OpenStreetMap,elev_source=original_elev; prom_source=missing...
1,abao_16.8323_120.9149,Mount Abao,2527.0,1095.0,16.832307,120.914886,"16.8323069, 120.9148864",NaN,Ifugao,CAR,Cordillera Administrative Region (CAR),Luzon,large,OpenStreetMap,elev_source=original_elev; prom_source=existin...
2,abel's peak_12.4311_122.5738,Abel's Peak,1464.0,100.0,12.431130,122.573755,"12.4311298, 122.5737552",NaN,Romblon,MIMAROPA,MIMAROPA Region,Luzon,medium,OpenStreetMap,elev_source=original_elev; prom_source=existin...
3,abnataclayong_5.675_125.3669,Mount Abnataclayong,370.0,NaN,5.675000,125.366944,"5.675, 125.366944",NaN,Sarangani,Region XII,Region XII (SOCCSKSARGEN),Mindanao,small,OpenStreetMap,elev_source=original_elev; prom_source=missing...
4,aboabo_9.1475_118.06,Mount Aboabo,395.0,NaN,9.147528,118.060024,"9.1475284, 118.060024",NaN,Palawan,MIMAROPA,MIMAROPA Region,Luzon,small,OpenStreetMap,elev_source=peakvisor_missing_elev_enrichment;...


In [288]:
# ============================================================
# Save simple BI-ready CSV
# ============================================================

final_df_bi_simple.to_csv(
    "philippines_mountains_bi_ready_simple.csv",
    index=False
)

print("Saved: philippines_mountains_bi_ready_simple.csv")

Saved: philippines_mountains_bi_ready_simple.csv
